# Freddie Mac Mortgage PD — SQL, Modeling & Validation

Build a SQL-backed feature pipeline from the cleaned loan-level data and evaluate probability-of-default models. The workflow covers SQL feature engineering, model comparison, calibration, and out-of-time validation.

In [ ]:
import sqlite3
import csv
import os

# FREDDIE MAC MASTER PD DATASET → SQLITE

csv_path = r"./data/raw/freddie_master_pd_2018_2022.csv"

# SQLite database is created under ./data/raw/
db_path = r"./data/raw/freddie_pd.db"

print("CSV exists:", os.path.exists(csv_path))
print("CSV path:", csv_path)
print("Database:", db_path)

# CONNECT TO SQLITE

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Faster bulk loading
cur.execute("PRAGMA journal_mode = WAL")
cur.execute("PRAGMA synchronous = OFF")

# CREATE RAW TABLE

cur.execute("DROP TABLE IF EXISTS freddie_master_pd_raw")

cur.execute("""
CREATE TABLE freddie_master_pd_raw (
    CreditScore TEXT,
    FirstPaymentDate TEXT,
    FirstTimeHomebuyerFlag TEXT,
    MaturityDate TEXT,
    MIPercent TEXT,
    NumberOfUnits TEXT,
    OccupancyStatus TEXT,
    OriginalCLTV TEXT,
    OriginalDTI TEXT,
    OriginalUPB TEXT,
    OriginalLTV TEXT,
    OriginalInterestRate TEXT,
    Channel TEXT,
    PPMFlag TEXT,
    AmortizationType TEXT,
    PropertyState TEXT,
    PropertyType TEXT,
    PostalCode TEXT,
    LoanSequenceNumber TEXT,
    LoanPurpose TEXT,
    OriginalLoanTerm TEXT,
    NumberOfBorrowers TEXT,
    SellerName TEXT,
    ServicerName TEXT,
    SpecialEligibilityProgram TEXT,
    ReliefRefinanceIndicator TEXT,
    PropertyValuationMethod TEXT,
    InterestOnlyIndicator TEXT,
    Default_36M TEXT,
    Vintage TEXT
)
""")

conn.commit()

# IMPORT CSV

placeholders = ",".join(["?"] * 30)

insert_sql = f"""
INSERT INTO freddie_master_pd_raw
VALUES ({placeholders})
"""

batch = []
batch_size = 10000
count = 0
bad_rows = 0

with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:

    reader = csv.reader(f)

    # Read header
    header = next(reader)

    print("\nCSV columns:", len(header))

    if len(header) != 30:
        raise ValueError(
            f"Expected 30 columns but found {len(header)}"
        )

    print("Header check: OK")

    for row in reader:

        # Make sure every row has exactly 30 columns
        if len(row) != 30:
            bad_rows += 1
            continue

        batch.append(row)

        if len(batch) >= batch_size:

            cur.executemany(insert_sql, batch)
            conn.commit()

            count += len(batch)
            batch = []

            if count % 50000 == 0:
                print(f"Imported {count:,} rows...")

    # Remaining rows
    if batch:
        cur.executemany(insert_sql, batch)
        conn.commit()
        count += len(batch)

# VERIFY

cur.execute("""
SELECT COUNT(*)
FROM freddie_master_pd_raw
""")

db_count = cur.fetchone()[0]

print("\n" + "=" * 60)
print("IMPORT COMPLETE")
print("=" * 60)

print(f"Rows imported: {count:,}")
print(f"Rows in SQLite: {db_count:,}")
print(f"Bad rows skipped: {bad_rows:,}")

# Check columns
cur.execute("""
PRAGMA table_info(freddie_master_pd_raw)
""")

columns = cur.fetchall()

print(f"Columns in table: {len(columns)}")

# Show first 5 rows
cur.execute("""
SELECT *
FROM freddie_master_pd_raw
LIMIT 5
""")

rows = cur.fetchall()

print("\nFirst 5 rows:")
for row in rows:
    print(row)

print("\nSQLite database created at:")
print(db_path)

conn.close()


## 1. Build the SQLite analytical dataset

Load the loan-level data into SQLite and create the tables used for the downstream analysis.

In [ ]:
import sqlite3

db_path = r"./data/raw/freddie_pd.db"

conn = sqlite3.connect(db_path)

print("Connected to SQLite successfully.")


In [ ]:
query = """
SELECT *
FROM freddie_master_pd_raw
LIMIT 10;
"""

result = conn.execute(query).fetchall()

for row in result:
    print(row)


In [ ]:
import sqlite3

db_path = r"./data/raw/freddie_pd.db"

conn = sqlite3.connect(db_path)

print("Connected to SQLite successfully.")

query = """
SELECT *
FROM freddie_master_pd_raw
LIMIT 10;
"""

result = conn.execute(query).fetchall()

for row in result:
    print(row)


In [ ]:
import sqlite3
import pandas as pd

db_path = r"./data/raw/freddie_pd.db"

conn = sqlite3.connect(db_path)

print("Connected successfully!")


In [ ]:
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

tables


In [ ]:
df = pd.read_sql("""
SELECT *
FROM freddie_master_pd_raw
""", conn)

print("Shape:", df.shape)
df.head()


In [ ]:
df.columns.tolist()


In [ ]:
numeric_cols = [
    'CreditScore',
    'MIPercent',
    'NumberOfUnits',
    'OriginalCLTV',
    'OriginalDTI',
    'OriginalUPB',
    'OriginalLTV',
    'OriginalInterestRate',
    'OriginalLoanTerm',
    'NumberOfBorrowers',
    'Default_36M',
    'Vintage'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df[numeric_cols].dtypes)


In [ ]:
df['Default_36M'].value_counts(dropna=False)


## 2. SQL exploratory analysis

Use SQL aggregations and window functions to examine default patterns across borrower and loan characteristics.

In [ ]:
df['Default_36M'].value_counts(normalize=True, dropna=False)


In [ ]:
query = """
SELECT *
FROM freddie_master_pd_raw
LIMIT 10;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT COUNT(*) AS total_loans
FROM freddie_master_pd_raw;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    Default_36M,
    COUNT(*) AS loans,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM freddie_master_pd_raw), 2
    ) AS default_percentage
FROM freddie_master_pd_raw
GROUP BY Default_36M
ORDER BY Default_36M;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    Vintage,
    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,
    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct
FROM freddie_master_pd_raw
GROUP BY Vintage
ORDER BY Vintage;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    COUNT(*) AS total_loans,
    SUM(CASE WHEN CreditScore IS NULL THEN 1 ELSE 0 END) AS missing_credit_score,
    SUM(CASE WHEN OriginalDTI IS NULL THEN 1 ELSE 0 END) AS missing_dti,
    SUM(CASE WHEN OriginalLTV IS NULL THEN 1 ELSE 0 END) AS missing_ltv,
    SUM(CASE WHEN OriginalCLTV IS NULL THEN 1 ELSE 0 END) AS missing_cltv
FROM freddie_master_pd_raw;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    CASE
        WHEN CreditScore < 620 THEN '<620'
        WHEN CreditScore < 680 THEN '620-679'
        WHEN CreditScore < 740 THEN '680-739'
        WHEN CreditScore < 800 THEN '740-799'
        ELSE '800+'
    END AS credit_score_band,

    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,

    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct

FROM freddie_master_pd_raw

WHERE CreditScore IS NOT NULL
  AND CreditScore != 9999

GROUP BY credit_score_band

ORDER BY
    MIN(CreditScore);
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    CASE
        WHEN OriginalLTV < 60 THEN '<60'
        WHEN OriginalLTV < 70 THEN '60-69'
        WHEN OriginalLTV < 80 THEN '70-79'
        WHEN OriginalLTV < 90 THEN '80-89'
        ELSE '90+'
    END AS ltv_band,

    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,

    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct

FROM freddie_master_pd_raw

GROUP BY ltv_band

ORDER BY MIN(OriginalLTV);
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    CASE
        WHEN OriginalDTI < 20 THEN '<20'
        WHEN OriginalDTI < 30 THEN '20-29'
        WHEN OriginalDTI < 40 THEN '30-39'
        WHEN OriginalDTI < 50 THEN '40-49'
        ELSE '50+'
    END AS dti_band,

    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,

    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct

FROM freddie_master_pd_raw

WHERE OriginalDTI IS NOT NULL
  AND OriginalDTI != 999

GROUP BY dti_band

ORDER BY MIN(OriginalDTI);
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    LoanPurpose,
    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,
    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct
FROM freddie_master_pd_raw
GROUP BY LoanPurpose
ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    OccupancyStatus,
    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,
    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct
FROM freddie_master_pd_raw
GROUP BY OccupancyStatus
ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    Channel,
    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,
    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct
FROM freddie_master_pd_raw
GROUP BY Channel
ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT DISTINCT
    PropertyState
FROM freddie_master_pd_raw
ORDER BY PropertyState;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    PropertyState,
    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,
    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct
FROM freddie_master_pd_raw
GROUP BY PropertyState
HAVING COUNT(*) >= 1000
ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    COUNT(*) AS high_risk_loans
FROM freddie_master_pd_raw
WHERE CreditScore < 680
  AND OriginalLTV >= 80
  AND OriginalDTI >= 40;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    COUNT(*) AS high_risk_loans,
    SUM(Default_36M) AS defaults,
    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct
FROM freddie_master_pd_raw
WHERE CreditScore < 680
  AND OriginalLTV >= 80
  AND OriginalDTI >= 40;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    CASE
        WHEN CreditScore < 680
             AND OriginalLTV >= 80
             AND OriginalDTI >= 40
            THEN 'High Risk'

        WHEN CreditScore >= 740
             AND OriginalLTV < 80
             AND OriginalDTI < 40
            THEN 'Lower Risk'

        ELSE 'Intermediate'
    END AS risk_profile,

    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,

    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct

FROM freddie_master_pd_raw

GROUP BY risk_profile
ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
WITH risk_groups AS (
    SELECT
        CASE
            WHEN CreditScore < 680
                 AND OriginalLTV >= 80
                 AND OriginalDTI >= 40
                THEN 'High Risk'

            WHEN CreditScore >= 740
                 AND OriginalLTV < 80
                 AND OriginalDTI < 40
                THEN 'Lower Risk'

            ELSE 'Intermediate'
        END AS risk_profile,
        Default_36M
    FROM freddie_master_pd_raw
)

SELECT
    risk_profile,
    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,
    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct
FROM risk_groups
GROUP BY risk_profile
ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
WITH vintage_stats AS (
    SELECT
        Vintage,
        COUNT(*) AS loans,
        SUM(Default_36M) AS defaults
    FROM freddie_master_pd_raw
    GROUP BY Vintage
)

SELECT
    Vintage,
    loans,
    defaults,
    ROUND(
        100.0 * defaults / loans, 2
    ) AS default_rate_pct
FROM vintage_stats
ORDER BY Vintage;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
WITH credit_bands AS (
    SELECT
        CASE
            WHEN CreditScore < 620 THEN '<620'
            WHEN CreditScore < 680 THEN '620-679'
            WHEN CreditScore < 740 THEN '680-739'
            WHEN CreditScore < 800 THEN '740-799'
            ELSE '800+'
        END AS credit_band,
        Default_36M
    FROM freddie_master_pd_raw
    WHERE CreditScore IS NOT NULL
      AND CreditScore != 9999
)

SELECT
    credit_band,
    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,
    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*), 2
    ) AS default_rate_pct
FROM credit_bands
GROUP BY credit_band
ORDER BY default_rate_pct DESC;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
DROP TABLE IF EXISTS vintage_summary;
"""

conn.execute(query)
conn.commit()

query = """
CREATE TABLE vintage_summary AS

SELECT
    Vintage,
    COUNT(*) AS vintage_loans,
    SUM(Default_36M) AS vintage_defaults,
    ROUND(
        1.0 * SUM(Default_36M) / COUNT(*),
        4
    ) AS vintage_default_rate
FROM freddie_master_pd_raw
GROUP BY Vintage;
"""

conn.execute(query)
conn.commit()

print("vintage_summary created.")


In [ ]:
query = """
SELECT *
FROM vintage_summary
ORDER BY Vintage;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    f.LoanSequenceNumber,
    f.Vintage,
    f.CreditScore,
    f.OriginalLTV,
    f.OriginalDTI,
    f.Default_36M,

    v.vintage_loans,
    v.vintage_defaults,
    v.vintage_default_rate

FROM freddie_master_pd_raw AS f

LEFT JOIN vintage_summary AS v
    ON f.Vintage = v.Vintage

LIMIT 20;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT COUNT(*) AS joined_rows
FROM freddie_master_pd_raw AS f
LEFT JOIN vintage_summary AS v
    ON f.Vintage = v.Vintage;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    Vintage,
    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,
    ROUND(
        1.0 * SUM(Default_36M) / COUNT(*),
        4
    ) AS default_rate,

    RANK() OVER (
        ORDER BY
            1.0 * SUM(Default_36M) / COUNT(*) DESC
    ) AS default_rate_rank

FROM freddie_master_pd_raw
GROUP BY Vintage
ORDER BY default_rate_rank;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    LoanSequenceNumber,
    Vintage,
    CreditScore,
    Default_36M,

    RANK() OVER (
        PARTITION BY Vintage
        ORDER BY CreditScore DESC
    ) AS credit_score_rank

FROM freddie_master_pd_raw

WHERE CreditScore IS NOT NULL
  AND CreditScore != 9999

LIMIT 50;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    LoanSequenceNumber,
    Vintage,
    CreditScore,
    OriginalLTV,
    OriginalDTI,
    Default_36M,

    COUNT(*) OVER (
        PARTITION BY Vintage
    ) AS vintage_loans,

    SUM(Default_36M) OVER (
        PARTITION BY Vintage
    ) AS vintage_defaults,

    ROUND(
        1.0 * SUM(Default_36M) OVER (
            PARTITION BY Vintage
        )
        /
        COUNT(*) OVER (
            PARTITION BY Vintage
        ),
        4
    ) AS vintage_default_rate

FROM freddie_master_pd_raw

LIMIT 20;
"""

pd.read_sql_query(query, conn)


## 3. Modeling dataset and preprocessing

Prepare the final feature matrix while keeping preprocessing inside a reproducible scikit-learn pipeline.

In [ ]:
query = """
SELECT
    LoanSequenceNumber,
    Vintage,
    CreditScore,

    ROW_NUMBER() OVER (
        PARTITION BY Vintage
        ORDER BY CreditScore DESC
    ) AS loan_rank

FROM freddie_master_pd_raw

WHERE CreditScore IS NOT NULL
  AND CreditScore != 9999

LIMIT 50;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
WITH ranked_loans AS (

    SELECT
        LoanSequenceNumber,
        Vintage,
        CreditScore,
        OriginalLTV,
        OriginalDTI,
        Default_36M,

        ROW_NUMBER() OVER (
            PARTITION BY Vintage
            ORDER BY CreditScore DESC
        ) AS rn

    FROM freddie_master_pd_raw

    WHERE CreditScore IS NOT NULL
      AND CreditScore != 9999
)

SELECT
    LoanSequenceNumber,
    Vintage,
    CreditScore,
    OriginalLTV,
    OriginalDTI,
    Default_36M

FROM ranked_loans

WHERE rn <= 10

ORDER BY Vintage, CreditScore DESC;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
DROP TABLE IF EXISTS pd_model_sql;
"""

conn.execute(query)
conn.commit()

query = """
CREATE TABLE pd_model_sql AS

SELECT
    LoanSequenceNumber,
    Vintage,

    -- Core credit-risk variables
    CreditScore,
    OriginalDTI,
    OriginalLTV,
    OriginalCLTV,
    OriginalUPB,
    OriginalInterestRate,
    OriginalLoanTerm,

    -- Borrower / loan characteristics
    NumberOfBorrowers,
    NumberOfUnits,
    MIPercent,

    -- Categorical variables
    FirstTimeHomebuyerFlag,
    OccupancyStatus,
    Channel,
    PropertyType,
    LoanPurpose,
    AmortizationType,
    PropertyState,

    -- Target
    Default_36M,

    -- SQL-engineered credit-risk bands

    CASE
        WHEN CreditScore < 620 THEN '<620'
        WHEN CreditScore < 680 THEN '620-679'
        WHEN CreditScore < 740 THEN '680-739'
        WHEN CreditScore < 800 THEN '740-799'
        ELSE '800+'
    END AS CreditScoreBand,

    CASE
        WHEN OriginalLTV < 60 THEN '<60'
        WHEN OriginalLTV < 70 THEN '60-69'
        WHEN OriginalLTV < 80 THEN '70-79'
        WHEN OriginalLTV < 90 THEN '80-89'
        ELSE '90+'
    END AS LTVBand,

    CASE
        WHEN OriginalDTI < 20 THEN '<20'
        WHEN OriginalDTI < 30 THEN '20-29'
        WHEN OriginalDTI < 40 THEN '30-39'
        WHEN OriginalDTI < 50 THEN '40-49'
        ELSE '50+'
    END AS DTIBand,

    -- High-risk flag
    CASE
        WHEN CreditScore < 680
         AND OriginalLTV >= 80
         AND OriginalDTI >= 40
        THEN 1
        ELSE 0
    END AS HighRiskFlag

FROM freddie_master_pd_raw

WHERE Default_36M IS NOT NULL;
"""

conn.execute(query)
conn.commit()

print("pd_model_sql created successfully.")


In [ ]:
query = """
SELECT *
FROM pd_model_sql
LIMIT 10;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT COUNT(*) AS modelling_loans
FROM pd_model_sql;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    Default_36M,
    COUNT(*) AS loans,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM pd_model_sql),
        2
    ) AS percentage
FROM pd_model_sql
GROUP BY Default_36M
ORDER BY Default_36M;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    HighRiskFlag,
    COUNT(*) AS loans,
    SUM(Default_36M) AS defaults,
    ROUND(
        100.0 * SUM(Default_36M) / COUNT(*),
        2
    ) AS default_rate_pct
FROM pd_model_sql
GROUP BY HighRiskFlag
ORDER BY HighRiskFlag DESC;
"""

pd.read_sql_query(query, conn)


In [ ]:
# Pull the final SQL modelling table into pandas

model_df = pd.read_sql_query("""
    SELECT *
    FROM pd_model_sql
""", conn)

print("Shape:", model_df.shape)
print("\nColumns:")
print(model_df.columns.tolist())


In [ ]:
model_df.info()


In [ ]:
print(model_df['Default_36M'].value_counts())
print()
print(model_df['Default_36M'].value_counts(normalize=True))


In [ ]:
missing = (
    model_df.isna()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]


In [ ]:
print("Shape:", model_df.shape)

print("\nTarget:")
print(model_df["Default_36M"].value_counts())

print("\nMissing values:")
print(model_df.isna().sum().sort_values(ascending=False).head(10))


In [ ]:
print("\nData types:")
print(model_df.dtypes)


In [ ]:
numeric_cols = [
    "Vintage",
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "Default_36M",
    "HighRiskFlag"
]

for col in numeric_cols:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

print(model_df[numeric_cols].dtypes)


In [ ]:
print("Missing values after conversion:\n")

missing = model_df[numeric_cols].isna().sum()

print(missing[missing > 0])


In [ ]:
model_df["CreditScore"] = model_df["CreditScore"].replace(9999, pd.NA)
model_df["OriginalDTI"] = model_df["OriginalDTI"].replace(999, pd.NA)

print("CreditScore missing:", model_df["CreditScore"].isna().sum())
print("DTI missing:", model_df["OriginalDTI"].isna().sum())


In [ ]:
model_df[
    [
        "CreditScore",
        "OriginalDTI",
        "OriginalLTV",
        "OriginalCLTV",
        "OriginalUPB",
        "OriginalInterestRate",
        "OriginalLoanTerm",
        "NumberOfBorrowers",
        "Default_36M"
    ]
].describe()


In [ ]:
missing_check = pd.DataFrame({
    "CreditScore_missing": model_df["CreditScore"].isna(),
    "DTI_missing": model_df["OriginalDTI"].isna(),
    "CLTV_missing": model_df["OriginalCLTV"].isna(),
    "Default_36M": model_df["Default_36M"]
})

print("CreditScore missing:")
print(missing_check.loc[
    missing_check["CreditScore_missing"],
    "Default_36M"
].value_counts())

print("\nDTI missing:")
print(missing_check.loc[
    missing_check["DTI_missing"],
    "Default_36M"
].value_counts())

print("\nCLTV missing:")
print(missing_check.loc[
    missing_check["CLTV_missing"],
    "Default_36M"
].value_counts())


In [ ]:
categorical_cols = [
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState"
]

for col in categorical_cols:
    print(f"\n{col}")
    print(model_df[col].value_counts(dropna=False).head(15))


In [ ]:
from sklearn.model_selection import train_test_split

# Features and target
X = model_df.drop(columns=["Default_36M"])
y = model_df["Default_36M"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining default rate:")
print(y_train.mean())

print("\nTest default rate:")
print(y_test.mean())


## 4. Predictive models

Compare a linear probability-of-default benchmark with a nonlinear XGBoost model.

In [ ]:
numeric_features = [
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent"
]

categorical_features = [
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "PropertyState"
]

print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            drop="first"
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor ready.")


In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

logistic_model.fit(X_train, y_train)

print("Logistic regression fitted successfully.")


In [ ]:
y_pred_prob = logistic_model.predict_proba(X_test)[:, 1]

print("First 10 predicted PDs:")
print(y_pred_prob[:10])


In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

roc_auc = roc_auc_score(y_test, y_pred_prob)
pr_auc = average_precision_score(y_test, y_pred_prob)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")


In [ ]:
import numpy as np

def calculate_ks(y_true, y_prob):
    data = pd.DataFrame({
        "actual": y_true.values,
        "probability": y_prob
    }).sort_values("probability", ascending=False)

    total_bad = (data["actual"] == 1).sum()
    total_good = (data["actual"] == 0).sum()

    data["cum_bad"] = (
        (data["actual"] == 1).cumsum() / total_bad
    )

    data["cum_good"] = (
        (data["actual"] == 0).cumsum() / total_good
    )

    data["ks"] = abs(data["cum_bad"] - data["cum_good"])

    return data["ks"].max()

ks = calculate_ks(y_test, y_pred_prob)

print(f"KS Statistic: {ks:.4f}")


In [ ]:
y_pred = (y_pred_prob >= 0.50).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))


In [ ]:
evaluation_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted_PD": y_pred_prob
})

evaluation_df["PD_Decile"] = pd.qcut(
    evaluation_df["Predicted_PD"],
    10,
    labels=False,
    duplicates="drop"
) + 1

decile_table = (
    evaluation_df
    .groupby("PD_Decile")
    .agg(
        Loans=("Actual", "count"),
        Defaults=("Actual", "sum"),
        Average_PD=("Predicted_PD", "mean"),
        Actual_Default_Rate=("Actual", "mean")
    )
    .reset_index()
)

decile_table["Actual_Default_Rate"] *= 100
decile_table["Average_PD"] *= 100

decile_table


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc = roc_auc_score(y_test, y_pred_prob)
pr_auc = average_precision_score(y_test, y_pred_prob)
ks = calculate_ks(y_test, y_pred_prob)

print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")
print(f"KS      : {ks:.4f}")


In [ ]:
top_decile = decile_table.iloc[-1]

overall_default_rate = y_test.mean() * 100
top_decile_default_rate = top_decile["Actual_Default_Rate"]

lift = top_decile_default_rate / overall_default_rate

print(f"Overall default rate: {overall_default_rate:.2f}%")
print(f"Top-decile default rate: {top_decile_default_rate:.2f}%")
print(f"Top-decile lift: {lift:.2f}x")


In [ ]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)


In [ ]:
scale_pos_weight = (
    (y_train == 0).sum() /
    (y_train == 1).sum()
)

print("Scale pos weight:", scale_pos_weight)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            categorical_features
        )
    ]
)


In [ ]:
xgb_model = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),

        ("model", xgb.XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            reg_alpha=0.1,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="auc",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

xgb_model.fit(X_train, y_train)

print("XGBoost fitted successfully.")


In [ ]:
xgb_pred_prob = xgb_model.predict_proba(X_test)[:, 1]

print("First 10 XGBoost predictions:")
print(xgb_pred_prob[:10])


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

xgb_auc = roc_auc_score(y_test, xgb_pred_prob)
xgb_pr_auc = average_precision_score(y_test, xgb_pred_prob)
xgb_ks = calculate_ks(y_test, xgb_pred_prob)

print(f"XGBoost ROC-AUC : {xgb_auc:.4f}")
print(f"XGBoost PR-AUC  : {xgb_pr_auc:.4f}")
print(f"XGBoost KS      : {xgb_ks:.4f}")

print("\n--- Logistic Baseline ---")
print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")
print(f"KS      : {ks:.4f}")


In [ ]:
xgb_eval = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted_PD": xgb_pred_prob
})

xgb_eval["PD_Decile"] = pd.qcut(
    xgb_eval["Predicted_PD"],
    10,
    labels=False,
    duplicates="drop"
) + 1

xgb_decile_table = (
    xgb_eval
    .groupby("PD_Decile")
    .agg(
        Loans=("Actual", "count"),
        Defaults=("Actual", "sum"),
        Average_PD=("Predicted_PD", "mean"),
        Actual_Default_Rate=("Actual", "mean")
    )
    .reset_index()
)

xgb_decile_table["Average_PD"] *= 100
xgb_decile_table["Actual_Default_Rate"] *= 100

xgb_decile_table


In [ ]:
xgb_top_decile = xgb_decile_table.iloc[-1]

xgb_top_rate = xgb_top_decile["Actual_Default_Rate"]
overall_rate = y_test.mean() * 100

xgb_lift = xgb_top_rate / overall_rate

print(f"Overall default rate: {overall_rate:.2f}%")
print(f"XGBoost top-decile default rate: {xgb_top_rate:.2f}%")
print(f"XGBoost top-decile lift: {xgb_lift:.2f}x")


In [ ]:
# Extract the fitted XGBoost model
xgb_fitted = xgb_model.named_steps["model"]

# Get feature names after preprocessing
feature_names = xgb_model.named_steps["preprocessor"].get_feature_names_out()

# Feature importance
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_fitted.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df.head(20)


In [ ]:
import matplotlib.pyplot as plt

top_features = importance_df.head(15).sort_values("Importance")

plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("XGBoost Feature Importance")
plt.ylabel("Feature")
plt.title("Top 15 Drivers of Predicted Default Risk")
plt.tight_layout()
plt.show()


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score

xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.02, 0.05, 0.08, 0.10],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 2, 5]
}

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=20,
    scoring="average_precision",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("BEST PARAMETERS:")
print(search.best_params_)

print("\nBEST CV PR-AUC:")
print(search.best_score_)


In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# XGBOOST HYPERPARAMETER TUNING

xgb_tuned = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

# Put preprocessing + XGBoost together
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_tuned)
])

# Parameter grid
param_grid = {
    "model__n_estimators": [200, 300, 400, 500, 600],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__learning_rate": [0.02, 0.03, 0.05, 0.08, 0.10],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__gamma": [0, 0.1, 0.3, 0.5],
    "model__reg_alpha": [0, 0.1, 0.5, 1],
    "model__reg_lambda": [1, 2, 5, 10]
}

search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_grid,
    n_iter=30,
    scoring="roc_auc",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("BEST PARAMETERS:")
print(search.best_params_)

print("\nBEST CV ROC-AUC:")
print(search.best_score_)


In [ ]:
# EVALUATE TUNED XGBOOST

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

best_xgb = search.best_estimator_

y_pred_prob_tuned = best_xgb.predict_proba(X_test)[:, 1]

roc_auc_tuned = roc_auc_score(y_test, y_pred_prob_tuned)
pr_auc_tuned = average_precision_score(y_test, y_pred_prob_tuned)

print("TUNED XGBOOST")
print("-----------------------------")
print(f"ROC-AUC : {roc_auc_tuned:.4f}")
print(f"PR-AUC  : {pr_auc_tuned:.4f}")


In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob_tuned)

ks_tuned = max(tpr - fpr)

print(f"KS      : {ks_tuned:.4f}")


In [ ]:
evaluation_tuned = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted_PD": y_pred_prob_tuned
})

evaluation_tuned["PD_Decile"] = pd.qcut(
    evaluation_tuned["Predicted_PD"],
    10,
    labels=False,
    duplicates="drop"
) + 1

decile_tuned = (
    evaluation_tuned
    .groupby("PD_Decile")
    .agg(
        Loans=("Actual", "count"),
        Defaults=("Actual", "sum"),
        Average_PD=("Predicted_PD", "mean"),
        Actual_Default_Rate=("Actual", "mean")
    )
    .reset_index()
)

decile_tuned["Average_PD"] *= 100
decile_tuned["Actual_Default_Rate"] *= 100

print(decile_tuned)


In [ ]:
overall_default_rate = y_test.mean() * 100

top_decile_default_rate = decile_tuned.iloc[-1]["Actual_Default_Rate"]

lift_tuned = top_decile_default_rate / overall_default_rate

print(f"Overall default rate: {overall_default_rate:.2f}%")
print(f"Tuned XGBoost top-decile default rate: {top_decile_default_rate:.2f}%")
print(f"Tuned XGBoost top-decile lift: {lift_tuned:.2f}x")


In [ ]:
# OUT-OF-TIME VALIDATION
# Train: 2018-2021
# Test : 2022

print("Vintage distribution:")
print(model_df["Vintage"].value_counts().sort_index())


In [ ]:
# OUT-OF-TIME VALIDATION — TRAIN 2018-2021, TEST 2022

import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

# 1. CREATE OUT-OF-TIME TRAIN / TEST SETS

oot_train = model_df[model_df["Vintage"] < 2022].copy()
oot_test  = model_df[model_df["Vintage"] == 2022].copy()

X_oot_train = oot_train.drop(columns=["Default_36M"])
y_oot_train = oot_train["Default_36M"].astype(int)

X_oot_test = oot_test.drop(columns=["Default_36M"])
y_oot_test = oot_test["Default_36M"].astype(int)

print("=" * 60)
print("OUT-OF-TIME SPLIT")
print("=" * 60)

print("Train shape:", X_oot_train.shape)
print("Test shape :", X_oot_test.shape)

print("\nTrain vintages:")
print(oot_train["Vintage"].value_counts().sort_index())

print("\nTest vintage:")
print(oot_test["Vintage"].value_counts().sort_index())

print(f"\nTrain default rate: {y_oot_train.mean():.4%}")
print(f"Test default rate : {y_oot_test.mean():.4%}")

# 2. TUNED XGBOOST — SAME BEST PARAMETERS

oot_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,

    n_estimators=600,
    max_depth=5,
    learning_rate=0.02,
    subsample=0.7,
    colsample_bytree=1.0,
    min_child_weight=3,
    gamma=0,
    reg_alpha=1,
    reg_lambda=2
)

# 3. PREPROCESSING + XGBOOST PIPELINE

oot_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", oot_xgb)
])

# 4. TRAIN ON 2018-2021 ONLY

print("\n" + "=" * 60)
print("TRAINING OOT MODEL")
print("=" * 60)

oot_pipeline.fit(X_oot_train, y_oot_train)

print("OOT model trained successfully.")

# 5. PREDICT 2022

y_oot_prob = oot_pipeline.predict_proba(X_oot_test)[:, 1]

# 6. OOT PERFORMANCE

oot_roc_auc = roc_auc_score(y_oot_test, y_oot_prob)
oot_pr_auc = average_precision_score(y_oot_test, y_oot_prob)

fpr, tpr, thresholds = roc_curve(y_oot_test, y_oot_prob)
oot_ks = max(tpr - fpr)

print("\n" + "=" * 60)
print("OUT-OF-TIME PERFORMANCE — 2022")
print("=" * 60)

print(f"ROC-AUC : {oot_roc_auc:.4f}")
print(f"PR-AUC  : {oot_pr_auc:.4f}")
print(f"KS      : {oot_ks:.4f}")

# 7. OOT PD DECILES

oot_eval = pd.DataFrame({
    "Actual": y_oot_test.values,
    "Predicted_PD": y_oot_prob
})

oot_eval["PD_Decile"] = pd.qcut(
    oot_eval["Predicted_PD"],
    10,
    labels=False,
    duplicates="drop"
) + 1

oot_decile = (
    oot_eval
    .groupby("PD_Decile")
    .agg(
        Loans=("Actual", "count"),
        Defaults=("Actual", "sum"),
        Average_PD=("Predicted_PD", "mean"),
        Actual_Default_Rate=("Actual", "mean")
    )
    .reset_index()
)

oot_decile["Average_PD"] *= 100
oot_decile["Actual_Default_Rate"] *= 100

print("\n" + "=" * 60)
print("2022 PD DECILE TABLE")
print("=" * 60)

print(oot_decile.to_string(index=False))

# 8. TOP-DECILE LIFT

overall_oot_rate = y_oot_test.mean() * 100

top_oot_rate = oot_decile.iloc[-1]["Actual_Default_Rate"]

oot_lift = top_oot_rate / overall_oot_rate

print("\n" + "=" * 60)
print("2022 RISK RANKING")
print("=" * 60)

print(f"2022 overall default rate       : {overall_oot_rate:.2f}%")
print(f"2022 top-decile default rate    : {top_oot_rate:.2f}%")
print(f"2022 top-decile lift            : {oot_lift:.2f}x")

# 9. FINAL COMPARISON WITH RANDOM TEST RESULT

print("\n" + "=" * 60)
print("RANDOM TEST vs OUT-OF-TIME TEST")
print("=" * 60)

print(f"Random-test ROC-AUC             : {roc_auc_tuned:.4f}")
print(f"OOT 2022 ROC-AUC                : {oot_roc_auc:.4f}")

print(f"\nRandom-test PR-AUC              : {pr_auc_tuned:.4f}")
print(f"OOT 2022 PR-AUC                 : {oot_pr_auc:.4f}")

print(f"\nRandom-test KS                  : {ks_tuned:.4f}")
print(f"OOT 2022 KS                     : {oot_ks:.4f}")

print(f"\nRandom-test top-decile lift     : {lift_tuned:.2f}x")
print(f"OOT 2022 top-decile lift        : {oot_lift:.2f}x")


In [ ]:
# VINTAGE-BY-VINTAGE MODEL STABILITY

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

vintage_results = []

for vintage in sorted(model_df["Vintage"].unique()):

    vintage_data = model_df[model_df["Vintage"] == vintage].copy()

    X_vintage = vintage_data.drop(columns=["Default_36M"])
    y_vintage = vintage_data["Default_36M"].astype(int)

    # Predict using the already trained random-split model
    prob_vintage = best_xgb.predict_proba(X_vintage)[:, 1]

    auc = roc_auc_score(y_vintage, prob_vintage)
    pr = average_precision_score(y_vintage, prob_vintage)

    fpr, tpr, _ = roc_curve(y_vintage, prob_vintage)
    ks = max(tpr - fpr)

    # Decile lift
    temp = pd.DataFrame({
        "Actual": y_vintage.values,
        "PD": prob_vintage
    })

    temp["Decile"] = pd.qcut(
        temp["PD"],
        10,
        labels=False,
        duplicates="drop"
    ) + 1

    top_decile = temp[temp["Decile"] == 10]

    overall_rate = y_vintage.mean()
    top_rate = top_decile["Actual"].mean()

    lift = top_rate / overall_rate if overall_rate > 0 else np.nan

    vintage_results.append({
        "Vintage": vintage,
        "Loans": len(vintage_data),
        "Default_Rate_%": y_vintage.mean() * 100,
        "ROC_AUC": auc,
        "PR_AUC": pr,
        "KS": ks,
        "Top_Decile_Lift": lift
    })

vintage_results_df = pd.DataFrame(vintage_results)

print(vintage_results_df.to_string(index=False))


## 5. Weight of Evidence / Information Value diagnostics

Use WOE/IV as credit-risk feature diagnostics and interpretability tools, alongside out-of-time validation.

In [ ]:
# WOE + IV ANALYSIS

import pandas as pd
import numpy as np

# VARIABLES FOR WOE / IV

woe_features = [
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent"
]

woe_df = model_df[woe_features + ["Default_36M"]].copy()

# BIN NUMERIC VARIABLES

for col in woe_features:

    # Special handling for categorical/discrete variables
    if col in ["NumberOfBorrowers", "NumberOfUnits", "MIPercent"]:
        woe_df[col] = woe_df[col].astype(str)

    else:
        try:
            woe_df[col + "_bin"] = pd.qcut(
                woe_df[col],
                q=10,
                duplicates="drop"
            )
        except:
            woe_df[col + "_bin"] = woe_df[col].astype(str)

# WOE / IV FUNCTION

def calculate_woe_iv(data, variable, target="Default_36M"):

    temp = data[[variable, target]].copy()

    temp[variable] = temp[variable].astype(str)

    grouped = (
        temp.groupby(variable, dropna=False)[target]
        .agg(["count", "sum"])
        .reset_index()
    )

    grouped.columns = [
        "Bin",
        "Total",
        "Defaults"
    ]

    grouped["Non_Defaults"] = (
        grouped["Total"] - grouped["Defaults"]
    )

    total_defaults = grouped["Defaults"].sum()
    total_non_defaults = grouped["Non_Defaults"].sum()

    # Avoid division by zero
    grouped["Default_Dist"] = (
        grouped["Defaults"] / total_defaults
    ).replace(0, 0.0001)

    grouped["NonDefault_Dist"] = (
        grouped["Non_Defaults"] / total_non_defaults
    ).replace(0, 0.0001)

    grouped["WOE"] = np.log(
        grouped["NonDefault_Dist"] /
        grouped["Default_Dist"]
    )

    grouped["IV"] = (
        grouped["NonDefault_Dist"] -
        grouped["Default_Dist"]
    ) * grouped["WOE"]

    iv = grouped["IV"].sum()

    return grouped, iv

# CALCULATE IV FOR ALL VARIABLES

iv_results = []

woe_tables = {}

for col in woe_features:

    if col in ["NumberOfBorrowers", "NumberOfUnits", "MIPercent"]:
        variable = col
    else:
        variable = col + "_bin"

    table, iv = calculate_woe_iv(
        woe_df,
        variable
    )

    woe_tables[col] = table

    iv_results.append({
        "Variable": col,
        "IV": iv
    })

iv_table = pd.DataFrame(iv_results)

iv_table = iv_table.sort_values(
    "IV",
    ascending=False
).reset_index(drop=True)

# INTERPRET IV

def iv_strength(iv):

    if iv < 0.02:
        return "Not useful"
    elif iv < 0.10:
        return "Weak"
    elif iv < 0.30:
        return "Medium"
    elif iv < 0.50:
        return "Strong"
    else:
        return "Very strong"

iv_table["Strength"] = iv_table["IV"].apply(iv_strength)

print("=" * 70)
print("WEIGHT OF EVIDENCE / INFORMATION VALUE")
print("=" * 70)

print(
    iv_table.to_string(index=False)
)

# SHOW WOE TABLES FOR TOP 3 VARIABLES

print("\n" + "=" * 70)
print("TOP 3 VARIABLES — WOE TABLES")
print("=" * 70)

for variable in iv_table.head(3)["Variable"]:

    print(f"\n--- {variable} ---")

    print(
        woe_tables[variable].to_string(index=False)
    )


In [ ]:
# XGBOOST FEATURE IMPORTANCE

import pandas as pd
import matplotlib.pyplot as plt

# GET PREPROCESSED FEATURE NAMES

preprocessor_fitted = best_xgb.named_steps["preprocessor"]
xgb_model = best_xgb.named_steps["model"]

feature_names = preprocessor_fitted.get_feature_names_out()

importance_values = xgb_model.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})

feature_importance = (
    feature_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

# DISPLAY TOP 20

print("=" * 70)
print("XGBOOST FEATURE IMPORTANCE — TOP 20")
print("=" * 70)

print(
    feature_importance.head(20).to_string(index=False)
)

# PLOT TOP 15

top_features = feature_importance.head(15).sort_values(
    "Importance"
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 15 XGBoost Features")

plt.tight_layout()
plt.show()


In [ ]:
# EXPECTED LOSS — EAD PROXY

el_df = model_df.loc[
    X_test.index
].copy()

el_df["Predicted_PD"] = y_pred_prob_tuned

# EAD PROXY

el_df["EAD"] = el_df["OriginalUPB"].astype(float)

# LGD ASSUMPTION

# Base-case illustrative LGD assumption.
# We will perform sensitivity analysis rather than claiming
# this is an observed LGD.

LGD = 0.45

el_df["LGD"] = LGD

# EXPECTED LOSS

el_df["Expected_Loss"] = (
    el_df["Predicted_PD"] *
    el_df["LGD"] *
    el_df["EAD"]
)

# PORTFOLIO SUMMARY

total_ead = el_df["EAD"].sum()

total_expected_loss = el_df["Expected_Loss"].sum()

expected_loss_rate = (
    total_expected_loss / total_ead
)

print("=" * 70)
print("EXPECTED LOSS ANALYSIS")
print("=" * 70)

print(f"Total EAD: ₹{total_ead:,.2f}")
print(f"Expected Loss: ₹{total_expected_loss:,.2f}")
print(
    f"Expected Loss Rate: {expected_loss_rate:.2%}"
)

print(f"\nAssumed LGD: {LGD:.0%}")
print("EAD proxy: OriginalUPB")
print("PD: Tuned XGBoost predicted probability")


In [ ]:
# EXPECTED LOSS BY RISK BAND

el_df["Risk_Band"] = pd.cut(
    el_df["Predicted_PD"],
    bins=[
        -np.inf,
        0.02,
        0.05,
        0.10,
        np.inf
    ],
    labels=[
        "Low Risk",
        "Moderate Risk",
        "High Risk",
        "Very High Risk"
    ]
)

el_summary = (
    el_df
    .groupby("Risk_Band", observed=True)
    .agg(
        Loans=("LoanSequenceNumber", "count"),
        EAD=("EAD", "sum"),
        Average_PD=("Predicted_PD", "mean"),
        Expected_Loss=("Expected_Loss", "sum")
    )
    .reset_index()
)

el_summary["Average_PD"] *= 100

el_summary["Expected_Loss_Rate"] = (
    el_summary["Expected_Loss"] /
    el_summary["EAD"]
)

el_summary["Expected_Loss_Rate"] *= 100

print("=" * 70)
print("EXPECTED LOSS BY RISK BAND")
print("=" * 70)

print(
    el_summary.to_string(index=False)
)


In [ ]:
# PD CALIBRATION CHECK — TUNED XGBOOST

from sklearn.calibration import calibration_curve
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Actual vs predicted PD
calib_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted_PD": y_pred_prob_tuned
})

# 10 equal-frequency bins
calib_df["PD_Bin"] = pd.qcut(
    calib_df["Predicted_PD"],
    10,
    labels=False,
    duplicates="drop"
) + 1

calibration_table = (
    calib_df
    .groupby("PD_Bin")
    .agg(
        Loans=("Actual", "count"),
        Defaults=("Actual", "sum"),
        Average_PD=("Predicted_PD", "mean"),
        Actual_Default_Rate=("Actual", "mean")
    )
    .reset_index()
)

calibration_table["Average_PD_%"] = (
    calibration_table["Average_PD"] * 100
)

calibration_table["Actual_Default_Rate_%"] = (
    calibration_table["Actual_Default_Rate"] * 100
)

print("=" * 70)
print("PD CALIBRATION TABLE")
print("=" * 70)

print(
    calibration_table[
        [
            "PD_Bin",
            "Loans",
            "Defaults",
            "Average_PD_%",
            "Actual_Default_Rate_%"
        ]
    ].round(3).to_string(index=False)
)

# Calibration plot
plt.figure(figsize=(7, 6))

plt.plot(
    calibration_table["Average_PD_%"],
    calibration_table["Actual_Default_Rate_%"],
    marker="o",
    label="XGBoost"
)

min_val = min(
    calibration_table["Average_PD_%"].min(),
    calibration_table["Actual_Default_Rate_%"].min()
)

max_val = max(
    calibration_table["Average_PD_%"].max(),
    calibration_table["Actual_Default_Rate_%"].max()
)

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    label="Perfect Calibration"
)

plt.xlabel("Average Predicted PD (%)")
plt.ylabel("Actual Default Rate (%)")
plt.title("XGBoost PD Calibration")
plt.legend()
plt.grid(alpha=0.3)

plt.show()


In [ ]:
# FINAL PD RISK BANDS

risk_df = pd.DataFrame({
    "Predicted_PD": y_pred_prob_tuned,
    "Default": y_test.values
})

# Convert PD to percentage
risk_df["PD_%"] = risk_df["Predicted_PD"] * 100

# Define risk bands
risk_df["Risk_Band"] = pd.cut(
    risk_df["PD_%"],
    bins=[-np.inf, 2, 5, 10, np.inf],
    labels=[
        "Low Risk",
        "Moderate Risk",
        "High Risk",
        "Very High Risk"
    ]
)

# Summary
risk_summary = (
    risk_df
    .groupby("Risk_Band", observed=False)
    .agg(
        Loans=("Default", "count"),
        Defaults=("Default", "sum"),
        Average_PD=("PD_%", "mean"),
        Actual_Default_Rate=("Default", "mean")
    )
    .reset_index()
)

risk_summary["Actual_Default_Rate"] *= 100

print("=" * 70)
print("FINAL PD RISK BAND ANALYSIS")
print("=" * 70)

print(
    risk_summary[
        [
            "Risk_Band",
            "Loans",
            "Defaults",
            "Average_PD",
            "Actual_Default_Rate"
        ]
    ].round(2).to_string(index=False)
)


In [ ]:
import sqlite3

db_path = r"./data/raw/freddie_pd.db"

conn = sqlite3.connect(db_path)

print("Connected to SQLite:", db_path)

sql_create_view = """

DROP VIEW IF EXISTS freddie_model_sql;

CREATE VIEW freddie_model_sql AS

SELECT

    Vintage,

    -- Numerical variables
    CASE
        WHEN CAST(CreditScore AS REAL) = 9999 THEN NULL
        ELSE CAST(CreditScore AS REAL)
    END AS CreditScore,

    CASE
        WHEN CAST(OriginalDTI AS REAL) = 999 THEN NULL
        ELSE CAST(OriginalDTI AS REAL)
    END AS OriginalDTI,

    CAST(OriginalLTV AS REAL) AS OriginalLTV,

    CAST(OriginalCLTV AS REAL) AS OriginalCLTV,

    CAST(OriginalUPB AS REAL) AS OriginalUPB,

    CAST(OriginalInterestRate AS REAL) AS OriginalInterestRate,

    CAST(OriginalLoanTerm AS INTEGER) AS OriginalLoanTerm,

    CAST(NumberOfBorrowers AS INTEGER) AS NumberOfBorrowers,

    CAST(NumberOfUnits AS INTEGER) AS NumberOfUnits,

    CAST(MIPercent AS REAL) AS MIPercent,

    -- Categorical variables
    FirstTimeHomebuyerFlag,

    OccupancyStatus,

    Channel,

    PropertyType,

    LoanPurpose,

    AmortizationType,

    PropertyState,

    -- Target
    CAST(Default_36M AS INTEGER) AS Default_36M,

    -- SQL-engineered risk flags

    CASE
        WHEN CAST(CreditScore AS REAL) < 680 THEN 1
        ELSE 0
    END AS LowCreditFlag,

    CASE
        WHEN CAST(OriginalLTV AS REAL) >= 80 THEN 1
        ELSE 0
    END AS HighLTVFlag,

    CASE
        WHEN CAST(OriginalDTI AS REAL) >= 43 THEN 1
        ELSE 0
    END AS HighDTIFlag,

    CASE
        WHEN CAST(OriginalCLTV AS REAL) >= 80 THEN 1
        ELSE 0
    END AS HighCLTVFlag,

    CASE
        WHEN CAST(OriginalInterestRate AS REAL) >= 5 THEN 1
        ELSE 0
    END AS HighRateFlag

FROM freddie_master_pd_raw

WHERE Default_36M IS NOT NULL;

"""

# Execute the SQL
conn.executescript(sql_create_view)

print("SQL modelling view created successfully.")


In [ ]:
# VERIFY SQL MODELLING VIEW

sql_check = """
SELECT *
FROM freddie_model_sql
LIMIT 5;
"""

sql_preview = pd.read_sql_query(sql_check, conn)

print("SQL VIEW SHAPE PREVIEW:")
print(sql_preview.shape)

display(sql_preview)


In [ ]:
# SQL DATA QUALITY CHECK

sql_quality = """
SELECT

    COUNT(*) AS Total_Loans,

    SUM(Default_36M) AS Defaults,

    ROUND(
        100.0 * AVG(Default_36M),
        4
    ) AS Default_Rate_Percent,

    SUM(
        CASE WHEN CreditScore IS NULL THEN 1 ELSE 0 END
    ) AS Missing_CreditScore,

    SUM(
        CASE WHEN OriginalDTI IS NULL THEN 1 ELSE 0 END
    ) AS Missing_DTI,

    SUM(
        CASE WHEN OriginalCLTV IS NULL THEN 1 ELSE 0 END
    ) AS Missing_CLTV

FROM freddie_model_sql;
"""

quality_check = pd.read_sql_query(sql_quality, conn)

display(quality_check)


In [ ]:

sql_raw_check = """
SELECT
    SUM(CASE WHEN CreditScore = '9999' THEN 1 ELSE 0 END)
        AS CreditScore_9999,

    SUM(CASE WHEN OriginalDTI = '999' THEN 1 ELSE 0 END)
        AS DTI_999,

    SUM(CASE WHEN OriginalCLTV IS NULL OR OriginalCLTV = '' THEN 1 ELSE 0 END)
        AS CLTV_Missing

FROM freddie_master_pd_raw;
"""

raw_check = pd.read_sql_query(sql_raw_check, conn)

display(raw_check)


In [ ]:

for col in ["CreditScore", "OriginalDTI", "OriginalCLTV"]:

    print("\n" + "="*60)
    print(col)
    print("="*60)

    query = f"""
    SELECT
        typeof({col}) AS SQLite_Type,
        {col} AS Value,
        COUNT(*) AS Count
    FROM freddie_master_pd_raw
    GROUP BY typeof({col}), {col}
    ORDER BY Count DESC
    LIMIT 15;
    """

    result = pd.read_sql_query(query, conn)
    display(result)


In [ ]:

for col in ["CreditScore", "OriginalDTI", "OriginalCLTV"]:

    print("\n" + "="*60)
    print(col)
    print("="*60)

    query = f"""
    SELECT
        {col} AS Value,
        COUNT(*) AS Count
    FROM freddie_master_pd_raw
    WHERE
        TRIM({col}) = ''
        OR {col} IS NULL
        OR {col} GLOB '*[^0-9.]*'
    GROUP BY {col}
    ORDER BY Count DESC;
    """

    result = pd.read_sql_query(query, conn)
    display(result)


In [ ]:

sql_create_view = """

DROP VIEW IF EXISTS freddie_model_sql;

CREATE VIEW freddie_model_sql AS

SELECT

    -- ========================================================
    -- NUMERICAL VARIABLES
    -- ========================================================

    CAST(Vintage AS INTEGER) AS Vintage,

    CASE
        WHEN TRIM(CreditScore) = '' THEN NULL
        ELSE CAST(CreditScore AS REAL)
    END AS CreditScore,

    CASE
        WHEN TRIM(OriginalDTI) = '' THEN NULL
        ELSE CAST(OriginalDTI AS REAL)
    END AS OriginalDTI,

    CASE
        WHEN TRIM(OriginalLTV) = '' THEN NULL
        ELSE CAST(OriginalLTV AS REAL)
    END AS OriginalLTV,

    CASE
        WHEN TRIM(OriginalCLTV) = '' THEN NULL
        ELSE CAST(OriginalCLTV AS REAL)
    END AS OriginalCLTV,

    CAST(OriginalUPB AS REAL) AS OriginalUPB,

    CAST(OriginalInterestRate AS REAL) AS OriginalInterestRate,

    CAST(OriginalLoanTerm AS INTEGER) AS OriginalLoanTerm,

    CAST(NumberOfBorrowers AS INTEGER) AS NumberOfBorrowers,

    CAST(NumberOfUnits AS INTEGER) AS NumberOfUnits,

    CAST(MIPercent AS REAL) AS MIPercent,

    -- ========================================================
    -- CATEGORICAL VARIABLES
    -- ========================================================

    FirstTimeHomebuyerFlag,
    OccupancyStatus,
    Channel,
    PropertyType,
    LoanPurpose,
    AmortizationType,
    PropertyState,

    -- ========================================================
    -- TARGET
    -- ========================================================

    CAST(Default_36M AS INTEGER) AS Default_36M,

    -- ========================================================
    -- SQL-ENGINEERED RISK FEATURES
    -- ========================================================

    CASE
        WHEN CAST(CreditScore AS REAL) < 680 THEN 1
        ELSE 0
    END AS LowCreditFlag,

    CASE
        WHEN CAST(OriginalLTV AS REAL) >= 80 THEN 1
        ELSE 0
    END AS HighLTVFlag,

    CASE
        WHEN CAST(OriginalDTI AS REAL) >= 43 THEN 1
        ELSE 0
    END AS HighDTIFlag,

    CASE
        WHEN CAST(OriginalCLTV AS REAL) >= 80 THEN 1
        ELSE 0
    END AS HighCLTVFlag,

    CASE
        WHEN CAST(OriginalInterestRate AS REAL) >= 5 THEN 1
        ELSE 0
    END AS HighRateFlag,

    -- Combined underwriting risk flag
    CASE
        WHEN
            CAST(CreditScore AS REAL) < 680
            AND CAST(OriginalLTV AS REAL) >= 80
        THEN 1
        ELSE 0
    END AS HighCreditLTVFlag,

    -- High DTI + high LTV combination
    CASE
        WHEN
            CAST(OriginalDTI AS REAL) >= 43
            AND CAST(OriginalLTV AS REAL) >= 80
        THEN 1
        ELSE 0
    END AS HighDTILTVFlag

FROM freddie_master_pd_raw

WHERE Default_36M IS NOT NULL;

"""

conn.executescript(sql_create_view)

print("SQL modelling view rebuilt successfully.")


In [ ]:

sql_quality = """

SELECT

    COUNT(*) AS Total_Loans,

    SUM(Default_36M) AS Defaults,

    ROUND(
        100.0 * AVG(Default_36M),
        4
    ) AS Default_Rate_Percent,

    SUM(
        CASE WHEN CreditScore IS NULL THEN 1 ELSE 0 END
    ) AS Missing_CreditScore,

    SUM(
        CASE WHEN OriginalDTI IS NULL THEN 1 ELSE 0 END
    ) AS Missing_DTI,

    SUM(
        CASE WHEN OriginalCLTV IS NULL THEN 1 ELSE 0 END
    ) AS Missing_CLTV

FROM freddie_model_sql;

"""

quality_check = pd.read_sql_query(sql_quality, conn)

display(quality_check)


In [ ]:

sql_risk_analysis = """

SELECT

    CASE
        WHEN CreditScore < 680 THEN 'Low Credit'
        WHEN CreditScore < 740 THEN 'Moderate Credit'
        WHEN CreditScore < 780 THEN 'Good Credit'
        ELSE 'High Credit'
    END AS Credit_Risk_Band,

    COUNT(*) AS Loans,

    SUM(Default_36M) AS Defaults,

    ROUND(
        100.0 * AVG(Default_36M),
        2
    ) AS Default_Rate_Percent,

    ROUND(
        AVG(OriginalInterestRate),
        3
    ) AS Average_Interest_Rate,

    ROUND(
        AVG(OriginalLTV),
        2
    ) AS Average_LTV,

    ROUND(
        AVG(OriginalDTI),
        2
    ) AS Average_DTI

FROM freddie_model_sql

GROUP BY Credit_Risk_Band

ORDER BY Default_Rate_Percent DESC;

"""

risk_analysis = pd.read_sql_query(
    sql_risk_analysis,
    conn
)

display(risk_analysis)


In [ ]:

sql_combined_risk = """

SELECT

    CASE
        WHEN CreditScore < 680
             AND OriginalLTV >= 80
             AND OriginalDTI >= 43
        THEN 'High Risk Combination'

        WHEN CreditScore < 680
             OR OriginalLTV >= 80
             OR OriginalDTI >= 43
        THEN 'Elevated Risk'

        ELSE 'Lower Risk'
    END AS Risk_Group,

    COUNT(*) AS Loans,

    SUM(Default_36M) AS Defaults,

    ROUND(
        100.0 * AVG(Default_36M),
        2
    ) AS Default_Rate_Percent,

    ROUND(
        100.0 * SUM(Default_36M)
        / SUM(SUM(Default_36M)) OVER (),
        2
    ) AS Share_of_All_Defaults

FROM freddie_model_sql

GROUP BY Risk_Group

ORDER BY Default_Rate_Percent DESC;

"""

combined_risk = pd.read_sql_query(
    sql_combined_risk,
    conn
)

display(combined_risk)


In [ ]:

sql_model_data = """

SELECT *

FROM freddie_model_sql

"""

model_sql_df = pd.read_sql_query(
    sql_model_data,
    conn
)

print("=" * 60)
print("SQL → PYTHON MODELLING DATASET")
print("=" * 60)

print("Shape:", model_sql_df.shape)

print("\nColumns:")
print(model_sql_df.columns.tolist())

print("\nTarget distribution:")
print(model_sql_df["Default_36M"].value_counts())

print("\nTarget rate:")
print(model_sql_df["Default_36M"].mean())


In [ ]:

sql_features = [
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

feature_check = model_sql_df[sql_features].agg(
    ["count", "sum", "mean"]
).T

feature_check["Default_Rate_When_Flagged"] = [
    model_sql_df.loc[
        model_sql_df[col] == 1,
        "Default_36M"
    ].mean()
    for col in sql_features
]

display(feature_check)


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# 1. COPY SQL DATASET

final_df = model_sql_df.copy()

# 2. TARGET

y = final_df["Default_36M"].astype(int)

# 3. FEATURES

features = [
    "Vintage",
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",

    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState",

    # SQL ENGINEERED FEATURES
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

X = final_df[features].copy()

print("=" * 60)
print("FINAL SQL → PYTHON MODEL DATASET")
print("=" * 60)

print("Shape:", X.shape)
print("Target rate:", y.mean())

# 4. IDENTIFY NUMERIC / CATEGORICAL

categorical_cols = [
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState"
]

numeric_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

print("\nNumeric features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))

# 5. TRAIN / TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTrain:", X_train.shape)
print("Test :", X_test.shape)

# 6. PREPROCESSING

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# 7. FINAL XGBOOST

xgb_final = XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.02,
    min_child_weight=3,
    subsample=0.7,
    colsample_bytree=1.0,
    reg_alpha=1,
    reg_lambda=2,
    gamma=0,

    objective="binary:logistic",
    eval_metric="auc",

    random_state=42,
    n_jobs=-1
)

final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_final)
    ]
)

# 8. TRAIN

print("\nTraining FINAL XGBoost...")

final_model.fit(
    X_train,
    y_train
)

print("FINAL MODEL TRAINED SUCCESSFULLY")


In [ ]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

y_pred_prob_final = final_model.predict_proba(X_test)[:, 1]

roc_auc_final = roc_auc_score(
    y_test,
    y_pred_prob_final
)

pr_auc_final = average_precision_score(
    y_test,
    y_pred_prob_final
)

fpr, tpr, thresholds = roc_curve(
    y_test,
    y_pred_prob_final
)

ks_final = max(tpr - fpr)

print("=" * 60)
print("FINAL SQL-ENGINEERED XGBOOST")
print("=" * 60)

print(f"ROC-AUC : {roc_auc_final:.4f}")
print(f"PR-AUC  : {pr_auc_final:.4f}")
print(f"KS      : {ks_final:.4f}")

# DECILE ANALYSIS

evaluation_final = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted_PD": y_pred_prob_final
})

evaluation_final["PD_Decile"] = pd.qcut(
    evaluation_final["Predicted_PD"],
    10,
    labels=False,
    duplicates="drop"
) + 1

decile_final = (
    evaluation_final
    .groupby("PD_Decile")
    .agg(
        Loans=("Actual", "count"),
        Defaults=("Actual", "sum"),
        Average_PD=("Predicted_PD", "mean"),
        Actual_Default_Rate=("Actual", "mean")
    )
    .reset_index()
)

decile_final["Average_PD"] *= 100
decile_final["Actual_Default_Rate"] *= 100

display(decile_final)

# LIFT

overall_rate = y_test.mean() * 100

top_decile_rate = (
    decile_final.iloc[-1]["Actual_Default_Rate"]
)

top_decile_lift = (
    top_decile_rate / overall_rate
)

print("=" * 60)
print("RISK RANKING")
print("=" * 60)

print(f"Overall default rate      : {overall_rate:.2f}%")
print(f"Top-decile default rate   : {top_decile_rate:.2f}%")
print(f"Top-decile lift            : {top_decile_lift:.2f}x")


## 6. Out-of-time validation

Train on earlier vintages and evaluate on the 2022 cohort to examine temporal generalization.

In [ ]:
# Train: 2018–2021
# OOT Test: 2022

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import pandas as pd
import numpy as np

print("=" * 70)
print("OUT-OF-TIME VALIDATION — FINAL SQL-ENGINEERED MODEL")
print("=" * 70)

# 1. USE THE SQL-ENGINEERED DATASET

oot_df = model_sql_df.copy()

# Make sure Vintage is numeric
oot_df["Vintage"] = pd.to_numeric(
    oot_df["Vintage"],
    errors="coerce"
)

# Make sure target is numeric
oot_df["Default_36M"] = pd.to_numeric(
    oot_df["Default_36M"],
    errors="coerce"
)

# 2. REMOVE 2022 FROM TRAINING

train_oot = oot_df[oot_df["Vintage"] <= 2021].copy()
test_oot = oot_df[oot_df["Vintage"] == 2022].copy()

print("\nTrain shape:", train_oot.shape)
print("OOT Test shape:", test_oot.shape)

print("\nTrain vintages:")
print(train_oot["Vintage"].value_counts().sort_index())

print("\nOOT vintage:")
print(test_oot["Vintage"].value_counts().sort_index())

print(
    f"\nTrain default rate: "
    f"{train_oot['Default_36M'].mean() * 100:.4f}%"
)

print(
    f"OOT default rate: "
    f"{test_oot['Default_36M'].mean() * 100:.4f}%"
)

# 3. SAME FEATURES USED BY FINAL SQL XGBOOST

target = "Default_36M"

feature_cols = [
    "Vintage",
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState",
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

X_train_oot = train_oot[feature_cols].copy()
y_train_oot = train_oot[target].copy()

X_test_oot = test_oot[feature_cols].copy()
y_test_oot = test_oot[target].copy()

# 4. TRAIN FINAL SQL-ENGINEERED XGBOOST

oot_model = best_xgb

oot_model.fit(
    X_train_oot,
    y_train_oot
)

print("\nOOT model trained successfully.")

# 5. PREDICT 2022

y_pred_prob_oot = oot_model.predict_proba(
    X_test_oot
)[:, 1]

# 6. PERFORMANCE

roc_auc_oot = roc_auc_score(
    y_test_oot,
    y_pred_prob_oot
)

pr_auc_oot = average_precision_score(
    y_test_oot,
    y_pred_prob_oot
)

fpr, tpr, thresholds = roc_curve(
    y_test_oot,
    y_pred_prob_oot
)

ks_oot = max(tpr - fpr)

print("\n" + "=" * 70)
print("OUT-OF-TIME PERFORMANCE — 2022")
print("=" * 70)

print(f"ROC-AUC : {roc_auc_oot:.4f}")
print(f"PR-AUC  : {pr_auc_oot:.4f}")
print(f"KS      : {ks_oot:.4f}")

# 7. PD DECILES

evaluation_oot = pd.DataFrame({
    "Actual": y_test_oot.values,
    "Predicted_PD": y_pred_prob_oot
})

evaluation_oot["PD_Decile"] = pd.qcut(
    evaluation_oot["Predicted_PD"],
    10,
    labels=False,
    duplicates="drop"
) + 1

decile_oot = (
    evaluation_oot
    .groupby("PD_Decile")
    .agg(
        Loans=("Actual", "count"),
        Defaults=("Actual", "sum"),
        Average_PD=("Predicted_PD", "mean"),
        Actual_Default_Rate=("Actual", "mean")
    )
    .reset_index()
)

decile_oot["Average_PD"] *= 100
decile_oot["Actual_Default_Rate"] *= 100

print("\n" + "=" * 70)
print("2022 PD DECILE TABLE")
print("=" * 70)

display(decile_oot)

# 8. TOP DECILE LIFT

overall_default_rate_oot = y_test_oot.mean() * 100

top_decile_rate_oot = (
    decile_oot.iloc[-1]["Actual_Default_Rate"]
)

lift_oot = (
    top_decile_rate_oot /
    overall_default_rate_oot
)

print("\n" + "=" * 70)
print("2022 RISK RANKING")
print("=" * 70)

print(
    f"Overall default rate       : "
    f"{overall_default_rate_oot:.2f}%"
)

print(
    f"Top-decile default rate    : "
    f"{top_decile_rate_oot:.2f}%"
)

print(
    f"Top-decile lift            : "
    f"{lift_oot:.2f}x"
)

# 9. COMPARE RANDOM TEST VS OOT

print("\n" + "=" * 70)
print("RANDOM TEST vs OUT-OF-TIME TEST")
print("=" * 70)

print(
    f"Random-test ROC-AUC       : {roc_auc_tuned:.4f}"
)
print(
    f"OOT 2022 ROC-AUC          : {roc_auc_oot:.4f}"
)

print(
    f"\nRandom-test PR-AUC        : {pr_auc_tuned:.4f}"
)
print(
    f"OOT 2022 PR-AUC           : {pr_auc_oot:.4f}"
)

print(
    f"\nRandom-test KS            : {ks_tuned:.4f}"
)
print(
    f"OOT 2022 KS               : {ks_oot:.4f}"
)

print(
    f"\nRandom-test top-decile lift : {lift_tuned:.2f}x"
)
print(
    f"OOT 2022 top-decile lift    : {lift_oot:.2f}x"
)


In [ ]:
# IDENTIFY FINAL SQL-ENGINEERED MODEL

print("MODEL OBJECTS CURRENTLY IN NOTEBOOK")
print("=" * 60)

for name in [
    "best_xgb",
    "search",
    "final_xgb",
    "xgb_final",
    "final_model",
    "sql_xgb",
    "final_sql_xgb"
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}  -->  {type(obj)}")


In [ ]:
# YEAR-BY-YEAR OUT-OF-TIME VALIDATION
# FINAL SQL-ENGINEERED XGBOOST

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve
)
import pandas as pd
import numpy as np

# 1. USE THE FINAL SQL-ENGINEERED DATASET

df = model_sql_df.copy()

target = "Default_36M"

# Make sure Vintage is numeric
df["Vintage"] = pd.to_numeric(df["Vintage"], errors="coerce")

# 2. DEFINE FEATURES

feature_cols = [
    "Vintage",
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState",
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

# Keep only columns that actually exist
feature_cols = [c for c in feature_cols if c in df.columns]

# 3. YEAR-BY-YEAR VALIDATION

results = []

for year in sorted(df["Vintage"].dropna().unique()):

    year_df = df[df["Vintage"] == year].copy()

    X_year = year_df[feature_cols]
    y_year = year_df[target]

    # Skip if only one target class exists
    if y_year.nunique() < 2:
        continue

    # Predict using FINAL MODEL

    pd_pred = final_model.predict_proba(X_year)[:, 1]

    # ROC-AUC

    roc_auc = roc_auc_score(y_year, pd_pred)

    # PR-AUC

    pr_auc = average_precision_score(y_year, pd_pred)

    # KS

    fpr, tpr, thresholds = roc_curve(y_year, pd_pred)
    ks = np.max(tpr - fpr)

    # TOP DECILE LIFT

    temp = pd.DataFrame({
        "Actual": y_year.values,
        "PD": pd_pred
    })

    temp["Decile"] = pd.qcut(
        temp["PD"],
        10,
        labels=False,
        duplicates="drop"
    ) + 1

    decile = (
        temp.groupby("Decile")
        .agg(
            Loans=("Actual", "count"),
            Defaults=("Actual", "sum"),
            Average_PD=("PD", "mean"),
            Actual_Default_Rate=("Actual", "mean")
        )
        .reset_index()
    )

    top_decile = decile.iloc[-1]

    overall_default_rate = y_year.mean()

    top_decile_rate = top_decile["Actual_Default_Rate"]

    lift = (
        top_decile_rate / overall_default_rate
        if overall_default_rate > 0
        else np.nan
    )

    # STORE RESULTS

    results.append({
        "Vintage": int(year),
        "Loans": len(year_df),
        "Defaults": int(y_year.sum()),
        "Default_Rate_%": y_year.mean() * 100,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "KS": ks,
        "Top_Decile_Default_%": top_decile_rate * 100,
        "Top_Decile_Lift": lift
    })

# 4. DISPLAY RESULTS

yearly_results = pd.DataFrame(results)

print("=" * 80)
print("YEAR-BY-YEAR VALIDATION — FINAL SQL-ENGINEERED XGBOOST")
print("=" * 80)

display(
    yearly_results.round({
        "Default_Rate_%": 2,
        "ROC_AUC": 4,
        "PR_AUC": 4,
        "KS": 4,
        "Top_Decile_Default_%": 2,
        "Top_Decile_Lift": 2
    })
)


In [ ]:
# PERFORMANCE STABILITY CHECK

print("=" * 80)
print("PERFORMANCE STABILITY")
print("=" * 80)

print(
    f"Average ROC-AUC : {yearly_results['ROC_AUC'].mean():.4f}"
)

print(
    f"Average PR-AUC  : {yearly_results['PR_AUC'].mean():.4f}"
)

print(
    f"Average KS      : {yearly_results['KS'].mean():.4f}"
)

print(
    f"Average Lift    : {yearly_results['Top_Decile_Lift'].mean():.2f}x"
)

print("\nBest ROC-AUC year:")
print(
    yearly_results.loc[
        yearly_results["ROC_AUC"].idxmax(),
        ["Vintage", "ROC_AUC", "PR_AUC", "KS", "Top_Decile_Lift"]
    ]
)

print("\nWorst ROC-AUC year:")
print(
    yearly_results.loc[
        yearly_results["ROC_AUC"].idxmin(),
        ["Vintage", "ROC_AUC", "PR_AUC", "KS", "Top_Decile_Lift"]
    ]
)


In [ ]:
# VINTAGE-LEVEL RISK CHARACTERISTICS

risk_vars = [
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalInterestRate",
    "MIPercent",
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

risk_vars = [c for c in risk_vars if c in df.columns]

vintage_profile = (
    df.groupby("Vintage")[risk_vars + [target]]
    .mean()
    .reset_index()
)

print("=" * 80)
print("VINTAGE-LEVEL RISK PROFILE")
print("=" * 80)

display(vintage_profile.round(4))


In [ ]:
# SQL-ENGINEERED FEATURES — INCREMENTAL VALUE TEST

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

# 1. BASE MODEL — ORIGINAL VARIABLES ONLY

base_features = [
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState"
]

# 2. SQL-ENGINEERED FEATURES

sql_engineered_features = [
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

# 3. CHECK THAT ALL FEATURES EXIST

print("Missing base features:")
print([x for x in base_features if x not in model_sql_df.columns])

print("\nMissing SQL features:")
print([
    x for x in sql_engineered_features
    if x not in model_sql_df.columns
])

# 4. BUILD TWO DATASETS

X_base = model_sql_df[base_features]
X_sql = model_sql_df[base_features + sql_engineered_features]

y = model_sql_df["Default_36M"]

print("\nBase feature count:", X_base.shape[1])
print("SQL-enhanced feature count:", X_sql.shape[1])

# 5. SAME TRAIN / TEST SPLIT

from sklearn.model_selection import train_test_split

X_base_train, X_base_test, y_base_train, y_base_test = train_test_split(
    X_base,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_sql_train, X_sql_test, y_sql_train, y_sql_test = train_test_split(
    X_sql,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# 6. USE THE SAME MODEL STRUCTURE

from xgboost import XGBClassifier

base_xgb = XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.02,
    min_child_weight=3,
    subsample=0.7,
    colsample_bytree=1.0,
    gamma=0,
    reg_alpha=1,
    reg_lambda=2,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

sql_xgb = XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.02,
    min_child_weight=3,
    subsample=0.7,
    colsample_bytree=1.0,
    gamma=0,
    reg_alpha=1,
    reg_lambda=2,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

# 7. IMPORTANT:
# XGBoost CANNOT USE RAW OBJECT COLUMNS
# Use your existing preprocessing pipelines.

base_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", base_xgb)
])

sql_pipeline = Pipeline([
    ("preprocessor", preprocessor_sql),
    ("model", sql_xgb)
])

# 8. FIT

print("\nTraining BASE MODEL...")
base_pipeline.fit(X_base_train, y_base_train)

print("Training SQL-ENHANCED MODEL...")
sql_pipeline.fit(X_sql_train, y_sql_train)

# 9. PREDICTIONS

base_prob = base_pipeline.predict_proba(X_base_test)[:, 1]
sql_prob = sql_pipeline.predict_proba(X_sql_test)[:, 1]

# 10. PERFORMANCE COMPARISON

base_auc = roc_auc_score(y_base_test, base_prob)
sql_auc = roc_auc_score(y_sql_test, sql_prob)

base_pr = average_precision_score(y_base_test, base_prob)
sql_pr = average_precision_score(y_sql_test, sql_prob)

# KS
fpr_base, tpr_base, _ = roc_curve(y_base_test, base_prob)
fpr_sql, tpr_sql, _ = roc_curve(y_sql_test, sql_prob)

base_ks = max(tpr_base - fpr_base)
sql_ks = max(tpr_sql - fpr_sql)

print("\n" + "=" * 70)
print("INCREMENTAL VALUE OF SQL-ENGINEERED FEATURES")
print("=" * 70)

print(f"\nBASE MODEL")
print(f"ROC-AUC : {base_auc:.4f}")
print(f"PR-AUC  : {base_pr:.4f}")
print(f"KS      : {base_ks:.4f}")

print(f"\nSQL-ENHANCED MODEL")
print(f"ROC-AUC : {sql_auc:.4f}")
print(f"PR-AUC  : {sql_pr:.4f}")
print(f"KS      : {sql_ks:.4f}")

print("\nINCREMENTAL IMPROVEMENT")
print(f"ROC-AUC improvement : {sql_auc - base_auc:+.4f}")
print(f"PR-AUC improvement  : {sql_pr - base_pr:+.4f}")
print(f"KS improvement      : {sql_ks - base_ks:+.4f}")


In [ ]:
# INCREMENTAL VALUE TEST — BASE vs SQL-ENGINEERED FEATURES
# USING THE EXISTING PREPROCESSING SETUP

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve
)

# 1. FEATURE SETS

base_features = [
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState"
]

sql_engineered_features = [
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

all_features = base_features + sql_engineered_features

X_base = model_sql_df[base_features].copy()
X_sql = model_sql_df[all_features].copy()
y = model_sql_df["Default_36M"].copy()

# 2. SAME TRAIN / TEST INDICES

from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    range(len(model_sql_df)),
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_base_train = X_base.iloc[train_idx]
X_base_test  = X_base.iloc[test_idx]

X_sql_train = X_sql.iloc[train_idx]
X_sql_test  = X_sql.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test_incremental = y.iloc[test_idx]

# 3. IDENTIFY THE PREPROCESSOR FROM YOUR EXISTING FINAL MODEL

print("Existing final model:")
print(final_model)

existing_preprocessor = final_model.named_steps["preprocessor"]

print("\nExisting preprocessor recovered successfully.")

# 4. CREATE BASE MODEL

base_model = Pipeline([
    ("preprocessor", clone(existing_preprocessor)),
    ("model", XGBClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.02,
        min_child_weight=3,
        subsample=0.7,
        colsample_bytree=1.0,
        gamma=0,
        reg_alpha=1,
        reg_lambda=2,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        n_jobs=-1
    ))
])

# 5. SQL-ENHANCED MODEL
# Same preprocessing logic.
# The SQL flags are already numeric, so they can pass through
# the numeric branch.

sql_model = Pipeline([
    ("preprocessor", clone(existing_preprocessor)),
    ("model", XGBClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.02,
        min_child_weight=3,
        subsample=0.7,
        colsample_bytree=1.0,
        gamma=0,
        reg_alpha=1,
        reg_lambda=2,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        n_jobs=-1
    ))
])

# 6. TRAIN

print("\n" + "=" * 70)
print("TRAINING BASE MODEL")
print("=" * 70)

base_model.fit(X_base_train, y_train)

print("\n" + "=" * 70)
print("TRAINING SQL-ENHANCED MODEL")
print("=" * 70)

sql_model.fit(X_sql_train, y_train)

# 7. PREDICT

base_prob = base_model.predict_proba(X_base_test)[:, 1]

sql_prob = sql_model.predict_proba(X_sql_test)[:, 1]

# 8. PERFORMANCE FUNCTION

def calculate_metrics(y_true, prob):

    auc = roc_auc_score(y_true, prob)

    pr_auc = average_precision_score(
        y_true,
        prob
    )

    fpr, tpr, _ = roc_curve(
        y_true,
        prob
    )

    ks = max(tpr - fpr)

    return auc, pr_auc, ks

base_auc, base_pr, base_ks = calculate_metrics(
    y_test_incremental,
    base_prob
)

sql_auc, sql_pr, sql_ks = calculate_metrics(
    y_test_incremental,
    sql_prob
)

# 9. TOP-DECILE LIFT

def top_decile_lift(y_true, prob):

    temp = pd.DataFrame({
        "Actual": y_true.values,
        "PD": prob
    })

    temp["Decile"] = pd.qcut(
        temp["PD"],
        10,
        labels=False,
        duplicates="drop"
    ) + 1

    decile = (
        temp
        .groupby("Decile")
        .agg(
            Loans=("Actual", "count"),
            Defaults=("Actual", "sum"),
            Default_Rate=("Actual", "mean")
        )
    )

    overall_rate = y_true.mean()

    top_rate = decile.iloc[-1]["Default_Rate"]

    return top_rate / overall_rate

base_lift = top_decile_lift(
    y_test_incremental,
    base_prob
)

sql_lift = top_decile_lift(
    y_test_incremental,
    sql_prob
)

# 10. FINAL COMPARISON

comparison = pd.DataFrame({
    "Model": [
        "Base XGBoost",
        "SQL-Enhanced XGBoost"
    ],

    "ROC_AUC": [
        base_auc,
        sql_auc
    ],

    "PR_AUC": [
        base_pr,
        sql_pr
    ],

    "KS": [
        base_ks,
        sql_ks
    ],

    "Top_Decile_Lift": [
        base_lift,
        sql_lift
    ]
})

print("\n")
print("=" * 70)
print("BASE vs SQL-ENHANCED XGBOOST")
print("=" * 70)

display(comparison.round(4))

# 11. INCREMENTAL IMPROVEMENT

print("\n" + "=" * 70)
print("INCREMENTAL IMPROVEMENT FROM SQL FEATURES")
print("=" * 70)

print(
    f"ROC-AUC improvement : "
    f"{sql_auc - base_auc:+.4f}"
)

print(
    f"PR-AUC improvement  : "
    f"{sql_pr - base_pr:+.4f}"
)

print(
    f"KS improvement      : "
    f"{sql_ks - base_ks:+.4f}"
)

print(
    f"Lift improvement    : "
    f"{sql_lift - base_lift:+.2f}x"
)


In [ ]:
# CLEAN BASE vs SQL-ENGINEERED MODEL COMPARISON

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve
)
from xgboost import XGBClassifier

# 1. DEFINE TARGET

target = "Default_36M"

# 2. DEFINE BASE FEATURES
#    These are the original variables BEFORE SQL engineering

base_features = [
    "Vintage",
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState"
]

# 3. DEFINE SQL-ENGINEERED FEATURES

sql_features = [
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

# SQL-enhanced model = base + SQL features
sql_enhanced_features = base_features + sql_features

# 4. CHECK THAT EVERYTHING EXISTS

print("=" * 70)
print("FEATURE CHECK")
print("=" * 70)

missing_base = [
    c for c in base_features
    if c not in model_sql_df.columns
]

missing_sql = [
    c for c in sql_features
    if c not in model_sql_df.columns
]

print("Missing base features:", missing_base)
print("Missing SQL features :", missing_sql)

assert len(missing_base) == 0, "Base feature missing!"
assert len(missing_sql) == 0, "SQL feature missing!"

print("\nBase feature count :", len(base_features))
print("SQL feature count  :", len(sql_enhanced_features))

# 5. CREATE COMPLETE MODELLING DATASET

model_data = model_sql_df[
    sql_enhanced_features + [target]
].copy()

# Remove rows where target is missing
model_data = model_data.dropna(subset=[target])

print("\nFinal modelling shape:", model_data.shape)

# 6. USE THE SAME OOT SPLIT
#    2022 = TEST

train_mask = model_data["Vintage"] < 2022
test_mask  = model_data["Vintage"] == 2022

train_data = model_data.loc[train_mask].copy()
test_data  = model_data.loc[test_mask].copy()

print("\nTRAIN:", train_data.shape)
print("TEST :", test_data.shape)

print("\nTrain vintages:")
print(train_data["Vintage"].value_counts().sort_index())

print("\nTest vintages:")
print(test_data["Vintage"].value_counts().sort_index())

# 7. TARGET

y_train = train_data[target].astype(int)
y_test = test_data[target].astype(int)

# 8. CREATE BASE / SQL MATRICES

X_base_train = train_data[base_features].copy()
X_base_test  = test_data[base_features].copy()

X_sql_train = train_data[sql_enhanced_features].copy()
X_sql_test  = test_data[sql_enhanced_features].copy()

print("\nBASE TRAIN:", X_base_train.shape)
print("BASE TEST :", X_base_test.shape)

print("SQL TRAIN :", X_sql_train.shape)
print("SQL TEST  :", X_sql_test.shape)

# 9. IDENTIFY NUMERIC / CATEGORICAL VARIABLES

base_numeric = [
    c for c in base_features
    if pd.api.types.is_numeric_dtype(X_base_train[c])
]

base_categorical = [
    c for c in base_features
    if c not in base_numeric
]

sql_numeric = [
    c for c in sql_enhanced_features
    if pd.api.types.is_numeric_dtype(X_sql_train[c])
]

sql_categorical = [
    c for c in sql_enhanced_features
    if c not in sql_numeric
]

print("\nBase numeric:", len(base_numeric))
print("Base categorical:", len(base_categorical))

print("SQL numeric:", len(sql_numeric))
print("SQL categorical:", len(sql_categorical))

# 10. BUILD BASE PREPROCESSOR

base_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            base_numeric
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=True
                    )
                )
            ]),
            base_categorical
        )
    ],
    remainder="drop"
)

# 11. BUILD SQL PREPROCESSOR

sql_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            sql_numeric
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=True
                    )
                )
            ]),
            sql_categorical
        )
    ],
    remainder="drop"
)

# 12. USE SAME TUNED XGBOOST SPECIFICATION

xgb_params = {
    "n_estimators": 600,
    "max_depth": 5,
    "learning_rate": 0.02,
    "min_child_weight": 3,
    "subsample": 0.7,
    "colsample_bytree": 1.0,
    "gamma": 0,
    "reg_alpha": 1,
    "reg_lambda": 2,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
}

base_xgb = XGBClassifier(**xgb_params)
sql_xgb = XGBClassifier(**xgb_params)

# 13. BUILD PIPELINES

base_model = Pipeline([
    ("preprocessor", base_preprocessor),
    ("model", base_xgb)
])

sql_model = Pipeline([
    ("preprocessor", sql_preprocessor),
    ("model", sql_xgb)
])

# 14. TRAIN BASE MODEL

print("\n" + "=" * 70)
print("TRAINING BASE MODEL")
print("=" * 70)

base_model.fit(X_base_train, y_train)

print("Base model trained successfully.")

# 15. TRAIN SQL-ENHANCED MODEL

print("\n" + "=" * 70)
print("TRAINING SQL-ENHANCED MODEL")
print("=" * 70)

sql_model.fit(X_sql_train, y_train)

print("SQL-enhanced model trained successfully.")

# 16. PREDICT

base_pd = base_model.predict_proba(X_base_test)[:, 1]
sql_pd = sql_model.predict_proba(X_sql_test)[:, 1]

# 17. METRIC FUNCTION

def calculate_metrics(y_true, pd_pred):

    roc_auc = roc_auc_score(y_true, pd_pred)

    pr_auc = average_precision_score(
        y_true,
        pd_pred
    )

    fpr, tpr, thresholds = roc_curve(
        y_true,
        pd_pred
    )

    ks = np.max(tpr - fpr)

    # Top decile
    temp = pd.DataFrame({
        "Actual": y_true.values,
        "PD": pd_pred
    })

    temp["Decile"] = pd.qcut(
        temp["PD"],
        10,
        labels=False,
        duplicates="drop"
    ) + 1

    top_decile = temp[temp["Decile"] == 10]

    overall_rate = temp["Actual"].mean()

    top_rate = top_decile["Actual"].mean()

    lift = top_rate / overall_rate

    return {
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "KS": ks,
        "Top_Decile_Default_Rate": top_rate * 100,
        "Top_Decile_Lift": lift
    }

# 18. CALCULATE RESULTS

base_results = calculate_metrics(
    y_test,
    base_pd
)

sql_results = calculate_metrics(
    y_test,
    sql_pd
)

# 19. FINAL COMPARISON

comparison = pd.DataFrame(
    [base_results, sql_results],
    index=[
        "Base Model",
        "SQL-Enhanced Model"
    ]
)

print("\n" + "=" * 70)
print("BASE vs SQL-ENHANCED MODEL — OOT 2022")
print("=" * 70)

display(
    comparison.round(4)
)

# 20. IMPROVEMENT FROM SQL ENGINEERING

print("\n" + "=" * 70)
print("SQL FEATURE VALUE")
print("=" * 70)

print(
    f"ROC-AUC improvement : "
    f"{sql_results['ROC_AUC'] - base_results['ROC_AUC']:+.4f}"
)

print(
    f"PR-AUC improvement  : "
    f"{sql_results['PR_AUC'] - base_results['PR_AUC']:+.4f}"
)

print(
    f"KS improvement      : "
    f"{sql_results['KS'] - base_results['KS']:+.4f}"
)

print(
    f"Lift improvement    : "
    f"{sql_results['Top_Decile_Lift'] - base_results['Top_Decile_Lift']:+.2f}x"
)


In [ ]:
# FINAL SQL-ENGINEERED XGBOOST

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve
)

print("=" * 80)
print("ROLLING OUT-OF-TIME VALIDATION — SQL-ENHANCED XGBOOST")
print("=" * 80)

# 1. USE THE SQL-ENGINEERED DATASET

df_model = model_sql_df.copy()

target = "Default_36M"
vintage_col = "Vintage"

# 2. IDENTIFY FEATURES

sql_features = [
    "Vintage",
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState",
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

missing_features = [
    c for c in sql_features
    if c not in df_model.columns
]

print("\nMissing features:", missing_features)

if missing_features:
    raise ValueError(
        f"Missing SQL features: {missing_features}"
    )

# 3. YEAR-BY-YEAR OOT TESTING

years = sorted(df_model[vintage_col].dropna().unique())

rolling_results = []

for test_year in years[1:]:

    train_df = df_model[
        df_model[vintage_col] < test_year
    ].copy()

    test_df = df_model[
        df_model[vintage_col] == test_year
    ].copy()

    X_train = train_df[sql_features]
    y_train = train_df[target]

    X_test = test_df[sql_features]
    y_test = test_df[target]

    print("\n" + "-" * 80)
    print(f"TRAIN < {test_year}  |  TEST = {test_year}")
    print("-" * 80)

    print("Train shape:", X_train.shape)
    print("Test shape :", X_test.shape)

    print(
        f"Train default rate: "
        f"{y_train.mean()*100:.2f}%"
    )

    print(
        f"Test default rate : "
        f"{y_test.mean()*100:.2f}%"
    )

    # FIT A FRESH MODEL FOR EACH VINTAGE

    model = final_model

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]

    # METRICS

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    pr_auc = average_precision_score(
        y_test,
        y_prob
    )

    fpr, tpr, thresholds = roc_curve(
        y_test,
        y_prob
    )

    ks = np.max(tpr - fpr)

    # TOP DECILE

    temp = pd.DataFrame({
        "Actual": y_test.values,
        "PD": y_prob
    })

    temp = temp.sort_values(
        "PD",
        ascending=False
    ).reset_index(drop=True)

    top_n = int(np.ceil(len(temp) * 0.10))

    top_decile = temp.iloc[:top_n]

    top_decile_default_rate = (
        top_decile["Actual"].mean() * 100
    )

    overall_default_rate = (
        y_test.mean() * 100
    )

    lift = (
        top_decile_default_rate /
        overall_default_rate
        if overall_default_rate > 0
        else np.nan
    )

    rolling_results.append({
        "Vintage": test_year,
        "Train_Loans": len(train_df),
        "Test_Loans": len(test_df),
        "Train_Default_Rate_%": y_train.mean() * 100,
        "Test_Default_Rate_%": y_test.mean() * 100,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "KS": ks,
        "Top_Decile_Default_%": top_decile_default_rate,
        "Top_Decile_Lift": lift
    })

    print(
        f"ROC-AUC : {roc_auc:.4f}"
    )

    print(
        f"PR-AUC  : {pr_auc:.4f}"
    )

    print(
        f"KS      : {ks:.4f}"
    )

    print(
        f"Top-decile lift : {lift:.2f}x"
    )

# 4. FINAL RESULTS TABLE

rolling_results = pd.DataFrame(
    rolling_results
)

print("\n")
print("=" * 80)
print("ROLLING OOT RESULTS")
print("=" * 80)

display(
    rolling_results.round(4)
)


In [ ]:

print("=" * 80)
print("OOT STABILITY SUMMARY")
print("=" * 80)

metrics = [
    "ROC_AUC",
    "PR_AUC",
    "KS",
    "Top_Decile_Lift"
]

for metric in metrics:

    values = rolling_results[metric]

    print(
        f"\n{metric}"
    )

    print(
        f"  Mean : {values.mean():.4f}"
    )

    print(
        f"  Min  : {values.min():.4f}"
    )

    print(
        f"  Max  : {values.max():.4f}"
    )

    print(
        f"  Std  : {values.std():.4f}"
    )

# BEST / WORST YEAR

print("\n" + "=" * 80)
print("BEST / WORST OOT YEARS")
print("=" * 80)

best_year = rolling_results.loc[
    rolling_results["ROC_AUC"].idxmax()
]

worst_year = rolling_results.loc[
    rolling_results["ROC_AUC"].idxmin()
]

print("\nBest ROC-AUC year:")
display(
    best_year.to_frame().T
)

print("\nWorst ROC-AUC year:")
display(
    worst_year.to_frame().T
)


## 7. Probability calibration

Check whether predicted probabilities correspond to observed default frequencies across risk bands and vintages.

In [ ]:

from sklearn.calibration import calibration_curve

calibration_results = []

for test_year in years[1:]:

    train_df = df_model[
        df_model[vintage_col] < test_year
    ].copy()

    test_df = df_model[
        df_model[vintage_col] == test_year
    ].copy()

    X_train = train_df[sql_features]
    y_train = train_df[target]

    X_test = test_df[sql_features]
    y_test = test_df[target]

    model = final_model

    model.fit(
        X_train,
        y_train
    )

    y_prob = model.predict_proba(
        X_test
    )[:, 1]

    temp = pd.DataFrame({
        "Actual": y_test.values,
        "PD": y_prob
    })

    # PD DECILES

    temp["PD_Decile"] = pd.qcut(
        temp["PD"],
        q=10,
        labels=False,
        duplicates="drop"
    ) + 1

    decile_table = (
        temp
        .groupby("PD_Decile")
        .agg(
            Loans=("Actual", "size"),
            Defaults=("Actual", "sum"),
            Average_PD=("PD", "mean"),
            Actual_Default_Rate=("Actual", "mean")
        )
        .reset_index()
    )

    decile_table["Average_PD_%"] = (
        decile_table["Average_PD"] * 100
    )

    decile_table["Actual_Default_Rate_%"] = (
        decile_table["Actual_Default_Rate"] * 100
    )

    decile_table["Vintage"] = test_year

    calibration_results.append(
        decile_table
    )

calibration_results = pd.concat(
    calibration_results,
    ignore_index=True
)

print("=" * 80)
print("OOT PD CALIBRATION")
print("=" * 80)

display(
    calibration_results[
        [
            "Vintage",
            "PD_Decile",
            "Loans",
            "Defaults",
            "Average_PD_%",
            "Actual_Default_Rate_%"
        ]
    ].round(3)
)


In [ ]:

calibration_results["Calibration_Gap"] = (
    calibration_results["Actual_Default_Rate_%"]
    -
    calibration_results["Average_PD_%"]
)

print("=" * 80)
print("CALIBRATION GAP")
print("=" * 80)

display(
    calibration_results[
        [
            "Vintage",
            "PD_Decile",
            "Average_PD_%",
            "Actual_Default_Rate_%",
            "Calibration_Gap"
        ]
    ].round(3)
)

# Overall absolute calibration gap

mean_abs_gap = (
    calibration_results["Calibration_Gap"]
    .abs()
    .mean()
)

print(
    f"\nMean absolute calibration gap: "
    f"{mean_abs_gap:.3f} percentage points"
)


In [ ]:

import numpy as np
import pandas as pd

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    brier_score_loss
)

print("=" * 80)
print("TEMPORAL PD CALIBRATION")
print("=" * 80)

# 1. DATA

df_model = model_sql_df.copy()

target = "Default_36M"

sql_features = [
    "Vintage",
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState",
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

# 2. TEMPORAL SPLIT
#
# TRAIN       = 2018-2020
# CALIBRATION = 2021
# OOT TEST    = 2022

train_df = df_model[
    df_model["Vintage"] <= 2020
].copy()

cal_df = df_model[
    df_model["Vintage"] == 2021
].copy()

test_df = df_model[
    df_model["Vintage"] == 2022
].copy()

X_train = train_df[sql_features]
y_train = train_df[target]

X_cal = cal_df[sql_features]
y_cal = cal_df[target]

X_test = test_df[sql_features]
y_test = test_df[target]

print("\nTRAIN:")
print(train_df["Vintage"].value_counts().sort_index())

print("\nCALIBRATION:")
print(cal_df["Vintage"].value_counts())

print("\nTEST:")
print(test_df["Vintage"].value_counts())

print(
    f"\nTrain default rate: {y_train.mean()*100:.4f}%"
)

print(
    f"Calibration default rate: {y_cal.mean()*100:.4f}%"
)

print(
    f"2022 test default rate: {y_test.mean()*100:.4f}%"
)

# 3. TRAIN XGBOOST

print("\n" + "=" * 80)
print("TRAINING XGBOOST")
print("=" * 80)

xgb_model = final_model

xgb_model.fit(
    X_train,
    y_train
)

# 4. RAW PD

pd_cal_raw = xgb_model.predict_proba(
    X_cal
)[:, 1]

pd_test_raw = xgb_model.predict_proba(
    X_test
)[:, 1]

print("\nRaw calibration-year PD mean:")
print(f"{pd_cal_raw.mean()*100:.4f}%")

print("Actual calibration-year default rate:")
print(f"{y_cal.mean()*100:.4f}%")

# 5. PLATT / LOGISTIC CALIBRATION

# Use log-odds of raw PD as calibration input.
# This is much safer than fitting directly on the probability.

eps = 1e-6

pd_cal_clip = np.clip(
    pd_cal_raw,
    eps,
    1 - eps
)

pd_test_clip = np.clip(
    pd_test_raw,
    eps,
    1 - eps
)

logit_cal = np.log(
    pd_cal_clip /
    (1 - pd_cal_clip)
).reshape(-1, 1)

logit_test = np.log(
    pd_test_clip /
    (1 - pd_test_clip)
).reshape(-1, 1)

platt = LogisticRegression()

platt.fit(
    logit_cal,
    y_cal
)

pd_test_calibrated = platt.predict_proba(
    logit_test
)[:, 1]

# 6. RAW VS CALIBRATED METRICS

def calculate_metrics(y, pd_score):

    roc = roc_auc_score(
        y,
        pd_score
    )

    pr = average_precision_score(
        y,
        pd_score
    )

    fpr, tpr, _ = roc_curve(
        y,
        pd_score
    )

    ks = np.max(
        tpr - fpr
    )

    brier = brier_score_loss(
        y,
        pd_score
    )

    # top decile
    temp = pd.DataFrame({
        "Actual": y.values,
        "PD": pd_score
    })

    temp = temp.sort_values(
        "PD",
        ascending=False
    )

    n_top = int(
        np.ceil(len(temp) * 0.10)
    )

    top_decile = temp.iloc[:n_top]

    overall_rate = y.mean()

    top_rate = top_decile["Actual"].mean()

    lift = (
        top_rate / overall_rate
        if overall_rate > 0
        else np.nan
    )

    return {
        "ROC_AUC": roc,
        "PR_AUC": pr,
        "KS": ks,
        "Brier": brier,
        "Top_Decile_Default_%": top_rate * 100,
        "Top_Decile_Lift": lift
    }

raw_metrics = calculate_metrics(
    y_test,
    pd_test_raw
)

cal_metrics = calculate_metrics(
    y_test,
    pd_test_calibrated
)

comparison = pd.DataFrame(
    [raw_metrics, cal_metrics],
    index=[
        "Raw XGBoost",
        "Calibrated XGBoost"
    ]
)

print("\n" + "=" * 80)
print("2022 RAW VS CALIBRATED")
print("=" * 80)

display(
    comparison.round(4)
)

# 7. PD LEVEL COMPARISON

print("\n" + "=" * 80)
print("PD LEVEL COMPARISON")
print("=" * 80)

print(
    f"Actual 2022 default rate : "
    f"{y_test.mean()*100:.4f}%"
)

print(
    f"Raw average PD           : "
    f"{pd_test_raw.mean()*100:.4f}%"
)

print(
    f"Calibrated average PD    : "
    f"{pd_test_calibrated.mean()*100:.4f}%"
)

# 8. CALIBRATED DECILE TABLE

calibrated_df = pd.DataFrame({
    "Actual": y_test.values,
    "Raw_PD": pd_test_raw,
    "Calibrated_PD": pd_test_calibrated
})

calibrated_df["PD_Decile"] = pd.qcut(
    calibrated_df["Calibrated_PD"],
    q=10,
    labels=False,
    duplicates="drop"
) + 1

cal_deciles = (
    calibrated_df
    .groupby("PD_Decile")
    .agg(
        Loans=("Actual", "size"),
        Defaults=("Actual", "sum"),
        Average_Raw_PD=("Raw_PD", "mean"),
        Average_Calibrated_PD=("Calibrated_PD", "mean"),
        Actual_Default_Rate=("Actual", "mean")
    )
    .reset_index()
)

cal_deciles["Average_Raw_PD_%"] = (
    cal_deciles["Average_Raw_PD"] * 100
)

cal_deciles["Average_Calibrated_PD_%"] = (
    cal_deciles["Average_Calibrated_PD"] * 100
)

cal_deciles["Actual_Default_Rate_%"] = (
    cal_deciles["Actual_Default_Rate"] * 100
)

cal_deciles["Calibration_Gap"] = (
    cal_deciles["Actual_Default_Rate_%"]
    -
    cal_deciles["Average_Calibrated_PD_%"]
)

print("\n" + "=" * 80)
print("2022 CALIBRATED PD DECILES")
print("=" * 80)

display(
    cal_deciles[
        [
            "PD_Decile",
            "Loans",
            "Defaults",
            "Average_Raw_PD_%",
            "Average_Calibrated_PD_%",
            "Actual_Default_Rate_%",
            "Calibration_Gap"
        ]
    ].round(3)
)


In [ ]:

print("=" * 80)
print("DEFAULT LABEL / VINTAGE AUDIT")
print("=" * 80)

# Basic vintage distribution
vintage_audit = (
    model_sql_df
    .groupby("Vintage")
    .agg(
        Loans=("Default_36M", "size"),
        Defaults=("Default_36M", "sum"),
        Default_Rate=("Default_36M", "mean")
    )
    .reset_index()
)

vintage_audit["Default_Rate_%"] = (
    vintage_audit["Default_Rate"] * 100
)

display(
    vintage_audit[
        ["Vintage", "Loans", "Defaults", "Default_Rate_%"]
    ]
)

# Check whether default label is strictly binary
print("\nTarget values:")
print(model_sql_df["Default_36M"].value_counts(dropna=False))

# Check missing target
print("\nMissing target:")
print(model_sql_df["Default_36M"].isna().sum())

# Cross-tab vintage x default
print("\nVintage × Default:")
display(
    pd.crosstab(
        model_sql_df["Vintage"],
        model_sql_df["Default_36M"],
        margins=True
    )
)


In [ ]:

numeric_features = [
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

available_numeric = [
    c for c in numeric_features
    if c in model_sql_df.columns
]

vintage_profile = (
    model_sql_df
    .groupby("Vintage")[available_numeric]
    .mean()
    .round(4)
)

display(vintage_profile)


In [ ]:
# FIX — DEFINE SQL FEATURE LIST FROM YOUR CURRENT DATASET

target = "Default_36M"

# Columns that are definitely identifiers / target / non-model fields
exclude_cols = [
    target
]

# Your SQL-enhanced modelling dataset has 25 predictors
sql_features_final = [
    col for col in model_sql_df.columns
    if col not in exclude_cols
]

print("SQL features:", len(sql_features_final))
print(sql_features_final)


In [ ]:
# VERIFY FINAL MODEL INPUT

print("=" * 70)
print("FINAL MODEL INPUT CHECK")
print("=" * 70)

print("Dataset columns:", len(model_sql_df.columns))
print("SQL features:", len(sql_features_final))

print("\nSQL features:")
for i, col in enumerate(sql_features_final, 1):
    print(f"{i:2d}. {col}")

print("\nFinal model:")
print(final_model)

print("\nModel preprocessor expected columns:")

try:
    expected_cols = final_model.named_steps["preprocessor"].feature_names_in_
    print(len(expected_cols))
    print(list(expected_cols))
except Exception as e:
    print("Could not read expected columns:", e)


In [ ]:

# Get the exact columns used by the fitted preprocessor
expected_cols = list(
    final_model.named_steps["preprocessor"].feature_names_in_
)

print("Number of model features:", len(expected_cols))
print("Missing from dataset:")

missing = [
    c for c in expected_cols
    if c not in model_sql_df.columns
]

print(missing)

if missing:
    raise ValueError(
        f"These model features are missing from model_sql_df: {missing}"
    )

# Predict PD for every observation
X_all_sql = model_sql_df[expected_cols].copy()

y_all = model_sql_df["Default_36M"].copy()

all_pd = final_model.predict_proba(X_all_sql)[:, 1]

diagnostic_df = model_sql_df[
    ["Vintage", "Default_36M"]
].copy()

diagnostic_df["Predicted_PD"] = all_pd

# Vintage-level comparison

vintage_pd = (
    diagnostic_df
    .groupby("Vintage")
    .agg(
        Loans=("Default_36M", "size"),
        Defaults=("Default_36M", "sum"),
        Actual_Default_Rate=("Default_36M", "mean"),
        Average_Predicted_PD=("Predicted_PD", "mean")
    )
    .reset_index()
)

vintage_pd["Actual_Default_Rate_%"] = (
    vintage_pd["Actual_Default_Rate"] * 100
)

vintage_pd["Average_Predicted_PD_%"] = (
    vintage_pd["Average_Predicted_PD"] * 100
)

vintage_pd["Calibration_Gap_pp"] = (
    vintage_pd["Average_Predicted_PD_%"]
    - vintage_pd["Actual_Default_Rate_%"]
)

print("\n" + "=" * 80)
print("VINTAGE-LEVEL PREDICTED PD VS ACTUAL DEFAULT")
print("=" * 80)

display(
    vintage_pd[
        [
            "Vintage",
            "Loans",
            "Defaults",
            "Actual_Default_Rate_%",
            "Average_Predicted_PD_%",
            "Calibration_Gap_pp"
        ]
    ]
)


In [ ]:

covid_map = {
    2018: "Pre-COVID",
    2019: "Pre-COVID",
    2020: "COVID-Era",
    2021: "COVID-Era",
    2022: "Post-COVID / Recovery"
}

covid_df = model_sql_df.copy()

covid_df["Period"] = covid_df["Vintage"].map(covid_map)

covid_summary = (
    covid_df
    .groupby("Period")
    .agg(
        Loans=("Default_36M", "size"),
        Defaults=("Default_36M", "sum"),
        Default_Rate=("Default_36M", "mean"),
        Avg_CreditScore=("CreditScore", "mean"),
        Avg_DTI=("OriginalDTI", "mean"),
        Avg_LTV=("OriginalLTV", "mean"),
        Avg_CLTV=("OriginalCLTV", "mean"),
        Avg_InterestRate=("OriginalInterestRate", "mean")
    )
    .reset_index()
)

covid_summary["Default_Rate_%"] = (
    covid_summary["Default_Rate"] * 100
)

display(covid_summary)


In [ ]:
#
# Objective:
# Test whether model discrimination and risk ranking remain stable across:
#
#   1. PRE-COVID       : 2018-2019
#   2. COVID/PANDEMIC  : 2020-2021
#   3. RECOVERY        : 2022
#
# We use the existing FINAL SQL-ENHANCED MODEL.
#
# IMPORTANT:
# This is a robustness analysis, NOT a claim that all 2020-21 defaults
# were caused by COVID.

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve
)

print("=" * 80)
print("STEP 5 — REGIME ROBUSTNESS TEST")
print("=" * 80)

# 1. DEFINE SQL-ENGINEERED FEATURES EXPLICITLY

sql_features_final = [
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

# 2. DEFINE BASE FEATURES

base_features_final = [
    "Vintage",
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PropertyType",
    "LoanPurpose",
    "AmortizationType",
    "PropertyState"
]

# 3. CREATE FINAL SQL-ENHANCED FEATURE LIST

features_final = base_features_final + sql_features_final

# 4. CHECK THAT EVERYTHING EXISTS

missing_features = [
    col for col in features_final
    if col not in model_sql_df.columns
]

print("\nMissing features:")
print(missing_features)

if len(missing_features) > 0:
    raise ValueError(
        "Some required modelling features are missing from model_sql_df."
    )

# 5. CREATE X AND y

X_all_sql = model_sql_df[features_final].copy()

y_all = model_sql_df["Default_36M"].copy()

vintage_all = model_sql_df["Vintage"].copy()

# 6. GENERATE PREDICTED PDs FROM EXISTING FINAL MODEL

print("\nGenerating model PDs...")

pd_all = final_model.predict_proba(X_all_sql)[:, 1]

print("PD generation successful.")

# 7. BUILD ANALYSIS DATASET

regime_df = pd.DataFrame({
    "Vintage": vintage_all.values,
    "Actual": y_all.values,
    "PD": pd_all
})

# 8. DEFINE ECONOMIC / CREDIT REGIMES

def assign_regime(year):

    if year in [2018, 2019]:
        return "Pre-COVID"

    elif year in [2020, 2021]:
        return "COVID / Pandemic"

    elif year == 2022:
        return "Recovery / Post-COVID"

    else:
        return "Other"

regime_df["Regime"] = regime_df["Vintage"].apply(assign_regime)

# 9. KS FUNCTION

def calculate_ks(y_true, pd_score):

    fpr, tpr, thresholds = roc_curve(
        y_true,
        pd_score
    )

    return np.max(tpr - fpr)

# 10. TOP-DECILE LIFT FUNCTION

def calculate_top_decile_lift(y_true, pd_score):

    temp = pd.DataFrame({
        "Actual": np.asarray(y_true),
        "PD": np.asarray(pd_score)
    })

    temp = temp.sort_values(
        "PD",
        ascending=False
    )

    n_top = max(
        1,
        int(np.ceil(len(temp) * 0.10))
    )

    top_decile = temp.iloc[:n_top]

    top_rate = top_decile["Actual"].mean()

    overall_rate = temp["Actual"].mean()

    if overall_rate == 0:
        return np.nan

    return top_rate / overall_rate

# 11. CALCULATE REGIME PERFORMANCE

regime_results = []

for regime in [
    "Pre-COVID",
    "COVID / Pandemic",
    "Recovery / Post-COVID"
]:

    temp = regime_df[
        regime_df["Regime"] == regime
    ].copy()

    y_regime = temp["Actual"]
    pd_regime = temp["PD"]

    if y_regime.nunique() < 2:
        print(
            f"\nSkipping {regime}: "
            "only one target class is present."
        )
        continue

    roc_auc = roc_auc_score(
        y_regime,
        pd_regime
    )

    pr_auc = average_precision_score(
        y_regime,
        pd_regime
    )

    ks = calculate_ks(
        y_regime,
        pd_regime
    )

    lift = calculate_top_decile_lift(
        y_regime,
        pd_regime
    )

    sorted_temp = temp.sort_values(
        "PD",
        ascending=False
    )

    top_n = max(
        1,
        int(np.ceil(len(sorted_temp) * 0.10))
    )

    top_decile_rate = (
        sorted_temp.iloc[:top_n]["Actual"].mean()
        * 100
    )

    regime_results.append({

        "Regime": regime,

        "Vintage_Range": (
            f"{int(temp['Vintage'].min())}-"
            f"{int(temp['Vintage'].max())}"
        ),

        "Loans": len(temp),

        "Defaults": int(temp["Actual"].sum()),

        "Default_Rate_%": (
            temp["Actual"].mean() * 100
        ),

        "Average_PD_%": (
            temp["PD"].mean() * 100
        ),

        "ROC_AUC": roc_auc,

        "PR_AUC": pr_auc,

        "KS": ks,

        "Top_Decile_Default_%": top_decile_rate,

        "Top_Decile_Lift": lift
    })

# 12. DISPLAY RESULTS

regime_results = pd.DataFrame(
    regime_results
)

print("\n")
print("=" * 80)
print("REGIME-LEVEL MODEL PERFORMANCE")
print("=" * 80)

display(
    regime_results.round(4)
)

# 13. PD CALIBRATION BY REGIME

print("\n")
print("=" * 80)
print("REGIME-LEVEL PD CALIBRATION")
print("=" * 80)

regime_calibration = regime_results.copy()

regime_calibration["Calibration_Gap_pp"] = (
    regime_calibration["Average_PD_%"]
    - regime_calibration["Default_Rate_%"]
)

display(
    regime_calibration[
        [
            "Regime",
            "Vintage_Range",
            "Loans",
            "Default_Rate_%",
            "Average_PD_%",
            "Calibration_Gap_pp"
        ]
    ].round(4)
)

# 14. YEAR-BY-YEAR REGIME PROFILE

print("\n")
print("=" * 80)
print("YEAR-BY-YEAR CREDIT PROFILE")
print("=" * 80)

year_profile = (
    regime_df
    .groupby("Vintage")
    .agg(
        Loans=("Actual", "size"),
        Defaults=("Actual", "sum"),
        Default_Rate=("Actual", "mean"),
        Average_PD=("PD", "mean")
    )
    .reset_index()
)

year_profile["Default_Rate_%"] = (
    year_profile["Default_Rate"] * 100
)

year_profile["Average_PD_%"] = (
    year_profile["Average_PD"] * 100
)

year_profile["Calibration_Gap_pp"] = (
    year_profile["Average_PD_%"]
    - year_profile["Default_Rate_%"]
)

display(
    year_profile[
        [
            "Vintage",
            "Loans",
            "Defaults",
            "Default_Rate_%",
            "Average_PD_%",
            "Calibration_Gap_pp"
        ]
    ].round(4)
)

# 15. REGIME ROBUSTNESS INTERPRETATION

print("\n")
print("=" * 80)
print("REGIME ROBUSTNESS SUMMARY")
print("=" * 80)

if len(regime_results) >= 2:

    roc_min = regime_results["ROC_AUC"].min()
    roc_max = regime_results["ROC_AUC"].max()

    ks_min = regime_results["KS"].min()
    ks_max = regime_results["KS"].max()

    lift_min = regime_results["Top_Decile_Lift"].min()
    lift_max = regime_results["Top_Decile_Lift"].max()

    print(
        f"ROC-AUC range : {roc_min:.4f} - {roc_max:.4f}"
    )

    print(
        f"KS range      : {ks_min:.4f} - {ks_max:.4f}"
    )

    print(
        f"Lift range    : {lift_min:.2f}x - {lift_max:.2f}x"
    )

    print(
        "\nThis test evaluates whether the model's ranking ability "
        "survives across materially different vintages/regimes."
    )

print("\n")
print("=" * 80)
print("COVID INTERPRETATION NOTE")
print("=" * 80)

print(
    "2020-2021 are treated as a pandemic-era regime for robustness "
    "analysis only. This does NOT establish that COVID caused the "
    "observed defaults."
)

print(
    "2022 is treated separately as a recovery/post-pandemic vintage."
)

print(
    "If discrimination remains reasonably strong across all three "
    "regimes, the model is less likely to be dependent on a single "
    "economic episode."
)


In [ ]:
# FIXED: PASS THE FULL SQL FEATURE SET TO final_model

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)
import numpy as np
import pandas as pd

# 1. DEFINE THE FULL FEATURE SET USED BY THE FINAL MODEL

base_features_final = [
    'Vintage',
    'CreditScore',
    'OriginalDTI',
    'OriginalLTV',
    'OriginalCLTV',
    'OriginalUPB',
    'OriginalInterestRate',
    'OriginalLoanTerm',
    'NumberOfBorrowers',
    'NumberOfUnits',
    'MIPercent',
    'FirstTimeHomebuyerFlag',
    'OccupancyStatus',
    'Channel',
    'PropertyType',
    'LoanPurpose',
    'AmortizationType',
    'PropertyState'
]

sql_features_final = [
    'LowCreditFlag',
    'HighLTVFlag',
    'HighDTIFlag',
    'HighCLTVFlag',
    'HighRateFlag',
    'HighCreditLTVFlag',
    'HighDTILTVFlag'
]

# Full feature set = base + SQL engineered
all_features_final = (
    base_features_final +
    sql_features_final
)

# 2. CHECK THAT EVERYTHING EXISTS

missing_features = [
    col for col in all_features_final
    if col not in model_sql_df.columns
]

print("=" * 80)
print("STEP 6 — FEATURE CHECK")
print("=" * 80)

print("Missing features:", missing_features)
print("Total model features:", len(all_features_final))

if len(missing_features) > 0:
    raise ValueError(
        f"These required model features are missing: {missing_features}"
    )

# 3. YEAR-BY-YEAR PERFORMANCE

def ranking_calibration_by_year(model, df, features):

    rows = []

    for year in sorted(df["Vintage"].unique()):

        year_df = df[df["Vintage"] == year].copy()

        X = year_df[features].copy()
        y = year_df["Default_36M"].copy()

        # Model PD
        pd_pred = model.predict_proba(X)[:, 1]

        # ROC-AUC
        roc_auc = roc_auc_score(y, pd_pred)

        # PR-AUC
        pr_auc = average_precision_score(y, pd_pred)

        # KS
        default_scores = pd_pred[y == 1]
        nondefault_scores = pd_pred[y == 0]

        all_scores = np.sort(
            np.unique(pd_pred)
        )

        ks_values = []

        for threshold in all_scores:

            tpr = (
                (default_scores >= threshold).mean()
                if len(default_scores) > 0
                else 0
            )

            fpr = (
                (nondefault_scores >= threshold).mean()
                if len(nondefault_scores) > 0
                else 0
            )

            ks_values.append(abs(tpr - fpr))

        ks = max(ks_values) if ks_values else np.nan

        # TOP DECILE

        temp = pd.DataFrame({
            "Actual": y.values,
            "PD": pd_pred
        })

        temp = temp.sort_values(
            "PD",
            ascending=False
        )

        top_n = max(
            1,
            int(np.ceil(len(temp) * 0.10))
        )

        top_decile = temp.iloc[:top_n]

        top_decile_default_rate = (
            top_decile["Actual"].mean() * 100
        )

        overall_default_rate = y.mean() * 100

        top_decile_lift = (
            top_decile_default_rate /
            overall_default_rate
            if overall_default_rate > 0
            else np.nan
        )

        # CALIBRATION

        avg_pd = pd_pred.mean() * 100
        actual_rate = y.mean() * 100

        calibration_gap = (
            actual_rate - avg_pd
        )

        brier = brier_score_loss(
            y,
            pd_pred
        )

        rows.append({

            "Vintage": year,
            "Loans": len(year_df),
            "Defaults": int(y.sum()),

            "Default_Rate_%":
                actual_rate,

            "Average_PD_%":
                avg_pd,

            "Calibration_Gap_pp":
                calibration_gap,

            "ROC_AUC":
                roc_auc,

            "PR_AUC":
                pr_auc,

            "KS":
                ks,

            "Top_Decile_Default_%":
                top_decile_default_rate,

            "Top_Decile_Lift":
                top_decile_lift,

            "Brier":
                brier
        })

    return pd.DataFrame(rows)

# 4. RUN

ranking_calibration = ranking_calibration_by_year(
    final_model,
    model_sql_df,
    all_features_final
)

# 5. DISPLAY

print("\n" + "=" * 80)
print("YEAR-BY-YEAR RANKING + CALIBRATION")
print("=" * 80)

display(
    ranking_calibration.round(4)
)


In [ ]:

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 7 — FEATURE DISTRIBUTION SHIFT / PSI")
print("=" * 80)

# TRAINING PERIOD = 2018-2021
# TEST PERIOD     = 2022

train_df = model_sql_df[
    model_sql_df["Vintage"] <= 2021
].copy()

test_2022 = model_sql_df[
    model_sql_df["Vintage"] == 2022
].copy()

# Numeric variables only
psi_features = [
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalUPB",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent",
    "LowCreditFlag",
    "HighLTVFlag",
    "HighDTIFlag",
    "HighCLTVFlag",
    "HighRateFlag",
    "HighCreditLTVFlag",
    "HighDTILTVFlag"
]

def calculate_psi(expected, actual, bins=10):

    expected = pd.Series(expected).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    actual = pd.Series(actual).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    if expected.nunique() <= 1:
        return 0.0

    # Quantile bins from training data
    breakpoints = np.unique(
        np.nanpercentile(
            expected,
            np.linspace(0, 100, bins + 1)
        )
    )

    if len(breakpoints) < 3:
        return 0.0

    expected_counts, _ = np.histogram(
        expected,
        bins=breakpoints
    )

    actual_counts, _ = np.histogram(
        actual,
        bins=breakpoints
    )

    expected_pct = expected_counts / len(expected)
    actual_pct = actual_counts / len(actual)

    # Avoid zero divisions
    expected_pct = np.where(
        expected_pct == 0,
        0.0001,
        expected_pct
    )

    actual_pct = np.where(
        actual_pct == 0,
        0.0001,
        actual_pct
    )

    psi = np.sum(
        (actual_pct - expected_pct)
        * np.log(actual_pct / expected_pct)
    )

    return psi

psi_rows = []

for feature in psi_features:

    psi_value = calculate_psi(
        train_df[feature],
        test_2022[feature]
    )

    psi_rows.append({
        "Feature": feature,
        "PSI": psi_value
    })

psi_results = (
    pd.DataFrame(psi_rows)
    .sort_values("PSI", ascending=False)
    .reset_index(drop=True)
)

def psi_interpretation(x):

    if x < 0.10:
        return "Stable"

    elif x < 0.25:
        return "Moderate Shift"

    else:
        return "Significant Shift"

psi_results["Interpretation"] = (
    psi_results["PSI"].apply(psi_interpretation)
)

print("\n2022 vs 2018-2021:")
display(psi_results.round(4))

print("\n" + "=" * 80)
print("LARGEST DISTRIBUTION SHIFTS")
print("=" * 80)

display(
    psi_results.head(10).round(4)
)


In [ ]:

print("=" * 80)
print("STEP 8 — CONCEPT DRIFT")
print("=" * 80)

concept_features = [
    "CreditScore",
    "OriginalDTI",
    "OriginalLTV",
    "OriginalCLTV",
    "OriginalInterestRate"
]

# 1. Create risk bins

concept_df = model_sql_df.copy()

concept_df["CreditScore_Bin"] = pd.cut(
    concept_df["CreditScore"],
    bins=[0, 650, 680, 720, 760, 800, 900],
    labels=[
        "<650",
        "650-679",
        "680-719",
        "720-759",
        "760-799",
        "800+"
    ],
    include_lowest=True
)

concept_df["DTI_Bin"] = pd.cut(
    concept_df["OriginalDTI"],
    bins=[0, 20, 30, 36, 43, 50, 100],
    labels=[
        "<20",
        "20-29",
        "30-35",
        "36-42",
        "43-49",
        "50+"
    ],
    include_lowest=True
)

concept_df["LTV_Bin"] = pd.cut(
    concept_df["OriginalLTV"],
    bins=[0, 60, 70, 80, 90, 100, 150],
    labels=[
        "<60",
        "60-69",
        "70-79",
        "80-89",
        "90-99",
        "100+"
    ],
    include_lowest=True
)

concept_df["InterestRate_Bin"] = pd.cut(
    concept_df["OriginalInterestRate"],
    bins=[0, 2, 3, 4, 5, 6, 10, 20],
    labels=[
        "<2",
        "2-2.99",
        "3-3.99",
        "4-4.99",
        "5-5.99",
        "6-9.99",
        "10+"
    ],
    include_lowest=True
)

# 2. Function to calculate default rates by vintage

def default_rate_by_vintage(df, bin_col):

    result = (
        df.groupby(
            ["Vintage", bin_col],
            observed=True
        )
        .agg(
            Loans=("Default_36M", "size"),
            Defaults=("Default_36M", "sum"),
            Default_Rate=("Default_36M", "mean")
        )
        .reset_index()
    )

    result["Default_Rate_%"] = (
        result["Default_Rate"] * 100
    )

    return result

# 3. Generate concept-drift tables

credit_drift = default_rate_by_vintage(
    concept_df,
    "CreditScore_Bin"
)

dti_drift = default_rate_by_vintage(
    concept_df,
    "DTI_Bin"
)

ltv_drift = default_rate_by_vintage(
    concept_df,
    "LTV_Bin"
)

rate_drift = default_rate_by_vintage(
    concept_df,
    "InterestRate_Bin"
)

print("\n" + "=" * 80)
print("CREDIT SCORE → DEFAULT RELATIONSHIP")
print("=" * 80)

display(
    credit_drift.round(3)
)

print("\n" + "=" * 80)
print("DTI → DEFAULT RELATIONSHIP")
print("=" * 80)

display(
    dti_drift.round(3)
)

print("\n" + "=" * 80)
print("LTV → DEFAULT RELATIONSHIP")
print("=" * 80)

display(
    ltv_drift.round(3)
)

print("\n" + "=" * 80)
print("INTEREST RATE → DEFAULT RELATIONSHIP")
print("=" * 80)

display(
    rate_drift.round(3)
)


In [ ]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

print("=" * 80)
print("STEP 9 — REGIME ROBUSTNESS")
print("=" * 80)

regime_map = {
    2018: "Pre-COVID",
    2019: "Pre-COVID",
    2020: "COVID / Pandemic",
    2021: "COVID / Pandemic",
    2022: "Recovery / Post-COVID"
}

regime_df = model_sql_df.copy()

regime_df["Regime"] = (
    regime_df["Vintage"].map(regime_map)
)

# Generate model PD

print("\nGenerating model predictions...")

regime_df["Predicted_PD"] = (
    final_model.predict_proba(
        regime_df[all_features_final]
    )[:, 1]
)

print("PD generation successful.")

# KS function

def calculate_ks(y, pred):

    temp = pd.DataFrame({
        "y": y,
        "pred": pred
    }).sort_values(
        "pred",
        ascending=False
    )

    positives = (temp["y"] == 1).sum()
    negatives = (temp["y"] == 0).sum()

    if positives == 0 or negatives == 0:
        return np.nan

    temp["cum_bad"] = (
        (temp["y"] == 1).cumsum() / positives
    )

    temp["cum_good"] = (
        (temp["y"] == 0).cumsum() / negatives
    )

    return (
        temp["cum_bad"] -
        temp["cum_good"]
    ).abs().max()

# Regime performance

regime_rows = []

for regime in regime_df["Regime"].dropna().unique():

    sub = regime_df[
        regime_df["Regime"] == regime
    ].copy()

    y = sub["Default_36M"]
    pred = sub["Predicted_PD"]

    roc = roc_auc_score(y, pred)
    pr = average_precision_score(y, pred)
    ks = calculate_ks(y, pred)

    sub_sorted = sub.sort_values(
        "Predicted_PD",
        ascending=False
    )

    top_n = max(
        1,
        int(np.ceil(len(sub_sorted) * 0.10))
    )

    top_decile = sub_sorted.iloc[:top_n]

    top_rate = (
        top_decile["Default_36M"].mean()
    )

    overall_rate = y.mean()

    lift = (
        top_rate / overall_rate
        if overall_rate > 0
        else np.nan
    )

    regime_rows.append({

        "Regime": regime,
        "Loans": len(sub),
        "Defaults": int(y.sum()),

        "Default_Rate_%":
            y.mean() * 100,

        "Average_PD_%":
            pred.mean() * 100,

        "ROC_AUC": roc,
        "PR_AUC": pr,
        "KS": ks,

        "Top_Decile_Default_%":
            top_rate * 100,

        "Top_Decile_Lift":
            lift
    })

regime_results = (
    pd.DataFrame(regime_rows)
    .sort_values("Regime")
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("REGIME-LEVEL PERFORMANCE")
print("=" * 80)

display(
    regime_results.round(4)
)

# Calibration

regime_results["Calibration_Gap_pp"] = (
    regime_results["Default_Rate_%"]
    -
    regime_results["Average_PD_%"]
)

print("\n" + "=" * 80)
print("REGIME-LEVEL CALIBRATION")
print("=" * 80)

display(
    regime_results[
        [
            "Regime",
            "Loans",
            "Defaults",
            "Default_Rate_%",
            "Average_PD_%",
            "Calibration_Gap_pp"
        ]
    ].round(4)
)


In [ ]:

print("=" * 80)
print("STEP 10 — FINAL MODEL DIAGNOSIS")
print("=" * 80)

# 1. Overall model performance

overall_y = regime_df["Default_36M"]
overall_pd = regime_df["Predicted_PD"]

overall_roc = roc_auc_score(
    overall_y,
    overall_pd
)

overall_pr = average_precision_score(
    overall_y,
    overall_pd
)

overall_ks = calculate_ks(
    overall_y,
    overall_pd
)

# 2. 2022 performance

test_2022 = regime_df[
    regime_df["Vintage"] == 2022
].copy()

y_2022 = test_2022["Default_36M"]
pd_2022 = test_2022["Predicted_PD"]

roc_2022 = roc_auc_score(
    y_2022,
    pd_2022
)

pr_2022 = average_precision_score(
    y_2022,
    pd_2022
)

ks_2022 = calculate_ks(
    y_2022,
    pd_2022
)

actual_2022 = y_2022.mean() * 100
predicted_2022 = pd_2022.mean() * 100

calibration_gap_2022 = (
    actual_2022 -
    predicted_2022
)

# 3. Top-decile 2022

test_2022_sorted = test_2022.sort_values(
    "Predicted_PD",
    ascending=False
)

top_n_2022 = max(
    1,
    int(np.ceil(len(test_2022_sorted) * 0.10))
)

top_2022 = test_2022_sorted.iloc[:top_n_2022]

top_default_2022 = (
    top_2022["Default_36M"].mean() * 100
)

lift_2022 = (
    top_default_2022 /
    actual_2022
)

# 4. PSI summary

max_psi_feature = (
    psi_results.iloc[0]["Feature"]
)

max_psi_value = (
    psi_results.iloc[0]["PSI"]
)

# 5. Print final summary

print("\nOVERALL MODEL")
print("-" * 50)

print(
    f"ROC-AUC : {overall_roc:.4f}"
)

print(
    f"PR-AUC  : {overall_pr:.4f}"
)

print(
    f"KS      : {overall_ks:.4f}"
)

print("\n2022 OUT-OF-TIME PERFORMANCE")
print("-" * 50)

print(
    f"ROC-AUC : {roc_2022:.4f}"
)

print(
    f"PR-AUC  : {pr_2022:.4f}"
)

print(
    f"KS      : {ks_2022:.4f}"
)

print(
    f"Actual default rate : "
    f"{actual_2022:.2f}%"
)

print(
    f"Average predicted PD : "
    f"{predicted_2022:.2f}%"
)

print(
    f"Calibration gap : "
    f"{calibration_gap_2022:.2f} pp"
)

print(
    f"Top-decile default rate : "
    f"{top_default_2022:.2f}%"
)

print(
    f"Top-decile lift : "
    f"{lift_2022:.2f}x"
)

print("\nFEATURE DRIFT")
print("-" * 50)

print(
    f"Largest PSI feature : "
    f"{max_psi_feature}"
)

print(
    f"Largest PSI : "
    f"{max_psi_value:.4f}"
)

# 6. Automated interpretation

print("\n" + "=" * 80)
print("FINAL INTERPRETATION")
print("=" * 80)

if roc_2022 >= 0.70:
    print(
        "✓ 2022 ranking performance remains "
        "economically useful."
    )
else:
    print(
        "⚠ 2022 ranking performance is materially weaker."
    )

if lift_2022 >= 2:
    print(
        "✓ The model still provides meaningful "
        "risk ranking in the top decile."
    )
else:
    print(
        "⚠ Top-decile risk separation is weak."
    )

if abs(calibration_gap_2022) <= 1:
    print(
        "✓ 2022 PD calibration is reasonably close."
    )
else:
    print(
        "⚠ 2022 PD calibration is materially distorted."
    )

if max_psi_value < 0.10:
    print(
        "✓ No major feature distribution shift detected."
    )

elif max_psi_value < 0.25:
    print(
        "⚠ Moderate feature distribution shift detected."
    )

else:
    print(
        "⚠ Significant feature distribution shift detected."
    )

print("\nConclusion:")
print(
    "The model should be evaluated not only on "
    "discrimination but also on temporal stability, "
    "distribution shift, concept drift and PD calibration."
)


In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    brier_score_loss
)

# 1. Feature check

sql_features_final = [
    'Vintage',
    'CreditScore',
    'OriginalDTI',
    'OriginalLTV',
    'OriginalCLTV',
    'OriginalUPB',
    'OriginalInterestRate',
    'OriginalLoanTerm',
    'NumberOfBorrowers',
    'NumberOfUnits',
    'MIPercent',
    'FirstTimeHomebuyerFlag',
    'OccupancyStatus',
    'Channel',
    'PropertyType',
    'LoanPurpose',
    'AmortizationType',
    'PropertyState',
    'LowCreditFlag',
    'HighLTVFlag',
    'HighDTIFlag',
    'HighCLTVFlag',
    'HighRateFlag',
    'HighCreditLTVFlag',
    'HighDTILTVFlag'
]

missing_features = [
    c for c in sql_features_final
    if c not in model_sql_df.columns
]

print("=" * 80)
print("STEP 6 — FEATURE CHECK")
print("=" * 80)
print("Missing features:", missing_features)
print("Total model features:", len(sql_features_final))

if len(missing_features) > 0:
    raise ValueError(f"Missing features: {missing_features}")

# 2. KS function

def calculate_ks(y_true, y_score):

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_score
    )

    return np.max(tpr - fpr)

# 3. Year-by-year evaluation

def ranking_calibration_by_year(model, df, features):

    rows = []

    for year in sorted(df["Vintage"].dropna().unique()):

        year_df = df[df["Vintage"] == year].copy()

        X = year_df[features]
        y = year_df["Default_36M"]

        pd_pred = model.predict_proba(X)[:, 1]

        auc = roc_auc_score(y, pd_pred)
        pr_auc = average_precision_score(y, pd_pred)
        ks = calculate_ks(y, pd_pred)
        brier = brier_score_loss(y, pd_pred)

        n_top = max(1, int(np.ceil(len(year_df) * 0.10)))

        top_idx = np.argsort(pd_pred)[-n_top:]

        top_default_rate = y.iloc[top_idx].mean()

        overall_default_rate = y.mean()

        lift = (
            top_default_rate / overall_default_rate
            if overall_default_rate > 0
            else np.nan
        )

        avg_pd = pd_pred.mean()

        rows.append({
            "Vintage": year,
            "Loans": len(year_df),
            "Defaults": int(y.sum()),
            "Default_Rate_%": y.mean() * 100,
            "Average_PD_%": avg_pd * 100,
            "Calibration_Gap_pp": (
                y.mean() * 100 - avg_pd * 100
            ),
            "ROC_AUC": auc,
            "PR_AUC": pr_auc,
            "KS": ks,
            "Top_Decile_Default_%": top_default_rate * 100,
            "Top_Decile_Lift": lift,
            "Brier": brier
        })

    return pd.DataFrame(rows)

ranking_calibration = ranking_calibration_by_year(
    final_model,
    model_sql_df,
    sql_features_final
)

print("\n" + "=" * 80)
print("YEAR-BY-YEAR RANKING + CALIBRATION")
print("=" * 80)

display(
    ranking_calibration.round(4)
)


In [ ]:

def calculate_psi(expected, actual, bins=10):

    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    # Quantile bins based on reference population
    breakpoints = np.unique(
        np.percentile(
            expected,
            np.linspace(0, 100, bins + 1)
        )
    )

    if len(breakpoints) < 3:
        return 0.0

    expected_bins = pd.cut(
        expected,
        bins=breakpoints,
        include_lowest=True,
        duplicates="drop"
    )

    actual_bins = pd.cut(
        actual,
        bins=breakpoints,
        include_lowest=True,
        duplicates="drop"
    )

    expected_dist = (
        expected_bins.value_counts(normalize=True)
        .sort_index()
    )

    actual_dist = (
        actual_bins.value_counts(normalize=True)
        .reindex(expected_dist.index)
        .fillna(0)
    )

    eps = 1e-6

    expected_dist = expected_dist.clip(lower=eps)
    actual_dist = actual_dist.clip(lower=eps)

    psi = np.sum(
        (actual_dist - expected_dist)
        * np.log(actual_dist / expected_dist)
    )

    return float(psi)

numeric_psi_features = [
    'CreditScore',
    'OriginalDTI',
    'OriginalLTV',
    'OriginalCLTV',
    'OriginalUPB',
    'OriginalInterestRate',
    'OriginalLoanTerm',
    'NumberOfBorrowers',
    'NumberOfUnits',
    'MIPercent',
    'LowCreditFlag',
    'HighLTVFlag',
    'HighDTIFlag',
    'HighCLTVFlag',
    'HighRateFlag',
    'HighCreditLTVFlag',
    'HighDTILTVFlag'
]

reference_df = model_sql_df[
    model_sql_df["Vintage"].isin([2018, 2019, 2020, 2021])
].copy()

oot_df = model_sql_df[
    model_sql_df["Vintage"] == 2022
].copy()

psi_rows = []

for feature in numeric_psi_features:

    psi_value = calculate_psi(
        reference_df[feature],
        oot_df[feature]
    )

    if psi_value < 0.10:
        interpretation = "Stable"
    elif psi_value < 0.25:
        interpretation = "Moderate Shift"
    else:
        interpretation = "Significant Shift"

    psi_rows.append({
        "Feature": feature,
        "PSI": psi_value,
        "Interpretation": interpretation
    })

psi_results = (
    pd.DataFrame(psi_rows)
    .sort_values("PSI", ascending=False)
    .reset_index(drop=True)
)

print("=" * 80)
print("STEP 7 — FEATURE DISTRIBUTION SHIFT / PSI")
print("=" * 80)

display(
    psi_results.round(4)
)

print("\n" + "=" * 80)
print("LARGEST DISTRIBUTION SHIFTS")
print("=" * 80)

display(
    psi_results.head(10).round(4)
)


In [ ]:

concept_df = model_sql_df.copy()

concept_df["CreditScore_Bin"] = pd.cut(
    concept_df["CreditScore"],
    bins=[-np.inf, 649, 679, 719, 759, 799, np.inf],
    labels=["<650", "650-679", "680-719",
            "720-759", "760-799", "800+"]
)

concept_df["DTI_Bin"] = pd.cut(
    concept_df["OriginalDTI"],
    bins=[-np.inf, 19.99, 29.99, 35.99, 42.99, 49.99, np.inf],
    labels=["<20", "20-29", "30-35",
            "36-42", "43-49", "50+"]
)

concept_df["LTV_Bin"] = pd.cut(
    concept_df["OriginalLTV"],
    bins=[-np.inf, 59.99, 69.99, 79.99, 89.99, 99.99, np.inf],
    labels=["<60", "60-69", "70-79",
            "80-89", "90-99", "100+"]
)

concept_df["InterestRate_Bin"] = pd.cut(
    concept_df["OriginalInterestRate"],
    bins=[-np.inf, 1.99, 2.99, 3.99, 4.99, 5.99, np.inf],
    labels=["<2", "2-2.99", "3-3.99",
            "4-4.99", "5-5.99", "6-9.99"]
)

def default_rate_by_vintage(df, variable):

    result = (
        df
        .groupby(
            ["Vintage", variable],
            observed=False
        )
        .agg(
            Loans=("Default_36M", "size"),
            Defaults=("Default_36M", "sum"),
            Default_Rate=("Default_36M", "mean")
        )
        .reset_index()
    )

    result["Default_Rate_%"] = (
        result["Default_Rate"] * 100
    )

    return result

credit_drift = default_rate_by_vintage(
    concept_df,
    "CreditScore_Bin"
)

dti_drift = default_rate_by_vintage(
    concept_df,
    "DTI_Bin"
)

ltv_drift = default_rate_by_vintage(
    concept_df,
    "LTV_Bin"
)

rate_drift = default_rate_by_vintage(
    concept_df,
    "InterestRate_Bin"
)

print("=" * 80)
print("STEP 8 — CONCEPT DRIFT")
print("=" * 80)

print("\nCREDIT SCORE → DEFAULT RELATIONSHIP")
display(credit_drift.round(3))

print("\nDTI → DEFAULT RELATIONSHIP")
display(dti_drift.round(3))

print("\nLTV → DEFAULT RELATIONSHIP")
display(ltv_drift.round(3))

print("\nINTEREST RATE → DEFAULT RELATIONSHIP")
display(rate_drift.round(3))


In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

print("\n" + "=" * 80)
print("STEP 9 — REGIME ROBUSTNESS")
print("=" * 80)

# 1. DEFINE THE 25 SQL FEATURES

sql_features_final = [
    'Vintage',
    'CreditScore',
    'OriginalDTI',
    'OriginalLTV',
    'OriginalCLTV',
    'OriginalUPB',
    'OriginalInterestRate',
    'OriginalLoanTerm',
    'NumberOfBorrowers',
    'NumberOfUnits',
    'MIPercent',
    'FirstTimeHomebuyerFlag',
    'OccupancyStatus',
    'Channel',
    'PropertyType',
    'LoanPurpose',
    'AmortizationType',
    'PropertyState',
    'LowCreditFlag',
    'HighLTVFlag',
    'HighDTIFlag',
    'HighCLTVFlag',
    'HighRateFlag',
    'HighCreditLTVFlag',
    'HighDTILTVFlag'
]

# 2. CHECK FEATURES BEFORE PREDICTION

missing_features = [
    col for col in sql_features_final
    if col not in model_sql_df.columns
]

print("\nMissing features:")
print(missing_features)

if len(missing_features) > 0:
    raise ValueError(
        f"Missing required SQL features: {missing_features}"
    )

print(
    f"\nAll {len(sql_features_final)} SQL features are present."
)

# 3. CREATE REGIME DATAFRAME
# IMPORTANT:
# Use the ORIGINAL model_sql_df.
# Do NOT create regime_df from a reduced feature dataframe.

regime_df = model_sql_df.copy()

# 4. GENERATE MODEL PDs

print("\nGenerating model predictions...")

regime_df["PD"] = final_model.predict_proba(
    regime_df[sql_features_final]
)[:, 1]

print("PD generation successful.")

# 5. CREATE COVID / REGIME LABELS

regime_map = {
    2018: "Pre-COVID",
    2019: "Pre-COVID",
    2020: "COVID / Pandemic",
    2021: "COVID / Pandemic",
    2022: "Recovery / Post-COVID"
}

regime_df["Regime"] = regime_df["Vintage"].map(regime_map)

# 6. KS FUNCTION

def calculate_ks(y_true, pd_pred):

    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "pd": np.asarray(pd_pred)
    }).sort_values("pd", ascending=False)

    total_bad = temp["y"].sum()
    total_good = len(temp) - total_bad

    if total_bad == 0 or total_good == 0:
        return np.nan

    temp["cum_bad"] = temp["y"].cumsum() / total_bad
    temp["cum_good"] = (
        (1 - temp["y"]).cumsum() / total_good
    )

    temp["ks"] = (
        temp["cum_bad"] -
        temp["cum_good"]
    ).abs()

    return temp["ks"].max()

# 7. TOP DECILE LIFT

def calculate_top_decile_metrics(y_true, pd_pred):

    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "pd": np.asarray(pd_pred)
    }).sort_values("pd", ascending=False)

    n_top = max(1, int(np.ceil(len(temp) * 0.10)))

    top_decile = temp.iloc[:n_top]

    top_default_rate = top_decile["y"].mean()
    overall_default_rate = temp["y"].mean()

    if overall_default_rate == 0:
        lift = np.nan
    else:
        lift = (
            top_default_rate /
            overall_default_rate
        )

    return top_default_rate, lift

# 8. CALCULATE REGIME-LEVEL PERFORMANCE

regime_rows = []

for regime in [
    "Pre-COVID",
    "COVID / Pandemic",
    "Recovery / Post-COVID"
]:

    temp = regime_df[
        regime_df["Regime"] == regime
    ].copy()

    y = temp["Default_36M"]
    pd_pred = temp["PD"]

    if y.nunique() < 2:
        roc_auc = np.nan
        pr_auc = np.nan
        ks = np.nan
    else:
        roc_auc = roc_auc_score(
            y,
            pd_pred
        )

        pr_auc = average_precision_score(
            y,
            pd_pred
        )

        ks = calculate_ks(
            y,
            pd_pred
        )

    top_decile_default, top_decile_lift = (
        calculate_top_decile_metrics(
            y,
            pd_pred
        )
    )

    regime_rows.append({

        "Regime": regime,

        "Loans": len(temp),

        "Defaults": int(y.sum()),

        "Default_Rate_%": y.mean() * 100,

        "Average_PD_%": pd_pred.mean() * 100,

        "Calibration_Gap_pp":
            (pd_pred.mean() - y.mean()) * 100,

        "ROC_AUC": roc_auc,

        "PR_AUC": pr_auc,

        "KS": ks,

        "Top_Decile_Default_%":
            top_decile_default * 100,

        "Top_Decile_Lift":
            top_decile_lift
    })

regime_results = pd.DataFrame(regime_rows)

# 9. DISPLAY REGIME PERFORMANCE

print("\n" + "=" * 80)
print("REGIME-LEVEL PERFORMANCE")
print("=" * 80)

display(
    regime_results.round(4)
)

# 10. YEAR-BY-YEAR PERFORMANCE

year_rows = []

for year in sorted(regime_df["Vintage"].dropna().unique()):

    temp = regime_df[
        regime_df["Vintage"] == year
    ].copy()

    y = temp["Default_36M"]
    pd_pred = temp["PD"]

    roc_auc = roc_auc_score(
        y,
        pd_pred
    )

    pr_auc = average_precision_score(
        y,
        pd_pred
    )

    ks = calculate_ks(
        y,
        pd_pred
    )

    top_decile_default, top_decile_lift = (
        calculate_top_decile_metrics(
            y,
            pd_pred
        )
    )

    year_rows.append({

        "Vintage": year,

        "Loans": len(temp),

        "Defaults": int(y.sum()),

        "Default_Rate_%":
            y.mean() * 100,

        "Average_PD_%":
            pd_pred.mean() * 100,

        "Calibration_Gap_pp":
            (pd_pred.mean() - y.mean()) * 100,

        "ROC_AUC": roc_auc,

        "PR_AUC": pr_auc,

        "KS": ks,

        "Top_Decile_Default_%":
            top_decile_default * 100,

        "Top_Decile_Lift":
            top_decile_lift,

        "Brier":
            brier_score_loss(
                y,
                pd_pred
            )
    })

year_results = pd.DataFrame(year_rows)

# 11. DISPLAY YEAR RESULTS

print("\n" + "=" * 80)
print("YEAR-BY-YEAR PERFORMANCE")
print("=" * 80)

display(
    year_results.round(4)
)

# 12. 2022 OOT DIAGNOSTIC

test_2022 = regime_df[
    regime_df["Vintage"] == 2022
].copy()

y_2022 = test_2022["Default_36M"]
pd_2022 = test_2022["PD"]

roc_2022 = roc_auc_score(
    y_2022,
    pd_2022
)

pr_2022 = average_precision_score(
    y_2022,
    pd_2022
)

ks_2022 = calculate_ks(
    y_2022,
    pd_2022
)

top_decile_default_2022, lift_2022 = (
    calculate_top_decile_metrics(
        y_2022,
        pd_2022
    )
)

actual_default_2022 = y_2022.mean()

average_pd_2022 = pd_2022.mean()

calibration_gap_2022 = (
    average_pd_2022 -
    actual_default_2022
) * 100

# 13. FINAL 2022 SUMMARY

print("\n" + "=" * 80)
print("2022 OUT-OF-TIME SUMMARY")
print("=" * 80)

print(
    f"ROC-AUC              : {roc_2022:.4f}"
)

print(
    f"PR-AUC               : {pr_2022:.4f}"
)

print(
    f"KS                   : {ks_2022:.4f}"
)

print(
    f"Actual default rate  : "
    f"{actual_default_2022 * 100:.4f}%"
)

print(
    f"Average predicted PD : "
    f"{average_pd_2022 * 100:.4f}%"
)

print(
    f"Calibration gap      : "
    f"{calibration_gap_2022:.2f} pp"
)

print(
    f"Top-decile default   : "
    f"{top_decile_default_2022 * 100:.4f}%"
)

print(
    f"Top-decile lift      : "
    f"{lift_2022:.2f}x"
)

# 14. FINAL REGIME INTERPRETATION

print("\n" + "=" * 80)
print("REGIME ROBUSTNESS INTERPRETATION")
print("=" * 80)

print(
    "The regime test compares model discrimination and PD "
    "calibration across materially different origination periods."
)

print(
    "COVID / Pandemic is treated as a robustness regime only; "
    "this does not imply that COVID caused the observed defaults."
)

print(
    "Recovery / Post-COVID is evaluated separately because "
    "2022 exhibits materially different feature conditions."
)

print(
    "\nIMPORTANT:"
)

print(
    "A strong ROC-AUC or lift indicates that the model can still "
    "rank borrowers by relative risk."
)

print(
    "A large calibration gap indicates that the absolute PD levels "
    "are not reliable for that regime."
)


In [ ]:

overall_y = model_sql_df["Default_36M"]

overall_pd = final_model.predict_proba(
    model_sql_df[sql_features_final]
)[:, 1]

overall_auc = roc_auc_score(
    overall_y,
    overall_pd
)

overall_pr_auc = average_precision_score(
    overall_y,
    overall_pd
)

overall_ks = calculate_ks(
    overall_y,
    overall_pd
)

test_2022 = model_sql_df[
    model_sql_df["Vintage"] == 2022
].copy()

pd_2022 = final_model.predict_proba(
    test_2022[sql_features_final]
)[:, 1]

y_2022 = test_2022["Default_36M"]

roc_2022 = roc_auc_score(
    y_2022,
    pd_2022
)

pr_2022 = average_precision_score(
    y_2022,
    pd_2022
)

ks_2022 = calculate_ks(
    y_2022,
    pd_2022
)

n_top_2022 = max(
    1,
    int(np.ceil(len(test_2022) * 0.10))
)

top_idx_2022 = np.argsort(pd_2022)[-n_top_2022:]

top_default_2022 = (
    y_2022.iloc[top_idx_2022].mean()
)

lift_2022 = (
    top_default_2022 / y_2022.mean()
)

avg_pd_2022 = pd_2022.mean()

calibration_gap_2022 = (
    avg_pd_2022 * 100
    - y_2022.mean() * 100
)

max_psi_value = psi_results["PSI"].max()
max_psi_feature = psi_results.iloc[0]["Feature"]

print("=" * 80)
print("STEP 10 — FINAL MODEL DIAGNOSIS")
print("=" * 80)

print("\nOVERALL MODEL")
print("-" * 50)
print(f"ROC-AUC : {overall_auc:.4f}")
print(f"PR-AUC  : {overall_pr_auc:.4f}")
print(f"KS      : {overall_ks:.4f}")

print("\n2022 OUT-OF-TIME PERFORMANCE")
print("-" * 50)
print(f"ROC-AUC : {roc_2022:.4f}")
print(f"PR-AUC  : {pr_2022:.4f}")
print(f"KS      : {ks_2022:.4f}")
print(f"Actual default rate : {y_2022.mean()*100:.2f}%")
print(f"Average predicted PD : {avg_pd_2022*100:.2f}%")
print(f"Calibration gap : {calibration_gap_2022:.2f} pp")
print(f"Top-decile default rate : {top_default_2022*100:.2f}%")
print(f"Top-decile lift : {lift_2022:.2f}x")

print("\nFEATURE DRIFT")
print("-" * 50)
print(f"Largest PSI feature : {max_psi_feature}")
print(f"Largest PSI : {max_psi_value:.4f}")

print("\n" + "=" * 80)
print("FINAL INTERPRETATION")
print("=" * 80)

if roc_2022 >= 0.70:
    print("✓ 2022 ranking performance remains economically useful.")
else:
    print("⚠ 2022 ranking performance is materially weaker.")

if lift_2022 >= 2:
    print("✓ Top-decile risk separation remains meaningful.")
else:
    print("⚠ Top-decile risk separation is weak.")

if abs(calibration_gap_2022) <= 1:
    print("✓ 2022 PD calibration is reasonably close.")
else:
    print("⚠ 2022 PD calibration is materially distorted.")

if max_psi_value < 0.10:
    print("✓ No major feature distribution shift detected.")
elif max_psi_value < 0.25:
    print("⚠ Moderate feature distribution shift detected.")
else:
    print("⚠ Significant feature distribution shift detected.")

print("\nConclusion:")
print(
    "The model should be evaluated using discrimination, "
    "temporal stability, distribution shift, concept drift "
    "and PD calibration."
)


## 8. Seasoning and right-censoring checks

Check that the 36-month default definition is not being driven by incomplete loan histories.

In [ ]:

print("=" * 80)
print("STEP 11 — 36-MONTH SEASONING CHECK")
print("=" * 80)

print("\nDataset columns:")
print(model_sql_df.columns.tolist())

# Look for date/month variables
date_candidates = [
    c for c in model_sql_df.columns
    if any(
        term in c.lower()
        for term in [
            "date",
            "month",
            "period",
            "origination",
            "firstpay",
            "performance"
        ]
    )
]

print("\nPotential date / performance columns:")
print(date_candidates)

# Check Vintage distribution

print("\nVintage counts:")
display(
    model_sql_df["Vintage"]
    .value_counts()
    .sort_index()
)

# Check whether a 36-month eligibility variable already exists

possible_eligibility = [
    c for c in model_sql_df.columns
    if any(
        term in c.lower()
        for term in [
            "months_observed",
            "months_since",
            "season",
            "maturity",
            "observation",
            "eligible",
            "36m"
        ]
    )
]

print("\nPotential seasoning/eligibility columns:")
print(possible_eligibility)

# Basic label audit by vintage

seasoning_audit = (
    model_sql_df
    .groupby("Vintage")
    .agg(
        Loans=("Default_36M", "size"),
        Defaults=("Default_36M", "sum"),
        Default_Rate=("Default_36M", "mean")
    )
    .reset_index()
)

seasoning_audit["Default_Rate_%"] = (
    seasoning_audit["Default_Rate"] * 100
)

print("\n" + "=" * 80)
print("36-MONTH LABEL AUDIT BY VINTAGE")
print("=" * 80)

display(
    seasoning_audit.round(4)
)


In [ ]:
import glob
import os
import pandas as pd

print("=" * 80)
print("RAW FREDDIE MAC PERFORMANCE DATA CUTOFF CHECK")
print("=" * 80)

# The notebook expects raw performance files under ./data/raw/.
perf_files = glob.glob(
    r"./data/raw/sample_perf_*.txt"
)

print("\nFound performance files:")
for f in perf_files:
    print(" -", f)

if len(perf_files) == 0:
    print("\n⚠️ NO FILES FOUND.")
    print("Check the folder path and filename pattern.")
else:

    max_reporting_periods = {}

    for f in perf_files:

        try:
            df_perf = pd.read_csv(
                f,
                sep="|",
                header=None,
                usecols=[0, 1],
                names=[
                    "LoanSequenceNumber",
                    "MonthlyReportingPeriod"
                ],
                dtype={
                    "LoanSequenceNumber": "string",
                    "MonthlyReportingPeriod": "string"
                }
            )

            # Remove missing / malformed reporting periods
            periods = (
                df_perf["MonthlyReportingPeriod"]
                .dropna()
                .astype(str)
                .str.strip()
            )

            periods = periods[
                periods.str.match(r"^\d{6}$")
            ]

            if len(periods) == 0:
                print(
                    f"\n⚠️ No valid reporting periods found in:\n{f}"
                )
                continue

            max_period = periods.max()

            max_reporting_periods[f] = max_period

            print(
                f"\n{os.path.basename(f)}"
            )
            print(
                f"Maximum Monthly Reporting Period: {max_period}"
            )

        except Exception as e:

            print(
                f"\n⚠️ ERROR reading {f}"
            )
            print(e)

    # GLOBAL CUTOFF

    if len(max_reporting_periods) > 0:

        global_max = max(
            max_reporting_periods.values()
        )

        print("\n" + "=" * 80)
        print("GLOBAL PERFORMANCE DATA CUTOFF")
        print("=" * 80)

        print(
            f"Latest Monthly Reporting Period: {global_max}"
        )

        # Convert YYYYMM to timestamp
        global_cutoff = pd.to_datetime(
            global_max,
            format="%Y%m"
        )

        print(
            f"Latest reporting month: "
            f"{global_cutoff.strftime('%B %Y')}"
        )

        print("\n" + "=" * 80)
        print("36-MONTH SEASONING REQUIREMENTS")
        print("=" * 80)

        for year in [2018, 2019, 2020, 2021, 2022]:

            # Approximate latest possible 36M requirement
            # for December originations of that vintage.
            latest_origination = pd.Timestamp(
                year=year,
                month=12,
                day=1
            )

            required_cutoff = (
                latest_origination
                + pd.DateOffset(months=36)
            )

            if global_cutoff >= required_cutoff:

                status = "✓ FULL 36M WINDOW"

            else:

                status = "⚠ INCOMPLETE 36M WINDOW"

            print(
                f"{year}: "
                f"required through "
                f"{required_cutoff.strftime('%Y%m')} | "
                f"actual cutoff {global_max} | "
                f"{status}"
            )

        # IMPORTANT: 2022

        print("\n" + "=" * 80)
        print("2022 VINTAGE CHECK")
        print("=" * 80)

        december_2022 = pd.Timestamp(
            year=2022,
            month=12,
            day=1
        )

        required_2022 = (
            december_2022
            + pd.DateOffset(months=36)
        )

        print(
            "Latest possible 2022 origination: "
            "December 2022"
        )

        print(
            "Required performance through: "
            f"{required_2022.strftime('%B %Y')}"
        )

        print(
            "Actual performance cutoff: "
            f"{global_cutoff.strftime('%B %Y')}"
        )

        if global_cutoff >= required_2022:

            print(
                "\n✓ The raw performance data is sufficiently "
                "long to potentially support a full 36M window "
                "for December 2022 originations."
            )

        else:

            print(
                "\n⚠️ The raw performance data does NOT provide "
                "a full 36M window for late-2022 originations."
            )

            print(
                "The 2022 Default_36M rate may therefore be "
                "affected by right-censoring."
            )

        print("\n" + "=" * 80)
        print("NEXT STEP")
        print("=" * 80)

        print(
            "Do NOT rebuild the model yet."
        )

        print(
            "First inspect the global cutoff above."
        )

        print(
            "If the cutoff is before 2025-12, "
            "we need to determine exactly which 2022 loans "
            "have a complete 36-month observation window."
        )


In [ ]:

import glob
import os
import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 12 — LOAN-LEVEL 36-MONTH SEASONING AUDIT")
print("=" * 80)

# 1. FIND RAW PERFORMANCE FILES

perf_files = glob.glob(
    r"./data/raw/sample_perf_*.txt"
)

print("\nPerformance files found:")

for f in perf_files:
    print(" -", f)

if len(perf_files) == 0:
    raise FileNotFoundError(
        "No sample_perf_*.txt files found. Check the folder path."
    )

required_cols = [
    "LoanSequenceNumber",
    "Vintage",
    "Default_36M"
]

missing_cols = [
    c for c in required_cols
    if c not in model_sql_df.columns
]

if missing_cols:
    raise KeyError(
        f"model_sql_df is missing: {missing_cols}"
    )

model_loans = (
    model_sql_df[
        required_cols
    ]
    .drop_duplicates(
        subset=["LoanSequenceNumber"]
    )
    .copy()
)

model_loans["LoanSequenceNumber"] = (
    model_loans["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

model_loans["Vintage"] = pd.to_numeric(
    model_loans["Vintage"],
    errors="coerce"
)

model_loans = model_loans[
    model_loans["Vintage"].isin(
        [2018, 2019, 2020, 2021, 2022]
    )
].copy()

print(
    f"\nModel loans being audited: "
    f"{len(model_loans):,}"
)

print(
    "\nModel loans by vintage:"
)

print(
    model_loans["Vintage"].value_counts().sort_index()
)

# 3. READ ONLY TWO RAW PERFORMANCE COLUMNS

loan_first_dates = {}
loan_last_dates = {}

for f in perf_files:

    print("\n" + "-" * 70)
    print("Reading:", os.path.basename(f))
    print("-" * 70)

    # ONLY READ FIRST TWO COLUMNS
    perf = pd.read_csv(
        f,
        sep="|",
        header=None,
        usecols=[0, 1],
        names=[
            "LoanSequenceNumber",
            "MonthlyReportingPeriod"
        ],
        dtype={
            "LoanSequenceNumber": "string",
            "MonthlyReportingPeriod": "string"
        }
    )

    print(
        f"Raw performance rows: {len(perf):,}"
    )

    # Clean IDs
    perf["LoanSequenceNumber"] = (
        perf["LoanSequenceNumber"]
        .str.strip()
    )

    # Clean reporting period
    perf["MonthlyReportingPeriod"] = (
        perf["MonthlyReportingPeriod"]
        .str.strip()
    )

    # Keep only valid YYYYMM
    valid_period = (
        perf["MonthlyReportingPeriod"]
        .str.match(r"^\d{6}$", na=False)
    )

    perf = perf.loc[
        valid_period,
        [
            "LoanSequenceNumber",
            "MonthlyReportingPeriod"
        ]
    ]

    # Convert YYYYMM directly to datetime
    perf["ReportingDate"] = pd.to_datetime(
        perf["MonthlyReportingPeriod"],
        format="%Y%m",
        errors="coerce"
    )

    perf = perf.dropna(
        subset=["ReportingDate"]
    )

    # IMPORTANT:
    # Only keep loans that actually occur in our modeling dataset.

    relevant_ids = set(
        model_loans["LoanSequenceNumber"]
    )

    perf = perf[
        perf["LoanSequenceNumber"].isin(
            relevant_ids
        )
    ]

    print(
        f"Rows belonging to model loans: "
        f"{len(perf):,}"
    )

    if len(perf) == 0:
        print(
            "⚠ No matching model loans found in this file."
        )
        continue

    # Get first and last reporting month PER LOAN

    file_first = (
        perf
        .groupby("LoanSequenceNumber")["ReportingDate"]
        .min()
    )

    file_last = (
        perf
        .groupby("LoanSequenceNumber")["ReportingDate"]
        .max()
    )

    # Merge into dictionaries
    for loan, date in file_first.items():

        if (
            loan not in loan_first_dates
            or date < loan_first_dates[loan]
        ):
            loan_first_dates[loan] = date

    for loan, date in file_last.items():

        if (
            loan not in loan_last_dates
            or date > loan_last_dates[loan]
        ):
            loan_last_dates[loan] = date

    print(
        f"Unique model loans found: "
        f"{perf['LoanSequenceNumber'].nunique():,}"
    )

    # Free memory immediately
    del perf
    del file_first
    del file_last

# 4. CREATE LOAN-LEVEL AUDIT TABLE

loan_audit = model_loans.copy()

loan_audit["First_Reporting_Date"] = (
    loan_audit["LoanSequenceNumber"]
    .map(loan_first_dates)
)

loan_audit["Last_Reporting_Date"] = (
    loan_audit["LoanSequenceNumber"]
    .map(loan_last_dates)
)

# 5. CALCULATE 36-MONTH REQUIREMENT

loan_audit["Required_36M_Date"] = (
    loan_audit["First_Reporting_Date"]
    + pd.DateOffset(months=36)
)

# Number of months actually observed
loan_audit["Months_Observed"] = np.nan

valid_dates = (
    loan_audit["First_Reporting_Date"].notna()
    &
    loan_audit["Last_Reporting_Date"].notna()
)

loan_audit.loc[
    valid_dates,
    "Months_Observed"
] = (
    (
        loan_audit.loc[
            valid_dates,
            "Last_Reporting_Date"
        ].dt.year
        -
        loan_audit.loc[
            valid_dates,
            "First_Reporting_Date"
        ].dt.year
    ) * 12
    +
    (
        loan_audit.loc[
            valid_dates,
            "Last_Reporting_Date"
        ].dt.month
        -
        loan_audit.loc[
            valid_dates,
            "First_Reporting_Date"
        ].dt.month
    )
)

loan_audit["Has_36M_Window"] = (
    loan_audit["Last_Reporting_Date"]
    >= loan_audit["Required_36M_Date"]
)

# 6. PERFORMANCE DATA COVERAGE

loan_audit["Performance_Data_Found"] = (
    loan_audit["Last_Reporting_Date"].notna()
)

print("\n" + "=" * 80)
print("PERFORMANCE DATA COVERAGE")
print("=" * 80)

print(
    f"Model loans: "
    f"{len(loan_audit):,}"
)

print(
    f"Loans found in raw performance files: "
    f"{loan_audit['Performance_Data_Found'].sum():,}"
)

print(
    f"Loans NOT found in raw performance files: "
    f"{(~loan_audit['Performance_Data_Found']).sum():,}"
)

# 7. 36-MONTH ELIGIBILITY BY VINTAGE

seasoning_summary = (
    loan_audit
    .groupby("Vintage")
    .agg(
        Loans=(
            "LoanSequenceNumber",
            "count"
        ),

        Performance_Data_Found=(
            "Performance_Data_Found",
            "sum"
        ),

        Complete_36M_Window=(
            "Has_36M_Window",
            "sum"
        ),

        Incomplete_36M_Window=(
            "Has_36M_Window",
            lambda x: (~x).sum()
        ),

        Average_Months_Observed=(
            "Months_Observed",
            "mean"
        ),

        Minimum_Months_Observed=(
            "Months_Observed",
            "min"
        ),

        Maximum_Months_Observed=(
            "Months_Observed",
            "max"
        )
    )
    .reset_index()
)

seasoning_summary["Pct_36M_Complete"] = (
    seasoning_summary["Complete_36M_Window"]
    /
    seasoning_summary["Loans"]
    * 100
)

seasoning_summary["Pct_36M_Incomplete"] = (
    seasoning_summary["Incomplete_36M_Window"]
    /
    seasoning_summary["Loans"]
    * 100
)

print("\n" + "=" * 80)
print("36-MONTH ELIGIBILITY BY VINTAGE")
print("=" * 80)

display(
    seasoning_summary.round(2)
)

# 8. 2022-SPECIFIC CHECK

audit_2022 = loan_audit[
    loan_audit["Vintage"] == 2022
].copy()

print("\n" + "=" * 80)
print("2022 LOAN-LEVEL SEASONING AUDIT")
print("=" * 80)

print(
    f"Total 2022 loans: "
    f"{len(audit_2022):,}"
)

print(
    f"Performance records found: "
    f"{audit_2022['Performance_Data_Found'].sum():,}"
)

print(
    f"Complete 36M window: "
    f"{audit_2022['Has_36M_Window'].sum():,}"
)

print(
    f"Incomplete 36M window: "
    f"{(~audit_2022['Has_36M_Window']).sum():,}"
)

if len(audit_2022) > 0:

    print(
        f"Percentage fully seasoned: "
        f"{audit_2022['Has_36M_Window'].mean() * 100:.2f}%"
    )

# 9. DEFAULT RATE BY SEASONING STATUS

seasoning_default = (
    loan_audit
    .groupby(
        [
            "Vintage",
            "Has_36M_Window"
        ]
    )
    .agg(
        Loans=(
            "LoanSequenceNumber",
            "count"
        ),

        Defaults=(
            "Default_36M",
            "sum"
        ),

        Default_Rate=(
            "Default_36M",
            "mean"
        )
    )
    .reset_index()
)

seasoning_default["Default_Rate_%"] = (
    seasoning_default["Default_Rate"] * 100
)

print("\n" + "=" * 80)
print("DEFAULT RATE BY 36-MONTH ELIGIBILITY")
print("=" * 80)

display(
    seasoning_default.round(4)
)

# 10. CRITICAL 2022 DIAGNOSIS

print("\n" + "=" * 80)
print("2022 RIGHT-CENSORING DIAGNOSIS")
print("=" * 80)

if len(audit_2022) == 0:

    print(
        "⚠ No 2022 loans were found in the raw "
        "performance files."
    )

elif audit_2022["Has_36M_Window"].mean() >= 0.99:

    print(
        "✓ 2022 is essentially fully seasoned."
    )

    print(
        "✓ Right-censoring is very unlikely to explain "
        "the 2022 default-rate/calibration result."
    )

elif audit_2022["Has_36M_Window"].mean() >= 0.90:

    print(
        "⚠ Most 2022 loans are fully seasoned, "
        "but some loans lack a complete 36M window."
    )

    print(
        "The final OOT test should be repeated using "
        "only fully seasoned 2022 loans."
    )

else:

    print(
        "⚠ A substantial share of 2022 loans lack "
        "a complete 36M window."
    )

    print(
        "The current 2022 default rate should NOT "
        "be treated as a clean 36M outcome."
    )

# 11. SHOW UNSEASONED 2022 LOANS

unseasoned_2022 = audit_2022[
    ~audit_2022["Has_36M_Window"]
].copy()

print("\n" + "=" * 80)
print("UNSEASONED 2022 LOANS")
print("=" * 80)

if len(unseasoned_2022) > 0:

    display(
        unseasoned_2022[
            [
                "LoanSequenceNumber",
                "Vintage",
                "First_Reporting_Date",
                "Last_Reporting_Date",
                "Required_36M_Date",
                "Months_Observed",
                "Default_36M"
            ]
        ]
        .sort_values(
            "Months_Observed"
        )
        .head(25)
    )

else:

    print(
        "✓ No unseasoned 2022 loans."
    )

# 12. SAVE OBJECTS FOR NEXT STEP

seasoning_audit = loan_audit.copy()

print("\n" + "=" * 80)
print("STEP 12 COMPLETE")
print("=" * 80)

print(
    "No model was rebuilt."
)

print(
    "No predictions were changed."
)

print(
    "This step only audits whether Default_36M "
    "has a complete observation window."
)


In [ ]:

print("=" * 80)
print("STEP 12A — LOCATING LOAN SEQUENCE NUMBER")
print("=" * 80)

# Check all currently defined DataFrames
import pandas as pd

found = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        if "LoanSequenceNumber" in obj.columns:
            found.append((name, len(obj), obj.shape))

print("\nDataFrames containing LoanSequenceNumber:")
print("-" * 80)

if found:
    for item in found:
        print(
            f"DataFrame: {item[0]} | "
            f"Rows: {item[1]:,} | "
            f"Shape: {item[2]}"
        )
else:
    print("NONE FOUND")

print("\nmodel_sql_df columns:")
print("-" * 80)
print(model_sql_df.columns.tolist())

print("\nPossible loan-ID columns in model_sql_df:")
print("-" * 80)

possible_id_cols = [
    c for c in model_sql_df.columns
    if any(
        term in c.lower()
        for term in [
            "loan",
            "sequence",
            "id",
            "number"
        ]
    )
]

print(possible_id_cols)


In [ ]:

import pandas as pd
import numpy as np
import glob
import os

print("=" * 80)
print("STEP 12 — LOAN-LEVEL 36-MONTH SEASONING AUDIT")
print("=" * 80)

# 1. FIND RAW PERFORMANCE FILES

perf_files = glob.glob(
    r"./data/raw/sample_perf_*.txt"
)

print("\nPerformance files found:")
for f in perf_files:
    print(" -", f)

if not perf_files:
    raise FileNotFoundError(
        "No sample_perf_*.txt files found."
    )

# 2. USE model_df — IT STILL CONTAINS LoanSequenceNumber

required_model_cols = [
    "LoanSequenceNumber",
    "Vintage",
    "Default_36M"
]

missing_model_cols = [
    c for c in required_model_cols
    if c not in model_df.columns
]

if missing_model_cols:
    raise KeyError(
        f"model_df is missing: {missing_model_cols}"
    )

model_loans = model_df[
    required_model_cols
].copy()

model_loans["LoanSequenceNumber"] = (
    model_loans["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

model_loans["Vintage"] = pd.to_numeric(
    model_loans["Vintage"],
    errors="coerce"
)

model_loans["Default_36M"] = pd.to_numeric(
    model_loans["Default_36M"],
    errors="coerce"
)

model_loans = model_loans.dropna(
    subset=[
        "LoanSequenceNumber",
        "Vintage",
        "Default_36M"
    ]
).copy()

print("\nModel loans available:", len(model_loans))

print(
    "\nModel loans by vintage:"
)

print(
    model_loans["Vintage"]
    .value_counts()
    .sort_index()
)

# 3. READ ONLY THE TWO REQUIRED COLUMNS FROM PERFORMANCE FILES

performance_parts = []

for f in perf_files:

    print("\nReading:", os.path.basename(f))

    # Freddie Mac performance:
    # column 0 = Loan Sequence Number
    # column 1 = Monthly Reporting Period

    temp = pd.read_csv(
        f,
        sep="|",
        header=None,
        usecols=[0, 1],
        names=[
            "LoanSequenceNumber",
            "MonthlyReportingPeriod"
        ],
        dtype={
            "LoanSequenceNumber": "string",
            "MonthlyReportingPeriod": "string"
        }
    )

    print("Rows:", f"{len(temp):,}")

    temp["LoanSequenceNumber"] = (
        temp["LoanSequenceNumber"]
        .str.strip()
    )

    temp["MonthlyReportingPeriod"] = (
        temp["MonthlyReportingPeriod"]
        .str.strip()
    )

    performance_parts.append(temp)

# Combine ONLY two columns
performance_id = pd.concat(
    performance_parts,
    ignore_index=True
)

del performance_parts

print(
    "\nCombined performance rows:",
    f"{len(performance_id):,}"
)

# 4. CLEAN REPORTING PERIOD WITHOUT CREATING A LARGE DATETIME COLUMN

performance_id["ReportingYear"] = pd.to_numeric(
    performance_id["MonthlyReportingPeriod"].str[:4],
    errors="coerce"
)

performance_id["ReportingMonth"] = pd.to_numeric(
    performance_id["MonthlyReportingPeriod"].str[4:6],
    errors="coerce"
)

performance_id = performance_id.dropna(
    subset=[
        "LoanSequenceNumber",
        "ReportingYear",
        "ReportingMonth"
    ]
)

print(
    "Valid performance observations:",
    f"{len(performance_id):,}"
)

# 5. FIND THE LAST OBSERVED MONTH FOR EACH LOAN

loan_last_reporting = (
    performance_id
    .groupby("LoanSequenceNumber", sort=False)
    .agg(
        LastReportingYear=("ReportingYear", "max"),
        LastReportingMonth=("ReportingMonth", "max")
    )
    .reset_index()
)

del performance_id

print(
    "\nUnique loans with performance history:",
    f"{len(loan_last_reporting):,}"
)

# 6. MERGE LAST PERFORMANCE DATE BACK TO MODEL LOANS

seasoning = model_loans.merge(
    loan_last_reporting,
    on="LoanSequenceNumber",
    how="left"
)

del loan_last_reporting

# 7. CALCULATE REQUIRED 36-MONTH END DATE

# Vintage is the origination YEAR.
# We do NOT know the exact origination month from model_df.
#
# Therefore:
# - A 2022 loan could have originated anywhere from Jan-Dec 2022.
# - To guarantee 36 months for EVERY 2022 loan,
#   performance must reach Dec-2025.
#
# We therefore use the conservative full-vintage test.

seasoning["RequiredReportingYear"] = (
    seasoning["Vintage"] + 3
)

seasoning["RequiredReportingMonth"] = 12

# Convert year/month to integer YYYYMM
seasoning["RequiredYYYYMM"] = (
    seasoning["RequiredReportingYear"] * 100
    + seasoning["RequiredReportingMonth"]
)

seasoning["LastYYYYMM"] = (
    seasoning["LastReportingYear"] * 100
    + seasoning["LastReportingMonth"]
)

# 8. FULL 36-MONTH ELIGIBILITY

seasoning["Full_36M_Window"] = (
    seasoning["LastYYYYMM"]
    >= seasoning["RequiredYYYYMM"]
)

# 9. OVERALL AUDIT

print("\n" + "=" * 80)
print("36-MONTH SEASONING ELIGIBILITY")
print("=" * 80)

print(
    "\nFully seasoned loans:",
    f"{seasoning['Full_36M_Window'].sum():,}"
)

print(
    "Not fully seasoned:",
    f"{(~seasoning['Full_36M_Window']).sum():,}"
)

print(
    "Missing performance history:",
    f"{seasoning['LastYYYYMM'].isna().sum():,}"
)

# 10. VINTAGE-LEVEL SEASONING SUMMARY

seasoning_summary = (
    seasoning
    .groupby("Vintage")
    .agg(
        Loans=("LoanSequenceNumber", "size"),
        Fully_Seasoned=("Full_36M_Window", "sum"),
        Not_Fully_Seasoned=(
            "Full_36M_Window",
            lambda x: (~x).sum()
        ),
        Min_Last_YYYYMM=("LastYYYYMM", "min"),
        Max_Last_YYYYMM=("LastYYYYMM", "max")
    )
    .reset_index()
)

seasoning_summary["Fully_Seasoned_%"] = (
    seasoning_summary["Fully_Seasoned"]
    / seasoning_summary["Loans"]
    * 100
)

print(
    "\nVintage-level seasoning summary:"
)

display(
    seasoning_summary.round(2)
)

# 11. DEFAULT RATE — FULLY SEASONED VS ALL LOANS

seasoned_default_summary = (
    seasoning
    .groupby(
        ["Vintage", "Full_36M_Window"],
        dropna=False
    )
    .agg(
        Loans=("LoanSequenceNumber", "size"),
        Defaults=("Default_36M", "sum"),
        Default_Rate=("Default_36M", "mean")
    )
    .reset_index()
)

seasoned_default_summary["Default_Rate_%"] = (
    seasoned_default_summary["Default_Rate"] * 100
)

print("\n" + "=" * 80)
print("DEFAULT RATE — SEASONED VS NOT FULLY SEASONED")
print("=" * 80)

display(
    seasoned_default_summary.round(4)
)

# 12. FOCUS ON 2022

seasoning_2022 = seasoning[
    seasoning["Vintage"] == 2022
].copy()

print("\n" + "=" * 80)
print("2022 VINTAGE — 36-MONTH SEASONING CHECK")
print("=" * 80)

print(
    "\n2022 loans:",
    f"{len(seasoning_2022):,}"
)

print(
    "Fully seasoned 2022 loans:",
    f"{seasoning_2022['Full_36M_Window'].sum():,}"
)

print(
    "Not fully seasoned 2022 loans:",
    f"{(~seasoning_2022['Full_36M_Window']).sum():,}"
)

print(
    "2022 fully seasoned percentage:",
    f"{seasoning_2022['Full_36M_Window'].mean() * 100:.2f}%"
)

# 13. 2022 DEFAULT RATE AMONG FULLY SEASONED LOANS

if seasoning_2022["Full_36M_Window"].any():

    seasoned_2022 = seasoning_2022[
        seasoning_2022["Full_36M_Window"]
    ]

    print(
        "\n2022 fully seasoned loans:",
        f"{len(seasoned_2022):,}"
    )

    print(
        "2022 fully seasoned defaults:",
        f"{seasoned_2022['Default_36M'].sum():,}"
    )

    print(
        "2022 fully seasoned default rate:",
        f"{seasoned_2022['Default_36M'].mean() * 100:.4f}%"
    )

else:

    print(
        "\nWARNING: No fully seasoned 2022 loans found."
    )

# 14. CHECK 2021 AS WELL

seasoning_2021 = seasoning[
    seasoning["Vintage"] == 2021
].copy()

print("\n" + "=" * 80)
print("2021 VINTAGE — 36-MONTH SEASONING CHECK")
print("=" * 80)

print(
    "\n2021 loans:",
    f"{len(seasoning_2021):,}"
)

print(
    "Fully seasoned 2021 loans:",
    f"{seasoning_2021['Full_36M_Window'].sum():,}"
)

print(
    "Not fully seasoned 2021 loans:",
    f"{(~seasoning_2021['Full_36M_Window']).sum():,}"
)

print(
    "2021 fully seasoned percentage:",
    f"{seasoning_2021['Full_36M_Window'].mean() * 100:.2f}%"
)

# 15. FINAL DIAGNOSTIC

print("\n" + "=" * 80)
print("STEP 12 — FINAL SEASONING DIAGNOSIS")
print("=" * 80)

fully_2022_pct = (
    seasoning_2022["Full_36M_Window"].mean() * 100
    if len(seasoning_2022) > 0
    else 0
)

fully_2021_pct = (
    seasoning_2021["Full_36M_Window"].mean() * 100
    if len(seasoning_2021) > 0
    else 0
)

print(
    f"\n2021 fully seasoned : {fully_2021_pct:.2f}%"
)

print(
    f"2022 fully seasoned : {fully_2022_pct:.2f}%"
)

if fully_2022_pct >= 99:

    print(
        "\n✓ 2022 has a complete 36-month observation window "
        "under the conservative vintage-level test."
    )

    print(
        "The 2022 calibration problem is therefore NOT "
        "explained simply by insufficient seasoning."
    )

elif fully_2022_pct > 0:

    print(
        "\n⚠ 2022 is only partially fully seasoned."
    )

    print(
        "The OOT calibration result should be re-evaluated "
        "using only fully seasoned loans."
    )

else:

    print(
        "\n⚠ 2022 has no fully seasoned loans."
    )

    print(
        "The 2022 OOT calibration result cannot yet be "
        "treated as a valid 36-month comparison."
    )

print(
    "\nIMPORTANT:"
)

print(
    "This audit uses the conservative assumption that every "
    "loan in a vintage must have performance through December "
    "of Vintage + 3 years."
)

print(
    "This avoids falsely declaring a recent vintage fully "
    "seasoned when the exact origination month is unavailable."
)


In [ ]:

print("=" * 80)
print("STEP 12B — PERFORMANCE COVERAGE BY VINTAGE")
print("=" * 80)

coverage = (
    seasoning
    .groupby("Vintage")
    .agg(
        Model_Loans=("LoanSequenceNumber", "size"),
        Matched_Performance_Loans=(
            "LastYYYYMM",
            lambda x: x.notna().sum()
        ),
        Fully_Seasoned_Loans=(
            "Full_36M_Window",
            lambda x: (x == True).sum()
        )
    )
    .reset_index()
)

coverage["Performance_Match_%"] = (
    coverage["Matched_Performance_Loans"]
    / coverage["Model_Loans"]
    * 100
)

coverage["Fully_Seasoned_%_of_Matched"] = np.where(
    coverage["Matched_Performance_Loans"] > 0,
    coverage["Fully_Seasoned_Loans"]
    / coverage["Matched_Performance_Loans"]
    * 100,
    np.nan
)

display(
    coverage.round(2)
)

print("\n" + "=" * 80)
print("MODEL LOAN ID RANGE")
print("=" * 80)

print(
    "Model unique LoanSequenceNumber:",
    model_df["LoanSequenceNumber"].nunique()
)

print(
    "Model rows:",
    len(model_df)
)

print("\n" + "=" * 80)
print("PERFORMANCE LOAN ID RANGE")
print("=" * 80)

# Reconstruct IDs from the raw files already read
perf_ids = set(
    pd.concat(
        [
            pd.read_csv(
                f,
                sep="|",
                header=None,
                usecols=[0],
                names=["LoanSequenceNumber"],
                dtype={"LoanSequenceNumber": "string"}
            )
            for f in perf_files
        ],
        ignore_index=True
    )["LoanSequenceNumber"]
    .dropna()
    .str.strip()
)

model_ids = set(
    model_df["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

print(
    "Unique performance Loan IDs:",
    f"{len(perf_ids):,}"
)

print(
    "Unique model Loan IDs:",
    f"{len(model_ids):,}"
)

print(
    "Model IDs found in performance:",
    f"{len(model_ids.intersection(perf_ids)):,}"
)

print(
    "Model IDs NOT found in performance:",
    f"{len(model_ids - perf_ids):,}"
)

print("\n" + "=" * 80)
print("MATCHED MODEL LOANS BY VINTAGE")
print("=" * 80)

matched_model = model_df[
    model_df["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
    .isin(perf_ids)
].copy()

print(
    matched_model["Vintage"]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 80)
print("UNMATCHED MODEL LOANS BY VINTAGE")
print("=" * 80)

unmatched_model = model_df[
    ~model_df["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
    .isin(perf_ids)
].copy()

print(
    unmatched_model["Vintage"]
    .value_counts()
    .sort_index()
)


In [ ]:

import os
import glob

print("=" * 80)
print("STEP 13 — SEARCHING FOR PERFORMANCE FILES")
print("=" * 80)

search_paths = [
    r"./data/raw",
    r"./data/raw/*"
]

patterns = [
    "*perf*",
    "*Performance*",
    "*.txt",
    "*.zip",
    "*.gz"
]

found_files = set()

for base in search_paths:

    for pattern in patterns:

        search_pattern = os.path.join(base, pattern)

        for f in glob.glob(search_pattern):

            if os.path.isfile(f):

                name = os.path.basename(f).lower()

                # Keep files that plausibly relate to Freddie Mac performance
                if (
                    "perf" in name
                    or "performance" in name
                    or "sample" in name
                ):
                    found_files.add(os.path.abspath(f))

print("\nPotential performance-related files found:")
print("-" * 80)

for f in sorted(found_files):
    print(f)

print("\nTotal files found:", len(found_files))

print("\n" + "=" * 80)
print("FILES BY NAME")
print("=" * 80)

for f in sorted(found_files):

    try:
        size_mb = os.path.getsize(f) / (1024 ** 2)
    except:
        size_mb = np.nan

    print(
        f"{os.path.basename(f):60s} "
        f"{size_mb:10.2f} MB"
    )


In [ ]:

import os
import glob
import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 14 — COMPLETE PERFORMANCE COVERAGE + 36-MONTH SEASONING AUDIT")
print("=" * 80)

# 1. FIND ALL FIVE PERFORMANCE FILES

perf_files = []

for year in range(2018, 2023):
    patterns = [
        rf"./data/raw/sample_perf_{year}.txt",
        rf"./data/raw/sample_{year}/sample_perf_{year}.txt",
        rf"./data/raw/freddie_{year}_extracted/sample_perf_{year}.txt"
    ]

    found = None

    for p in patterns:
        if os.path.exists(p):
            found = p
            break

    if found is not None:
        perf_files.append(found)

print("\nPerformance files found:")
for f in perf_files:
    print(" -", f)

print("\nTotal performance files:", len(perf_files))

if len(perf_files) != 5:
    print(
        "\nWARNING: Expected five files for 2018–2022. "
        "Check the paths above before continuing."
    )

# 2. VERIFY MODEL DATAFRAME CONTAINS LOAN ID

print("\n" + "=" * 80)
print("MODEL LOAN IDENTIFIER CHECK")
print("=" * 80)

print("model_df shape:", model_df.shape)
print("model_df columns:")
print(list(model_df.columns))

if "LoanSequenceNumber" not in model_df.columns:
    raise KeyError(
        "LoanSequenceNumber is not in model_df. "
        "We need the dataframe that contains the loan ID."
    )

model_loans = model_df[
    ["LoanSequenceNumber", "Vintage"]
].copy()

model_loans["LoanSequenceNumber"] = (
    model_loans["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

model_loans["Vintage"] = pd.to_numeric(
    model_loans["Vintage"],
    errors="coerce"
).astype("Int64")

model_loans = model_loans.drop_duplicates(
    subset=["LoanSequenceNumber"]
).copy()

print("\nUnique model loans:", len(model_loans))

print("\nModel loans by vintage:")
print(model_loans["Vintage"].value_counts().sort_index())

# 3. READ ONLY THE TWO PERFORMANCE COLUMNS WE NEED
#
# Freddie Mac performance file:
#   column 0 = Loan Sequence Number
#   column 1 = Monthly Reporting Period
#
# Reading only these two columns prevents the previous MemoryError.

performance_max = {}

print("\n" + "=" * 80)
print("READING PERFORMANCE FILES")
print("=" * 80)

for f in perf_files:

    year = os.path.basename(f).split("_")[-1].replace(".txt", "")

    print("\nReading:", os.path.basename(f))

    chunks = []

    for chunk in pd.read_csv(
        f,
        sep="|",
        header=None,
        usecols=[0, 1],
        names=[
            "LoanSequenceNumber",
            "MonthlyReportingPeriod"
        ],
        dtype={
            "LoanSequenceNumber": "string",
            "MonthlyReportingPeriod": "string"
        },
        chunksize=250_000,
        low_memory=True
    ):

        chunk["LoanSequenceNumber"] = (
            chunk["LoanSequenceNumber"]
            .str.strip()
        )

        # Keep only valid reporting periods
        chunk["MonthlyReportingPeriod"] = pd.to_numeric(
            chunk["MonthlyReportingPeriod"],
            errors="coerce"
        )

        chunk = chunk.dropna(
            subset=[
                "LoanSequenceNumber",
                "MonthlyReportingPeriod"
            ]
        )

        # Only keep maximum reporting period for each loan
        chunk_max = (
            chunk.groupby("LoanSequenceNumber")[
                "MonthlyReportingPeriod"
            ]
            .max()
        )

        chunks.append(chunk_max)

    # Combine chunk-level maxima
    file_max = pd.concat(chunks)

    file_max = (
        file_max.groupby(level=0)
        .max()
    )

    performance_max[year] = file_max

    print(
        "Unique loans in file:",
        len(file_max)
    )

    print(
        "Latest reporting period:",
        int(file_max.max())
    )

    # Free memory
    del chunks
    del file_max

# 4. COMBINE ALL PERFORMANCE LOAN IDs

print("\n" + "=" * 80)
print("COMBINING PERFORMANCE COVERAGE")
print("=" * 80)

all_perf_max = pd.concat(
    performance_max.values()
)

all_perf_max = (
    all_perf_max.groupby(level=0)
    .max()
)

all_perf_max.name = "LastReportingYYYYMM"

print(
    "Unique performance loans across all files:",
    len(all_perf_max)
)

print(
    "Global latest reporting period:",
    int(all_perf_max.max())
)

# 5. MATCH MODEL LOANS TO PERFORMANCE

model_loans["LastReportingYYYYMM"] = (
    model_loans["LoanSequenceNumber"]
    .map(all_perf_max)
)

model_loans["PerformanceMatch"] = (
    model_loans["LastReportingYYYYMM"]
    .notna()
)

# 6. CALCULATE 36-MONTH ELIGIBILITY
#
# IMPORTANT:
# We only know Vintage = year in the processed model.
#
# Therefore use the conservative year-level test:
#
#
# This is conservative because a loan originated earlier than December
# would actually require less time.

required_cutoff = {
    2018: 202112,
    2019: 202212,
    2020: 202312,
    2021: 202412,
    2022: 202512
}

model_loans["Required36M_YYYYMM"] = (
    model_loans["Vintage"]
    .map(required_cutoff)
)

model_loans["FullySeasoned"] = (
    model_loans["PerformanceMatch"]
    &
    (
        model_loans["LastReportingYYYYMM"]
        >= model_loans["Required36M_YYYYMM"]
    )
)

# 7. OVERALL SEASONING SUMMARY

print("\n" + "=" * 80)
print("36-MONTH SEASONING RESULTS")
print("=" * 80)

print(
    "\nFully seasoned:",
    int(model_loans["FullySeasoned"].sum())
)

print(
    "Not fully seasoned:",
    int(
        (
            model_loans["PerformanceMatch"]
            &
            ~model_loans["FullySeasoned"]
        ).sum()
    )
)

print(
    "Missing performance history:",
    int(
        (~model_loans["PerformanceMatch"]).sum()
    )
)

# 8. VINTAGE-LEVEL COVERAGE

vintage_summary = (
    model_loans
    .groupby("Vintage")
    .agg(
        Model_Loans=("LoanSequenceNumber", "size"),
        Performance_Matched=("PerformanceMatch", "sum"),
        Fully_Seasoned=("FullySeasoned", "sum")
    )
    .reset_index()
)

vintage_summary["Not_Fully_Seasoned"] = (
    vintage_summary["Performance_Matched"]
    - vintage_summary["Fully_Seasoned"]
)

vintage_summary["Performance_Match_%"] = (
    vintage_summary["Performance_Matched"]
    /
    vintage_summary["Model_Loans"]
    * 100
)

vintage_summary["Fully_Seasoned_%_of_Matched"] = np.where(
    vintage_summary["Performance_Matched"] > 0,
    vintage_summary["Fully_Seasoned"]
    /
    vintage_summary["Performance_Matched"]
    * 100,
    np.nan
)

print("\n" + "=" * 80)
print("VINTAGE-LEVEL SEASONING SUMMARY")
print("=" * 80)

display(
    vintage_summary.round(2)
)

# 9. 2021 / 2022 FOCUSED CHECK

print("\n" + "=" * 80)
print("2021–2022 SEASONING CHECK")
print("=" * 80)

for year in [2021, 2022]:

    temp = model_loans[
        model_loans["Vintage"] == year
    ]

    total = len(temp)
    matched = temp["PerformanceMatch"].sum()
    seasoned = temp["FullySeasoned"].sum()

    print(f"\n{year} VINTAGE")
    print("-" * 40)
    print("Model loans:", total)
    print("Performance matched:", matched)
    print("Fully seasoned:", seasoned)

    if matched > 0:
        print(
            "Fully seasoned among matched: "
            f"{seasoned / matched * 100:.2f}%"
        )
    else:
        print(
            "Fully seasoned among matched: N/A"
        )

# 10. PERFORMANCE MATCH BY VINTAGE

print("\n" + "=" * 80)
print("MATCHED / UNMATCHED LOANS BY VINTAGE")
print("=" * 80)

matched_by_vintage = (
    model_loans[
        model_loans["PerformanceMatch"]
    ]
    .groupby("Vintage")
    .size()
    .rename("Matched")
)

unmatched_by_vintage = (
    model_loans[
        ~model_loans["PerformanceMatch"]
    ]
    .groupby("Vintage")
    .size()
    .rename("Unmatched")
)

coverage_table = pd.concat(
    [
        matched_by_vintage,
        unmatched_by_vintage
    ],
    axis=1
).fillna(0).astype(int)

display(coverage_table)

# 11. FINAL DIAGNOSIS

print("\n" + "=" * 80)
print("FINAL SEASONING DIAGNOSIS")
print("=" * 80)

for year in [2018, 2019, 2020, 2021, 2022]:

    temp = model_loans[
        model_loans["Vintage"] == year
    ]

    total = len(temp)
    matched = int(temp["PerformanceMatch"].sum())
    seasoned = int(temp["FullySeasoned"].sum())

    if matched > 0:
        pct = seasoned / matched * 100
    else:
        pct = np.nan

    print(
        f"{year}: "
        f"{seasoned:,}/{matched:,} matched loans fully seasoned "
        f"({pct:.2f}%)"
        if not np.isnan(pct)
        else
        f"{year}: {seasoned:,}/{matched:,} "
        f"(N/A — no performance matches)"
    )

# 12. INTERPRETATION

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)

summary_2022 = vintage_summary[
    vintage_summary["Vintage"] == 2022
]

if len(summary_2022) == 1:

    matched_2022 = int(
        summary_2022["Performance_Matched"].iloc[0]
    )

    seasoned_2022 = int(
        summary_2022["Fully_Seasoned"].iloc[0]
    )

    if matched_2022 == 0:

        print(
            "⚠ No 2022 model loans are currently matched to "
            "the performance files."
        )

        print(
            "The 2022 calibration result cannot yet be interpreted "
            "as a seasoning result."
        )

    elif seasoned_2022 / matched_2022 >= 0.99:

        print(
            "✓ The matched 2022 cohort has a complete conservative "
            "36-month observation window."
        )

        print(
            "Therefore, the 2022 calibration problem is NOT simply "
            "explained by insufficient seasoning."
        )

    else:

        print(
            "⚠ A meaningful share of matched 2022 loans is not "
            "fully seasoned."
        )

        print(
            "The 2022 calibration result may be affected by "
            "right-censoring."
        )

print("\nDone.")


In [ ]:

import pandas as pd
import os

orig_file = r"./data/raw/sample_2022/sample_orig_2022.txt"

print("=" * 80)
print("STEP 19A — ORIGINATION FILE INSPECTION")
print("=" * 80)

# Read ONLY the first 5 rows so this is memory-safe
orig_sample = pd.read_csv(
    orig_file,
    sep="|",
    header=None,
    nrows=5,
    dtype="string"
)

print("\nFile:")
print(orig_file)

print("\nShape of sample:")
print(orig_sample.shape)

print("\nRaw columns:")
print(list(orig_sample.columns))

print("\nFirst 5 rows:")
display(orig_sample)

print("\nMODEL_DF COLUMNS:")
print(list(model_df.columns))

print("\nFirst 5 model loan IDs:")
display(
    model_df[["LoanSequenceNumber", "Vintage"]].head()
)


In [ ]:

print("=" * 90)
print("STEP 19B — TRACING LoanSequenceNumber ORIGIN")
print("=" * 90)

# 1. Inspect model_df ID structure

print("\nMODEL_DF ID SUMMARY")
print("-" * 60)

print("Rows:", len(model_df))
print("Unique LoanSequenceNumber:",
      model_df["LoanSequenceNumber"].nunique())

print("\nFirst 20 LoanSequenceNumber values:")
print(
    model_df[
        ["LoanSequenceNumber", "Vintage"]
    ].head(20).to_string(index=False)
)

print("\nLoanSequenceNumber by vintage:")
print(
    model_df.groupby("Vintage")["LoanSequenceNumber"]
    .nunique()
)

# 2. Check whether IDs look like genuine Freddie Mac IDs

print("\nID FORMAT CHECK")
print("-" * 60)

sample_ids = (
    model_df["LoanSequenceNumber"]
    .dropna()
    .astype(str)
    .head(20)
    .tolist()
)

for x in sample_ids:
    print(x)

# 3. Inspect ALL current DataFrames for the same IDs

print("\nDATAFRAME ID TRACE")
print("-" * 60)

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        if "LoanSequenceNumber" in obj.columns:

            try:
                n_unique = obj["LoanSequenceNumber"].nunique()
                n_rows = len(obj)

                overlap = 0

                if name != "model_df":
                    overlap = len(
                        set(
                            model_df["LoanSequenceNumber"]
                            .dropna()
                            .astype(str)
                        )
                        &
                        set(
                            obj["LoanSequenceNumber"]
                            .dropna()
                            .astype(str)
                        )
                    )

                print(
                    f"{name:30s} "
                    f"rows={n_rows:10,d} "
                    f"unique_ids={n_unique:10,d} "
                    f"overlap_with_model={overlap:10,d}"
                )

            except Exception:
                pass

# 4. Specifically inspect known datasets

print("\nKNOWN DATASET CHECK")
print("-" * 60)

for name in [
    "model_df",
    "oot_train",
    "oot_test",
    "vintage_data",
    "X_oot_train",
    "X_oot_test",
    "el_df",
    "df_perf",
    "performance"
]:

    if name in globals():

        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):

            print(f"\n{name}")
            print("Shape:", obj.shape)
            print("Columns containing 'Loan':")

            print([
                c for c in obj.columns
                if "loan" in str(c).lower()
            ])

# 5. Check whether model IDs overlap directly with performance IDs

if "performance" in globals():

    perf_ids = set(
        performance["LoanSequenceNumber"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    model_ids = set(
        model_df["LoanSequenceNumber"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    overlap = model_ids & perf_ids

    print("\nMODEL ↔ PERFORMANCE ID OVERLAP")
    print("-" * 60)

    print("Model unique IDs:", len(model_ids))
    print("Performance unique IDs:", len(perf_ids))
    print("Overlap:", len(overlap))
    print(
        "Model IDs found in performance:",
        f"{len(overlap) / len(model_ids) * 100:.2f}%"
        if len(model_ids) > 0 else "NA"
    )

# 6. Check ID-vintage consistency

print("\nID PREFIX / VINTAGE CHECK")
print("-" * 60)

tmp_id = model_df[
    ["LoanSequenceNumber", "Vintage"]
].dropna().copy()

tmp_id["LoanSequenceNumber"] = (
    tmp_id["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

tmp_id["ID_PREFIX"] = (
    tmp_id["LoanSequenceNumber"]
    .str.extract(r"^(F\d{2})", expand=False)
)

print(
    tmp_id.groupby(
        ["Vintage", "ID_PREFIX"],
        dropna=False
    ).size()
    .reset_index(name="Loans")
    .to_string(index=False)
)

print("\n" + "=" * 90)
print("STEP 19B COMPLETE")
print("=" * 90)


In [ ]:

import os
import glob
import pandas as pd
import numpy as np

print("=" * 90)
print("STEP 20A — LOAN-LEVEL SEASONING USING FREDDIE MAC LOAN AGE")
print("=" * 90)

# 1. CORRECT PERFORMANCE FILES ONLY

perf_files = [
    r"./data/raw/sample_2018/sample_perf_2018.txt",
    r"./data/raw/sample_2019/sample_perf_2019.txt",
    r"./data/raw/sample_2020/sample_perf_2020.txt",
    r"./data/raw/sample_2021/sample_perf_2021.txt",
    r"./data/raw/sample_2022/sample_perf_2022.txt",
]

perf_files = [f for f in perf_files if os.path.exists(f)]

print("\nPerformance files:")
for f in perf_files:
    print(" -", f)

print("\nTotal:", len(perf_files))

if len(perf_files) != 5:
    raise ValueError(
        "Expected exactly 5 performance files (2018–2022)."
    )

model_ids = (
    model_df[
        [
            "LoanSequenceNumber",
            "Vintage",
            "Default_36M"
        ]
    ]
    .copy()
)

model_ids["LoanSequenceNumber"] = (
    model_ids["LoanSequenceNumber"]
    .astype("string")
    .str.strip()
)

model_ids["Vintage"] = pd.to_numeric(
    model_ids["Vintage"],
    errors="coerce"
).astype("Int64")

print("\nModel loans:", len(model_ids))
print(
    model_ids["Vintage"]
    .value_counts()
    .sort_index()
)

# 3. MEMORY-SAFE LOAN-LEVEL PERFORMANCE SUMMARY
#
# We only need:
#   LoanSequenceNumber
#   MonthlyReportingPeriod
#
# No need to load the full 35-column performance file.

loan_last_period = {}
loan_first_period = {}
loan_obs_count = {}

model_id_set = set(
    model_ids["LoanSequenceNumber"].dropna().tolist()
)

for f in perf_files:

    print("\nProcessing:", os.path.basename(f))

    file_last = None
    file_rows = 0
    file_matched = 0

    # Read in chunks to avoid memory errors
    reader = pd.read_csv(
        f,
        sep="|",
        header=None,
        usecols=[0, 1],
        names=[
            "LoanSequenceNumber",
            "MonthlyReportingPeriod"
        ],
        dtype="string",
        chunksize=200_000,
        low_memory=True,
        on_bad_lines="skip"
    )

    for chunk in reader:

        file_rows += len(chunk)

        chunk["LoanSequenceNumber"] = (
            chunk["LoanSequenceNumber"]
            .str.strip()
        )

        chunk["MonthlyReportingPeriod"] = (
            chunk["MonthlyReportingPeriod"]
            .str.strip()
        )

        # Keep only model loans
        chunk = chunk[
            chunk["LoanSequenceNumber"].isin(model_id_set)
        ]

        if len(chunk) == 0:
            continue

        file_matched += len(chunk)

        # Convert YYYYMM to integer for efficient comparison
        periods = pd.to_numeric(
            chunk["MonthlyReportingPeriod"],
            errors="coerce"
        )

        chunk = chunk.loc[
            periods.notna()
        ].copy()

        if len(chunk) == 0:
            continue

        chunk["PeriodInt"] = periods.loc[chunk.index].astype("int32")

        # Aggregate by loan within this chunk
        grouped = (
            chunk
            .groupby("LoanSequenceNumber")["PeriodInt"]
            .agg(["min", "max", "count"])
        )

        for loan_id, row in grouped.iterrows():

            if loan_id not in loan_first_period:
                loan_first_period[loan_id] = int(row["min"])
                loan_last_period[loan_id] = int(row["max"])
                loan_obs_count[loan_id] = int(row["count"])

            else:

                loan_first_period[loan_id] = min(
                    loan_first_period[loan_id],
                    int(row["min"])
                )

                loan_last_period[loan_id] = max(
                    loan_last_period[loan_id],
                    int(row["max"])
                )

                loan_obs_count[loan_id] += int(row["count"])

    print("Rows processed:", file_rows)
    print("Matched performance rows:", file_matched)

# 4. BUILD PERFORMANCE SUMMARY

performance_summary = pd.DataFrame({
    "LoanSequenceNumber": list(loan_last_period.keys()),
    "FirstReportingPeriod": [
        loan_first_period[x]
        for x in loan_last_period.keys()
    ],
    "LastReportingPeriod": [
        loan_last_period[x]
        for x in loan_last_period.keys()
    ],
    "PerformanceRows": [
        loan_obs_count[x]
        for x in loan_last_period.keys()
    ]
})

performance_summary["LoanSequenceNumber"] = (
    performance_summary["LoanSequenceNumber"]
    .astype("string")
)

print("\nPerformance loans matched:", len(performance_summary))

# 5. MERGE WITH MODEL

seasoning_check = model_ids.merge(
    performance_summary,
    on="LoanSequenceNumber",
    how="left"
)

print(
    "Model loans with performance:",
    seasoning_check["LastReportingPeriod"].notna().sum()
)

print(
    "Model loans without performance:",
    seasoning_check["LastReportingPeriod"].isna().sum()
)

# 6. CONVERT REPORTING PERIODS TO DATES

seasoning_check["FirstReportingDate"] = pd.to_datetime(
    seasoning_check["FirstReportingPeriod"].astype("string"),
    format="%Y%m",
    errors="coerce"
)

seasoning_check["LastReportingDate"] = pd.to_datetime(
    seasoning_check["LastReportingPeriod"].astype("string"),
    format="%Y%m",
    errors="coerce"
)

# 7. VINTAGE-BASED 36-MONTH REQUIREMENT
#
# Conservative test:
#
# This is conservative because it treats the entire vintage as requiring
# December of vintage + 3 years.

required_period_map = {
    2018: 202112,
    2019: 202212,
    2020: 202312,
    2021: 202412,
    2022: 202512,
}

seasoning_check["Required36MPeriod"] = (
    seasoning_check["Vintage"]
    .map(required_period_map)
)

# 8. FULLY SEASONED FLAG

seasoning_check["FullySeasoned36M"] = (
    seasoning_check["LastReportingPeriod"]
    >= seasoning_check["Required36MPeriod"]
)

seasoning_check["FullySeasoned36M"] = (
    seasoning_check["FullySeasoned36M"]
    .fillna(False)
)

# 9. SUMMARY BY VINTAGE

seasoning_summary = (
    seasoning_check
    .groupby("Vintage")
    .agg(
        Model_Loans=("LoanSequenceNumber", "size"),
        Performance_Matched=(
            "LastReportingPeriod",
            lambda x: x.notna().sum()
        ),
        Fully_Seasoned=(
            "FullySeasoned36M",
            "sum"
        )
    )
    .reset_index()
)

seasoning_summary["Not_Fully_Seasoned"] = (
    seasoning_summary["Performance_Matched"]
    - seasoning_summary["Fully_Seasoned"]
)

seasoning_summary["Performance_Match_%"] = (
    seasoning_summary["Performance_Matched"]
    / seasoning_summary["Model_Loans"]
    * 100
)

seasoning_summary["Fully_Seasoned_%_of_Matched"] = np.where(
    seasoning_summary["Performance_Matched"] > 0,
    seasoning_summary["Fully_Seasoned"]
    / seasoning_summary["Performance_Matched"]
    * 100,
    np.nan
)

print("\n" + "=" * 90)
print("36-MONTH SEASONING SUMMARY")
print("=" * 90)

print(
    seasoning_summary.to_string(index=False)
)

# 10. SPECIFIC 2021 / 2022 CHECK

print("\n" + "=" * 90)
print("2021–2022 SEASONING CHECK")
print("=" * 90)

for vintage in [2021, 2022]:

    s = seasoning_check[
        seasoning_check["Vintage"] == vintage
    ]

    total = len(s)
    matched = s["LastReportingPeriod"].notna().sum()
    seasoned = s["FullySeasoned36M"].sum()

    print(f"\n{vintage} VINTAGE")
    print("-" * 50)
    print(f"Model loans              : {total:,}")
    print(f"Performance matched      : {matched:,}")
    print(f"Fully seasoned           : {seasoned:,}")

    if matched > 0:
        print(
            f"Fully seasoned %        : "
            f"{seasoned / matched * 100:.2f}%"
        )
    else:
        print("Fully seasoned %        : NA")

# 11. CHECK THE ACTUAL LAST REPORTING PERIOD DISTRIBUTION

print("\n" + "=" * 90)
print("LAST REPORTING PERIOD DISTRIBUTION — 2022")
print("=" * 90)

s2022 = seasoning_check[
    seasoning_check["Vintage"] == 2022
].copy()

print(
    s2022["LastReportingPeriod"]
    .value_counts(dropna=False)
    .sort_index()
    .tail(20)
)

# 12. CRITICAL DIAGNOSTIC

print("\n" + "=" * 90)
print("CRITICAL DIAGNOSTIC")
print("=" * 90)

for vintage in [2018, 2019, 2020, 2021, 2022]:

    s = seasoning_check[
        seasoning_check["Vintage"] == vintage
    ]

    total = len(s)
    seasoned = s["FullySeasoned36M"].sum()

    pct = (
        seasoned / total * 100
        if total > 0
        else np.nan
    )

    print(
        f"{vintage}: "
        f"{seasoned:,}/{total:,} "
        f"fully seasoned "
        f"({pct:.2f}%)"
    )

print("\n" + "=" * 90)
print("STEP 20A COMPLETE")
print("=" * 90)


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    roc_curve
)

print("=" * 90)
print("STEP 21 — 2022 ALL VS FULLY-SEASONED PERFORMANCE COMPARISON")
print("=" * 90)

# 1. CHECK REQUIRED OBJECTS

required_objects = [
    "seasoning_check",
    "final_model",
    "sql_features_final"
]

missing_objects = [
    x for x in required_objects
    if x not in globals()
]

if missing_objects:
    raise ValueError(
        f"Missing required objects: {missing_objects}"
    )

print("\n✓ Required objects found.")

# 2. GET 2022 MODEL DATA

oot_2022 = model_df[
    model_df["Vintage"] == 2022
].copy()

print("\n2022 model loans:", len(oot_2022))

# 3. MAKE SURE ENGINEERED FLAGS EXIST

flag_rules = {
    "LowCreditFlag": lambda d: (d["CreditScore"] < 680).astype(int),

    "HighLTVFlag": lambda d: (d["OriginalLTV"] > 80).astype(int),

    "HighDTIFlag": lambda d: (d["OriginalDTI"] > 43).astype(int),

    "HighCLTVFlag": lambda d: (d["OriginalCLTV"] > 80).astype(int),

    "HighRateFlag": lambda d: (
        d["OriginalInterestRate"] > d["OriginalInterestRate"].median()
    ).astype(int),

    "HighCreditLTVFlag": lambda d: (
        (d["CreditScore"] < 680) &
        (d["OriginalLTV"] > 80)
    ).astype(int),

    "HighDTILTVFlag": lambda d: (
        (d["OriginalDTI"] > 43) &
        (d["OriginalLTV"] > 80)
    ).astype(int)
}

for flag, rule in flag_rules.items():

    if flag not in oot_2022.columns:
        oot_2022[flag] = rule(oot_2022)

# 4. VERIFY ALL MODEL FEATURES

missing_features = [
    c for c in sql_features_final
    if c not in oot_2022.columns
]

print("\nMissing model features:")
print(missing_features)

if missing_features:
    raise ValueError(
        f"2022 data is missing model features: {missing_features}"
    )

print(
    f"✓ All {len(sql_features_final)} model features are present."
)

# 5. GENERATE PREDICTED PD FOR ALL 2022

print("\nGenerating predictions for all 2022 loans...")

oot_2022["PD"] = final_model.predict_proba(
    oot_2022[sql_features_final]
)[:, 1]

print("✓ Predictions generated.")

# 6. MERGE SEASONING STATUS

seasoning_status = seasoning_check[
    [
        "LoanSequenceNumber",
        "FullySeasoned36M",
        "LastReportingPeriod",
        "Required36MPeriod"
    ]
].copy()

seasoning_status["LoanSequenceNumber"] = (
    seasoning_status["LoanSequenceNumber"]
    .astype("string")
)

oot_2022["LoanSequenceNumber"] = (
    oot_2022["LoanSequenceNumber"]
    .astype("string")
)

oot_2022 = oot_2022.merge(
    seasoning_status,
    on="LoanSequenceNumber",
    how="left"
)

print(
    "\nSeasoning status matched:",
    oot_2022["FullySeasoned36M"].notna().sum()
)

# 7. DEFINE METRIC FUNCTION

def calculate_metrics(df, label):

    y = pd.to_numeric(
        df["Default_36M"],
        errors="coerce"
    )

    pd_pred = pd.to_numeric(
        df["PD"],
        errors="coerce"
    )

    valid = (
        y.notna() &
        pd_pred.notna()
    )

    y = y.loc[valid].astype(int)
    pd_pred = pd_pred.loc[valid]

    n = len(y)
    defaults = int(y.sum())

    if n == 0:
        return {
            "Sample": label,
            "Loans": 0,
            "Defaults": np.nan,
            "Default_Rate_%": np.nan,
            "Average_PD_%": np.nan,
            "Calibration_Gap_pp": np.nan,
            "ROC_AUC": np.nan,
            "PR_AUC": np.nan,
            "KS": np.nan,
            "Brier": np.nan,
            "Top_Decile_Default_%": np.nan,
            "Top_Decile_Lift": np.nan
        }

    default_rate = y.mean()
    avg_pd = pd_pred.mean()

    calibration_gap = (
        avg_pd - default_rate
    ) * 100

    # ROC AUC

    if y.nunique() == 2:
        auc = roc_auc_score(y, pd_pred)
    else:
        auc = np.nan

    # PR AUC

    if y.nunique() == 2:
        pr_auc = average_precision_score(y, pd_pred)
    else:
        pr_auc = np.nan

    # KS

    if y.nunique() == 2:

        fpr, tpr, thresholds = roc_curve(
            y,
            pd_pred
        )

        ks = np.max(
            np.abs(tpr - fpr)
        )

    else:
        ks = np.nan

    # BRIER

    brier = brier_score_loss(
        y,
        pd_pred
    )

    # TOP DECILE

    cutoff = np.percentile(
        pd_pred,
        90
    )

    top_decile = pd_pred >= cutoff

    if top_decile.sum() > 0:

        top_default_rate = (
            y.loc[top_decile].mean()
        )

        top_lift = (
            top_default_rate / default_rate
            if default_rate > 0
            else np.nan
        )

    else:

        top_default_rate = np.nan
        top_lift = np.nan

    return {
        "Sample": label,
        "Loans": n,
        "Defaults": defaults,
        "Default_Rate_%": default_rate * 100,
        "Average_PD_%": avg_pd * 100,
        "Calibration_Gap_pp": calibration_gap,
        "ROC_AUC": auc,
        "PR_AUC": pr_auc,
        "KS": ks,
        "Brier": brier,
        "Top_Decile_Default_%": top_default_rate * 100,
        "Top_Decile_Lift": top_lift
    }

# 8. ALL 2022

all_2022 = oot_2022.copy()

# 9. FULLY SEASONED 2022

seasoned_2022 = oot_2022[
    oot_2022["FullySeasoned36M"] == True
].copy()

print("\n" + "=" * 90)
print("SAMPLE COUNTS")
print("=" * 90)

print(
    f"All 2022 loans       : {len(all_2022):,}"
)

print(
    f"Fully seasoned 2022  : {len(seasoned_2022):,}"
)

print(
    f"Not fully seasoned    : "
    f"{len(all_2022) - len(seasoned_2022):,}"
)

print(
    f"Seasoned percentage   : "
    f"{len(seasoned_2022) / len(all_2022) * 100:.2f}%"
)

# 10. CALCULATE BOTH SETS OF METRICS

results = []

results.append(
    calculate_metrics(
        all_2022,
        "All 2022"
    )
)

results.append(
    calculate_metrics(
        seasoned_2022,
        "Fully Seasoned 2022"
    )
)

comparison = pd.DataFrame(results)

# 11. DISPLAY RESULTS

print("\n" + "=" * 90)
print("2022 PERFORMANCE — ALL VS FULLY SEASONED")
print("=" * 90)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# 12. CALIBRATION GAP CHANGE

all_gap = comparison.loc[
    comparison["Sample"] == "All 2022",
    "Calibration_Gap_pp"
].iloc[0]

seasoned_gap = comparison.loc[
    comparison["Sample"] == "Fully Seasoned 2022",
    "Calibration_Gap_pp"
].iloc[0]

print("\n" + "=" * 90)
print("CALIBRATION DIAGNOSTIC")
print("=" * 90)

print(
    f"All 2022 calibration gap       : "
    f"{all_gap:.2f} pp"
)

print(
    f"Seasoned 2022 calibration gap  : "
    f"{seasoned_gap:.2f} pp"
)

if pd.notna(all_gap) and pd.notna(seasoned_gap):

    gap_change = seasoned_gap - all_gap

    print(
        f"Change after seasoning         : "
        f"{gap_change:+.2f} pp"
    )

    if abs(seasoned_gap) < abs(all_gap) - 1:

        print(
            "\n→ Calibration improves materially after "
            "removing incompletely seasoned loans."
        )

        print(
            "→ Right-censoring appears to contribute "
            "to the original calibration problem."
        )

    elif abs(seasoned_gap - all_gap) <= 1:

        print(
            "\n→ Calibration gap is largely unchanged "
            "after seasoning correction."
        )

        print(
            "→ Right-censoring does NOT appear to explain "
            "the main calibration deterioration."
        )

        print(
            "→ Population/regime shift becomes a substantially "
            "stronger explanation."
        )

    else:

        print(
            "\n→ Calibration gap becomes larger after "
            "seasoning correction."
        )

        print(
            "→ The original gap was not driven simply by "
            "right-censoring."
        )

# 13. DISCRIMINATION CHANGE

all_auc = comparison.loc[
    comparison["Sample"] == "All 2022",
    "ROC_AUC"
].iloc[0]

seasoned_auc = comparison.loc[
    comparison["Sample"] == "Fully Seasoned 2022",
    "ROC_AUC"
].iloc[0]

all_lift = comparison.loc[
    comparison["Sample"] == "All 2022",
    "Top_Decile_Lift"
].iloc[0]

seasoned_lift = comparison.loc[
    comparison["Sample"] == "Fully Seasoned 2022",
    "Top_Decile_Lift"
].iloc[0]

print("\n" + "=" * 90)
print("RANKING ROBUSTNESS DIAGNOSTIC")
print("=" * 90)

print(
    f"All 2022 ROC-AUC      : {all_auc:.4f}"
)

print(
    f"Seasoned 2022 ROC-AUC : {seasoned_auc:.4f}"
)

print(
    f"All 2022 lift         : {all_lift:.2f}x"
)

print(
    f"Seasoned 2022 lift    : {seasoned_lift:.2f}x"
)

# 14. DEFAULT RATE COMPARISON

all_default = comparison.loc[
    comparison["Sample"] == "All 2022",
    "Default_Rate_%"
].iloc[0]

seasoned_default = comparison.loc[
    comparison["Sample"] == "Fully Seasoned 2022",
    "Default_Rate_%"
].iloc[0]

print("\n" + "=" * 90)
print("REALIZED DEFAULT RATE")
print("=" * 90)

print(
    f"All 2022 default rate       : "
    f"{all_default:.2f}%"
)

print(
    f"Seasoned 2022 default rate  : "
    f"{seasoned_default:.2f}%"
)

print(
    f"Difference                   : "
    f"{seasoned_default - all_default:+.2f} pp"
)

# 15. FINAL DIAGNOSIS

print("\n" + "=" * 90)
print("STEP 21 — FINAL DIAGNOSTIC INTERPRETATION")
print("=" * 90)

if pd.notna(seasoned_gap):

    if abs(seasoned_gap) <= 2:

        print(
            "✓ Seasoning correction substantially improves PD calibration."
        )

        print(
            "The original 2022 calibration deterioration was materially "
            "affected by right-censoring."
        )

    elif abs(seasoned_gap) > 5:

        print(
            "⚠ A large calibration gap remains after restricting to "
            "fully seasoned 2022 loans."
        )

        print(
            "This strongly suggests that insufficient seasoning alone "
            "does NOT explain the 2022 calibration deterioration."
        )

        print(
            "Population shift / regime shift / model drift should now "
            "be investigated as the primary explanation."
        )

    else:

        print(
            "→ Some calibration deterioration remains after seasoning "
            "correction."
        )

        print(
            "Both right-censoring and population/regime shift may "
            "contribute."
        )

# 16. SAVE RESULTS IN MEMORY

comparison_2022 = comparison.copy()

seasoned_2022_final = seasoned_2022.copy()
all_2022_final = all_2022.copy()

print("\n" + "=" * 90)
print("STEP 21 COMPLETE")
print("=" * 90)


In [ ]:
# Purpose:
#   1. Confirm that the 2022 calibration deterioration survives seasoning.
#   2. Compare model behaviour across vintages.
#   3. Check whether the deterioration is mainly calibration or ranking.
#   4. Examine whether 2022 borrower characteristics shifted.
#   5. Produce clean tables suitable for the final project/report.

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    roc_curve
)

print("=" * 100)
print("STEP 22 — FINAL CREDIT RISK MODEL DRIFT / REGIME-SHIFT DIAGNOSTIC")
print("=" * 100)

# 1. FIND MODEL / FEATURES

print("\n" + "=" * 80)
print("STEP 22A — LOCATING MODEL AND FEATURE SET")
print("=" * 80)

# Try common model object names used earlier in the notebook.
_model_candidates = [
    "model",
    "xgb_model",
    "lgb_model",
    "best_model",
    "clf",
    "final_model"
]

trained_model = None

for _name in _model_candidates:
    if _name in globals():
        _candidate = globals()[_name]
        if hasattr(_candidate, "predict_proba"):
            trained_model = _candidate
            print(f"✓ Model found: {_name}")
            break

if trained_model is None:
    raise ValueError(
        "Could not find the trained model. "
        "Expected one of: model, xgb_model, lgb_model, "
        "best_model, clf, final_model."
    )

# Find feature list
_feature_candidates = [
    "feature_cols",
    "features",
    "model_features",
    "feature_columns",
    "sql_features_final"
]

feature_list = None

for _name in _feature_candidates:
    if _name in globals():
        _candidate = globals()[_name]

        if isinstance(_candidate, (list, tuple, np.ndarray, pd.Index)):
            feature_list = list(_candidate)
            print(f"✓ Feature list found: {_name}")
            break

if feature_list is None:
    raise ValueError(
        "Could not find the model feature list. "
        "Expected feature_cols, features, model_features, "
        "feature_columns, or sql_features_final."
    )

print(f"Number of model features: {len(feature_list)}")

# 2. CHECK 2022 DATA

print("\n" + "=" * 80)
print("STEP 22B — PREPARING 2022 OOT SAMPLE")
print("=" * 80)

if "oot_2022" in globals():
    final_2022 = oot_2022.copy()

elif "oot_test" in globals():
    final_2022 = oot_test.copy()

elif "vintage_data" in globals():
    final_2022 = vintage_data.copy()

else:
    final_2022 = model_df.loc[
        model_df["Vintage"].astype(int) == 2022
    ].copy()

final_2022["Vintage"] = pd.to_numeric(
    final_2022["Vintage"],
    errors="coerce"
).astype("Int64")

final_2022 = final_2022.loc[
    final_2022["Vintage"] == 2022
].copy()

print(f"2022 observations: {len(final_2022):,}")

# 3. VERIFY FEATURES

missing_features = [
    c for c in feature_list
    if c not in final_2022.columns
]

print("\nMissing features:", missing_features)

if missing_features:
    raise ValueError(
        f"2022 dataset is missing required model features: {missing_features}"
    )

print("✓ All model features available.")

# 4. GENERATE 2022 PREDICTIONS

print("\n" + "=" * 80)
print("STEP 22C — GENERATING 2022 PREDICTIONS")
print("=" * 80)

X_2022 = final_2022[feature_list].copy()

# Ensure numeric representation where possible
for col in X_2022.columns:
    if not pd.api.types.is_numeric_dtype(X_2022[col]):
        X_2022[col] = pd.to_numeric(
            X_2022[col],
            errors="coerce"
        )

# Fill missing values consistently
X_2022 = X_2022.replace(
    [np.inf, -np.inf],
    np.nan
)

X_2022 = X_2022.fillna(
    X_2022.median(numeric_only=True)
)

pred_2022 = trained_model.predict_proba(X_2022)[:, 1]

final_2022["PredictedPD"] = pred_2022

print("✓ 2022 predictions generated.")

# 5. IDENTIFY TARGET

target_candidates = [
    "Default_36M",
    "Default",
    "DefaultLabel",
    "DistressLabel"
]

target_col = None

for col in target_candidates:
    if col in final_2022.columns:
        target_col = col
        break

if target_col is None:
    raise ValueError(
        "Could not identify the default target column."
    )

print(f"Target variable: {target_col}")

y_2022 = pd.to_numeric(
    final_2022[target_col],
    errors="coerce"
)

valid_target = y_2022.notna()

y_2022 = y_2022.loc[valid_target].astype(int)

pred_2022 = final_2022.loc[
    valid_target,
    "PredictedPD"
].astype(float)

# 6. 2022 FULL SAMPLE METRICS

print("\n" + "=" * 80)
print("STEP 22D — 2022 DISCRIMINATION + CALIBRATION")
print("=" * 80)

actual_default_rate = y_2022.mean()
average_pd = pred_2022.mean()

roc_auc = roc_auc_score(
    y_2022,
    pred_2022
)

pr_auc = average_precision_score(
    y_2022,
    pred_2022
)

brier = brier_score_loss(
    y_2022,
    pred_2022
)

fpr, tpr, thresholds = roc_curve(
    y_2022,
    pred_2022
)

ks = np.max(
    tpr - fpr
)

calibration_gap = (
    average_pd -
    actual_default_rate
)

print(f"Loans                  : {len(y_2022):,}")
print(f"Defaults               : {y_2022.sum():,}")
print(f"Actual default rate    : {actual_default_rate * 100:.2f}%")
print(f"Average predicted PD   : {average_pd * 100:.2f}%")
print(f"Calibration gap        : {calibration_gap * 100:.2f} pp")
print(f"ROC-AUC                : {roc_auc:.4f}")
print(f"PR-AUC                 : {pr_auc:.4f}")
print(f"KS                     : {ks:.4f}")
print(f"Brier score            : {brier:.4f}")

# 7. TOP-DECILE SEPARATION

print("\n" + "=" * 80)
print("STEP 22E — TOP-DECILE RISK SEPARATION")
print("=" * 80)

risk_rank = final_2022.loc[
    valid_target,
    ["PredictedPD"]
].copy()

risk_rank["Default"] = y_2022.values

risk_rank = risk_rank.sort_values(
    "PredictedPD",
    ascending=False
).reset_index(drop=True)

top_n = max(
    1,
    int(np.ceil(len(risk_rank) * 0.10))
)

top_decile = risk_rank.iloc[:top_n]

top_decile_default_rate = top_decile["Default"].mean()

top_decile_lift = (
    top_decile_default_rate /
    actual_default_rate
)

print(f"Top-decile loans       : {len(top_decile):,}")
print(
    f"Top-decile default rate: "
    f"{top_decile_default_rate * 100:.2f}%"
)
print(f"Top-decile lift        : {top_decile_lift:.2f}x")

# 8. SEASONING STATUS

print("\n" + "=" * 80)
print("STEP 22F — SEASONING ROBUSTNESS")
print("=" * 80)

seasoning_status = None

if "seasoning" in globals():

    season_tmp = seasoning.copy()

    if "LoanSequenceNumber" in season_tmp.columns:

        season_tmp = season_tmp[
            [
                "LoanSequenceNumber",
                "FullySeasoned"
            ]
        ].drop_duplicates(
            "LoanSequenceNumber"
        )

        final_2022 = final_2022.merge(
            season_tmp,
            on="LoanSequenceNumber",
            how="left"
        )

        seasoning_status = final_2022["FullySeasoned"]

elif "seasoned_2022" in globals():

    seasoned_ids = set(
        seasoned_2022["LoanSequenceNumber"]
    )

    seasoning_status = final_2022[
        "LoanSequenceNumber"
    ].isin(seasoned_ids)

else:

    print(
        "⚠ Seasoning table not available. "
        "Using previously established Step 21 result."
    )

if seasoning_status is not None:

    final_2022["FullySeasoned"] = (
        seasoning_status
        .fillna(False)
        .astype(bool)
    )

    seasoned_mask = (
        valid_target.values &
        final_2022["FullySeasoned"].values
    )

    y_seasoned = y_2022.loc[
        final_2022.loc[valid_target, "FullySeasoned"].values
    ]

    pred_seasoned = pred_2022.loc[
        final_2022.loc[valid_target, "FullySeasoned"].values
    ]

    if len(y_seasoned) > 0 and y_seasoned.nunique() == 2:

        seasoned_actual = y_seasoned.mean()
        seasoned_pd = pred_seasoned.mean()

        seasoned_auc = roc_auc_score(
            y_seasoned,
            pred_seasoned
        )

        seasoned_pr = average_precision_score(
            y_seasoned,
            pred_seasoned
        )

        seasoned_brier = brier_score_loss(
            y_seasoned,
            pred_seasoned
        )

        seasoned_gap = (
            seasoned_pd -
            seasoned_actual
        )

        print(
            f"Fully seasoned loans : {len(y_seasoned):,}"
        )
        print(
            f"Seasoned default rate : "
            f"{seasoned_actual * 100:.2f}%"
        )
        print(
            f"Seasoned average PD   : "
            f"{seasoned_pd * 100:.2f}%"
        )
        print(
            f"Seasoned calibration  : "
            f"{seasoned_gap * 100:.2f} pp"
        )
        print(
            f"Seasoned ROC-AUC      : "
            f"{seasoned_auc:.4f}"
        )
        print(
            f"Seasoned PR-AUC       : "
            f"{seasoned_pr:.4f}"
        )
        print(
            f"Seasoned Brier        : "
            f"{seasoned_brier:.4f}"
        )

        print("\nComparison with all 2022:")
        print(
            f"Calibration change    : "
            f"{(seasoned_gap - calibration_gap) * 100:+.2f} pp"
        )
        print(
            f"ROC-AUC change        : "
            f"{seasoned_auc - roc_auc:+.4f}"
        )

    else:

        print(
            "⚠ Seasoned sample cannot support "
            "binary performance metrics."
        )

# 9. VINTAGE-LEVEL MODEL PERFORMANCE

print("\n" + "=" * 80)
print("STEP 22G — VINTAGE PERFORMANCE TREND")
print("=" * 80)

# Use existing out-of-time predictions if available.
# Otherwise generate predictions for model_df by vintage.

vintage_results = []

if "oot_train" in globals() and "oot_test" in globals():

    candidate_vintages = sorted(
        pd.to_numeric(
            model_df["Vintage"],
            errors="coerce"
        ).dropna().unique()
    )

else:

    candidate_vintages = sorted(
        pd.to_numeric(
            model_df["Vintage"],
            errors="coerce"
        ).dropna().unique()
    )

for vintage in candidate_vintages:

    vintage_df = model_df.loc[
        pd.to_numeric(
            model_df["Vintage"],
            errors="coerce"
        ) == vintage
    ].copy()

    missing_vintage_features = [
        c for c in feature_list
        if c not in vintage_df.columns
    ]

    if missing_vintage_features:
        continue

    y_v = pd.to_numeric(
        vintage_df[target_col],
        errors="coerce"
    )

    valid_v = y_v.notna()

    y_v = y_v.loc[valid_v].astype(int)

    X_v = vintage_df.loc[
        valid_v,
        feature_list
    ].copy()

    for col in X_v.columns:

        if not pd.api.types.is_numeric_dtype(
            X_v[col]
        ):

            X_v[col] = pd.to_numeric(
                X_v[col],
                errors="coerce"
            )

    X_v = X_v.replace(
        [np.inf, -np.inf],
        np.nan
    )

    X_v = X_v.fillna(
        X_v.median(numeric_only=True)
    )

    if len(y_v) == 0 or y_v.nunique() < 2:
        continue

    try:

        p_v = trained_model.predict_proba(
            X_v
        )[:, 1]

        auc_v = roc_auc_score(
            y_v,
            p_v
        )

        pr_v = average_precision_score(
            y_v,
            p_v
        )

        brier_v = brier_score_loss(
            y_v,
            p_v
        )

        actual_v = y_v.mean()
        pd_v = p_v.mean()
        gap_v = pd_v - actual_v

        vintage_results.append(
            {
                "Vintage": int(vintage),
                "Loans": len(y_v),
                "Defaults": int(y_v.sum()),
                "Actual_Default_Rate_%": actual_v * 100,
                "Average_PD_%": pd_v * 100,
                "Calibration_Gap_pp": gap_v * 100,
                "ROC_AUC": auc_v,
                "PR_AUC": pr_v,
                "Brier": brier_v
            }
        )

    except Exception as e:

        print(
            f"⚠ Could not calculate vintage {vintage}: {e}"
        )

vintage_results_df = pd.DataFrame(
    vintage_results
)

if len(vintage_results_df) > 0:

    vintage_results_df = vintage_results_df.sort_values(
        "Vintage"
    ).reset_index(drop=True)

    print(
        vintage_results_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# 10. FINAL DIAGNOSTIC TABLE

print("\n" + "=" * 80)
print("STEP 22H — FINAL DIAGNOSTIC TABLE")
print("=" * 80)

final_diagnostic = pd.DataFrame(
    {
        "Metric": [
            "2022 Loans",
            "2022 Actual Default Rate",
            "2022 Average Predicted PD",
            "2022 Calibration Gap",
            "2022 ROC-AUC",
            "2022 PR-AUC",
            "2022 KS",
            "2022 Brier",
            "Top-Decile Default Rate",
            "Top-Decile Lift"
        ],
        "Value": [
            f"{len(y_2022):,}",
            f"{actual_default_rate * 100:.2f}%",
            f"{average_pd * 100:.2f}%",
            f"{calibration_gap * 100:.2f} pp",
            f"{roc_auc:.4f}",
            f"{pr_auc:.4f}",
            f"{ks:.4f}",
            f"{brier:.4f}",
            f"{top_decile_default_rate * 100:.2f}%",
            f"{top_decile_lift:.2f}x"
        ]
    }
)

print(
    final_diagnostic.to_string(
        index=False
    )
)

# 11. AUTOMATIC INTERPRETATION

print("\n" + "=" * 100)
print("STEP 22I — FINAL INTERPRETATION")
print("=" * 100)

print(
    "\n1. SEASONING"
)

print(
    "The fully-seasoned comparison should be used to determine "
    "whether right-censoring explains the 2022 deterioration."
)

print(
    "\n2. CALIBRATION"
)

print(
    f"The 2022 model predicts an average PD of "
    f"{average_pd * 100:.2f}% against an observed default rate "
    f"of {actual_default_rate * 100:.2f}%, producing a calibration "
    f"gap of {calibration_gap * 100:.2f} percentage points."
)

print(
    "\n3. DISCRIMINATION"
)

print(
    f"The model retains ROC-AUC of {roc_auc:.4f} in 2022. "
    f"This means the model still contains useful ranking information "
    f"even though its absolute PD levels are poorly calibrated."
)

print(
    "\n4. IMPORTANT DISTINCTION"
)

print(
    "A model can simultaneously have acceptable discrimination "
    "and poor calibration. Therefore, the 2022 result should NOT "
    "be described simply as 'the model failed'."
)

print(
    "\n5. INTERPRETATION OF THE CURRENT EVIDENCE"
)

print(
    "Your Step 21 result shows that restricting the 2022 cohort "
    "to fully seasoned loans changes the calibration gap by only "
    "+0.12 percentage points (9.00 pp → 9.11 pp) while ROC-AUC "
    "changes only from 0.6829 to 0.6868."
)

print(
    "\nTherefore:"
)

print(
    "✓ Right-censoring does NOT explain the main deterioration."
)

print(
    "✓ The 2022 result is robust to the seasoning correction."
)

print(
    "✓ The model still has meaningful ranking ability."
)

print(
    "✓ The main weakness is calibration / population or regime "
    "shift rather than complete loss of predictive power."
)

print(
    "\nThis is a much stronger and more defensible credit-risk "
    "modeling conclusion than simply reporting the original 2022 AUC."
)

print("\n" + "=" * 100)
print("STEP 22 COMPLETE")
print("=" * 100)


In [ ]:

print("\n" + "=" * 80)
print("STEP 22F — SEASONING ROBUSTNESS")
print("=" * 80)

# 1. Inspect seasoning object

print("\nSeasoning object columns:")
print(list(seasoning.columns))

print("\nSeasoning shape:")
print(seasoning.shape)

# 2. Identify the loan ID column

loan_id_candidates = [
    "LoanSequenceNumber",
    "loan_sequence_number",
    "LoanSequence",
    "LoanID",
    "loan_id"
]

season_loan_col = None

for col in loan_id_candidates:
    if col in seasoning.columns:
        season_loan_col = col
        break

if season_loan_col is None:
    raise ValueError(
        "Could not identify LoanSequenceNumber in seasoning DataFrame. "
        f"Available columns are: {list(seasoning.columns)}"
    )

print(f"\n✓ Seasoning loan-ID column: {season_loan_col}")

# 3. Identify the seasoning-status column

seasoning_candidates = [
    "FullySeasoned",
    "Fully_Seasoned",
    "fully_seasoned",
    "IsFullySeasoned",
    "Seasoned",
    "SeasoningStatus"
]

season_col = None

for col in seasoning_candidates:
    if col in seasoning.columns:
        season_col = col
        break

# 4. If no explicit boolean exists, reconstruct from last reporting month

if season_col is None:

    reporting_candidates = [
        "LastReportingPeriod",
        "Last_Reporting_Period",
        "MaxReportingPeriod",
        "Max_Reporting_Period",
        "LastReportingDate",
        "Last_Reporting_Date"
    ]

    reporting_col = None

    for col in reporting_candidates:
        if col in seasoning.columns:
            reporting_col = col
            break

    if reporting_col is not None:

        print(
            f"✓ Reconstructing seasoning from: {reporting_col}"
        )

        # Convert YYYYMM values safely
        reporting_num = pd.to_numeric(
            seasoning[reporting_col],
            errors="coerce"
        )

        # For each vintage, calculate the required 36-month endpoint.
        seasoning["_Required36M"] = (
            pd.to_numeric(
                seasoning["Vintage"],
                errors="coerce"
            ) * 100 + 12
        )

        # The above gives December of vintage year.
        # Add 36 months = December of vintage year + 3 years.
        seasoning["_Required36M"] = (
            (pd.to_numeric(
                seasoning["Vintage"],
                errors="coerce"
            ) + 3) * 100 + 12
        )

        seasoning["_FullySeasoned_temp"] = (
            reporting_num >=
            seasoning["_Required36M"]
        )

        season_col = "_FullySeasoned_temp"

    else:

        raise ValueError(
            "Could not find a seasoning-status or last-reporting-period "
            "column in seasoning.\n"
            f"Available columns: {list(seasoning.columns)}"
        )

print(f"✓ Seasoning status column: {season_col}")

# 5. Keep only required columns

season_status = seasoning[
    [
        season_loan_col,
        season_col
    ]
].copy()

season_status = season_status.rename(
    columns={
        season_loan_col: "LoanSequenceNumber",
        season_col: "FullySeasoned"
    }
)

season_status = season_status.drop_duplicates(
    subset=["LoanSequenceNumber"]
)

season_status["LoanSequenceNumber"] = (
    season_status["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

season_status["FullySeasoned"] = (
    season_status["FullySeasoned"]
    .fillna(False)
    .astype(bool)
)

# 6. Match seasoning to 2022 loans

final_2022["LoanSequenceNumber"] = (
    final_2022["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

final_2022 = final_2022.drop(
    columns=["FullySeasoned"],
    errors="ignore"
)

final_2022 = final_2022.merge(
    season_status,
    on="LoanSequenceNumber",
    how="left"
)

final_2022["FullySeasoned"] = (
    final_2022["FullySeasoned"]
    .fillna(False)
    .astype(bool)
)

# 7. Seasoning counts

n_all = len(final_2022)

n_seasoned = int(
    final_2022["FullySeasoned"].sum()
)

n_not_seasoned = (
    n_all - n_seasoned
)

print("\n" + "=" * 80)
print("2022 SEASONING COUNTS")
print("=" * 80)

print(f"All 2022 loans       : {n_all:,}")
print(f"Fully seasoned       : {n_seasoned:,}")
print(f"Not fully seasoned   : {n_not_seasoned:,}")
print(
    f"Fully seasoned %     : "
    f"{n_seasoned / n_all * 100:.2f}%"
)

print("\n" + "=" * 80)
print("STEP 22G — ALL VS FULLY SEASONED PERFORMANCE")
print("=" * 80)

def calculate_metrics(df):
    """
    Calculate credit-risk performance metrics for a given sample.
    """

    y = pd.to_numeric(
        df["Default_36M"],
        errors="coerce"
    )

    p = pd.to_numeric(
        df["PredictedPD"],
        errors="coerce"
    )

    valid = (
        y.notna() &
        p.notna()
    )

    y = y.loc[valid].astype(int)
    p = p.loc[valid].astype(float)

    if len(y) == 0:
        return {
            "Loans": 0,
            "Defaults": np.nan,
            "Default_Rate_%": np.nan,
            "Average_PD_%": np.nan,
            "Calibration_Gap_pp": np.nan,
            "ROC_AUC": np.nan,
            "PR_AUC": np.nan,
            "KS": np.nan,
            "Brier": np.nan,
            "Top_Decile_Default_%": np.nan,
            "Top_Decile_Lift": np.nan
        }

    actual_rate = y.mean()
    avg_pd = p.mean()

    # ROC / PR require both classes
    if y.nunique() == 2:

        auc = roc_auc_score(
            y,
            p
        )

        pr = average_precision_score(
            y,
            p
        )

        fpr, tpr, _ = roc_curve(
            y,
            p
        )

        ks_value = np.max(
            tpr - fpr
        )

    else:

        auc = np.nan
        pr = np.nan
        ks_value = np.nan

    brier = brier_score_loss(
        y,
        p
    )

    # Top 10%
    temp = pd.DataFrame(
        {
            "PD": p.values,
            "Default": y.values
        }
    )

    temp = temp.sort_values(
        "PD",
        ascending=False
    )

    top_n = max(
        1,
        int(np.ceil(len(temp) * 0.10))
    )

    top_decile = temp.iloc[:top_n]

    top_default = top_decile["Default"].mean()

    if actual_rate > 0:
        lift = top_default / actual_rate
    else:
        lift = np.nan

    return {
        "Loans": len(y),
        "Defaults": int(y.sum()),
        "Default_Rate_%": actual_rate * 100,
        "Average_PD_%": avg_pd * 100,
        "Calibration_Gap_pp": (
            avg_pd - actual_rate
        ) * 100,
        "ROC_AUC": auc,
        "PR_AUC": pr,
        "KS": ks_value,
        "Brier": brier,
        "Top_Decile_Default_%": top_default * 100,
        "Top_Decile_Lift": lift
    }

# All 2022
metrics_all = calculate_metrics(
    final_2022
)

# Fully seasoned 2022
seasoned_df = final_2022.loc[
    final_2022["FullySeasoned"]
].copy()

metrics_seasoned = calculate_metrics(
    seasoned_df
)

comparison = pd.DataFrame(
    [
        {
            "Sample": "All 2022",
            **metrics_all
        },
        {
            "Sample": "Fully Seasoned 2022",
            **metrics_seasoned
        }
    ]
)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n" + "=" * 80)
print("STEP 22H — DIRECT SEASONING EFFECT")
print("=" * 80)

gap_change = (
    metrics_seasoned["Calibration_Gap_pp"]
    -
    metrics_all["Calibration_Gap_pp"]
)

auc_change = (
    metrics_seasoned["ROC_AUC"]
    -
    metrics_all["ROC_AUC"]
)

default_rate_change = (
    metrics_seasoned["Default_Rate_%"]
    -
    metrics_all["Default_Rate_%"]
)

print(
    f"Calibration gap — All 2022       : "
    f"{metrics_all['Calibration_Gap_pp']:.2f} pp"
)

print(
    f"Calibration gap — Seasoned 2022  : "
    f"{metrics_seasoned['Calibration_Gap_pp']:.2f} pp"
)

print(
    f"Change in calibration gap        : "
    f"{gap_change:+.2f} pp"
)

print(
    f"\nROC-AUC — All 2022              : "
    f"{metrics_all['ROC_AUC']:.4f}"
)

print(
    f"ROC-AUC — Seasoned 2022         : "
    f"{metrics_seasoned['ROC_AUC']:.4f}"
)

print(
    f"Change in ROC-AUC                : "
    f"{auc_change:+.4f}"
)

print(
    f"\nDefault rate — All 2022         : "
    f"{metrics_all['Default_Rate_%']:.2f}%"
)

print(
    f"Default rate — Seasoned 2022    : "
    f"{metrics_seasoned['Default_Rate_%']:.2f}%"
)

print(
    f"Change in default rate           : "
    f"{default_rate_change:+.2f} pp"
)

print("\n" + "=" * 100)
print("STEP 22I — FINAL CREDIT-RISK DIAGNOSIS")
print("=" * 100)

abs_gap_change = abs(gap_change)

if abs_gap_change < 0.50:

    print(
        "\n✓ RIGHT-CENSORING IS NOT THE MAIN EXPLANATION."
    )

    print(
        f"The calibration gap changes by only "
        f"{gap_change:+.2f} pp after restricting the sample "
        f"to fully seasoned loans."
    )

    print(
        "\nThe original 2022 deterioration therefore survives "
        "the seasoning correction."
    )

else:

    print(
        "\n⚠ Seasoning materially changes calibration."
    )

    print(
        f"The calibration gap changes by "
        f"{gap_change:+.2f} pp."
    )

print(
    "\n✓ IMPORTANT: THE MODEL HAS NOT 'FAILED'."
)

print(
    f"The model retains ROC-AUC of "
    f"{metrics_all['ROC_AUC']:.4f} in 2022."
)

print(
    f"It also achieves "
    f"{metrics_all['Top_Decile_Lift']:.2f}x "
    f"top-decile lift."
)

print(
    "\nThe evidence therefore points to a distinction between:"
)

print(
    "  • DISCRIMINATION — the model can still rank risk."
)

print(
    "  • CALIBRATION — predicted PD levels are substantially "
    "above realized default rates."
)

print(
    "\nThis is consistent with MODEL DRIFT / POPULATION SHIFT "
    "rather than complete predictive failure."
)

print(
    "\nMost importantly, the seasoning test shows that the "
    "calibration deterioration cannot be dismissed as a simple "
    "right-censoring artifact."
)

print("\n" + "=" * 100)
print("STEP 22 COMPLETE")
print("=" * 100)


In [ ]:
# Objective:
# Quantify whether the 2022 OOT population differs materially from the
# 2018–2021 training population.
#
# Diagnostics:
#   1. PSI (Population Stability Index)
#   2. Standardized Mean Difference
#   3. Missing-rate shift
#   4. Quantile comparison
#   5. Automatic ranking of the largest shifts
#
# IMPORTANT:
# This cell does NOT modify the model or the dataset.

import numpy as np
import pandas as pd

print("=" * 100)
print("STEP 23 — POPULATION SHIFT / MODEL DRIFT DIAGNOSTIC")
print("=" * 100)

required_objects = ["model_df", "oot_train", "oot_test", "feature_cols", "model"]

missing_objects = [
    x for x in required_objects
    if x not in globals()
]

if missing_objects:
    raise ValueError(
        f"Missing required objects: {missing_objects}"
    )

print("\n✓ Required objects found.")

# Training population = 2018–2021
train_pop = model_df[
    model_df["Vintage"].astype(int) <= 2021
].copy()

# 2022 OOT population
oot_2022 = model_df[
    model_df["Vintage"].astype(int) == 2022
].copy()

print("\n" + "=" * 80)
print("POPULATION COUNTS")
print("=" * 80)

print(f"Training population (2018–2021): {len(train_pop):,}")
print(f"2022 OOT population            : {len(oot_2022):,}")

if len(train_pop) == 0 or len(oot_2022) == 0:
    raise ValueError("Training or 2022 OOT population is empty.")

missing_train = [
    f for f in feature_cols
    if f not in train_pop.columns
]

missing_oot = [
    f for f in feature_cols
    if f not in oot_2022.columns
]

if missing_train or missing_oot:
    raise ValueError(
        f"Missing features.\n"
        f"Training: {missing_train}\n"
        f"2022 OOT: {missing_oot}"
    )

print(f"\n✓ All {len(feature_cols)} model features available.")

def calculate_psi(train_series, oot_series, bins=10):
    """
    Population Stability Index.

    PSI interpretation used here:
        < 0.10       = little/no shift
        0.10–0.25    = moderate shift
        >= 0.25      = substantial shift

    Bins are determined from the TRAINING population and then applied
    identically to the OOT population.
    """

    train = pd.to_numeric(
        train_series,
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    oot = pd.to_numeric(
        oot_series,
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(train) == 0 or len(oot) == 0:
        return np.nan

    # Constant variable
    if train.nunique() <= 1:
        return 0.0

    # Quantile-based bins from TRAINING population
    try:
        quantiles = np.linspace(0, 1, bins + 1)

        edges = np.unique(
            train.quantile(quantiles).values
        )

        if len(edges) < 3:
            return 0.0

        # Extend endpoints slightly
        edges[0] = -np.inf
        edges[-1] = np.inf

        train_bin = pd.cut(
            train,
            bins=edges,
            include_lowest=True
        )

        oot_bin = pd.cut(
            oot,
            bins=edges,
            include_lowest=True
        )

        train_pct = (
            train_bin.value_counts(
                sort=False,
                normalize=True
            )
            .astype(float)
        )

        oot_pct = (
            oot_bin.value_counts(
                sort=False,
                normalize=True
            )
            .astype(float)
        )

        # Align bins
        oot_pct = oot_pct.reindex(
            train_pct.index,
            fill_value=0
        )

        # Avoid log(0)
        eps = 1e-6

        train_pct = train_pct.clip(lower=eps)
        oot_pct = oot_pct.clip(lower=eps)

        psi = (
            (oot_pct - train_pct)
            * np.log(oot_pct / train_pct)
        ).sum()

        return float(psi)

    except Exception:
        return np.nan

def standardized_mean_difference(train_series, oot_series):

    train = pd.to_numeric(
        train_series,
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    oot = pd.to_numeric(
        oot_series,
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(train) == 0 or len(oot) == 0:
        return np.nan

    mean_train = train.mean()
    mean_oot = oot.mean()

    var_train = train.var()
    var_oot = oot.var()

    pooled_sd = np.sqrt(
        (var_train + var_oot) / 2
    )

    if pooled_sd == 0 or np.isnan(pooled_sd):
        return 0.0

    return float(
        (mean_oot - mean_train) / pooled_sd
    )

results = []

for feature in feature_cols:

    train_s = train_pop[feature]
    oot_s = oot_2022[feature]

    train_numeric = pd.to_numeric(
        train_s,
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    )

    oot_numeric = pd.to_numeric(
        oot_s,
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    )

    train_valid = train_numeric.dropna()
    oot_valid = oot_numeric.dropna()

    psi = calculate_psi(
        train_s,
        oot_s
    )

    smd = standardized_mean_difference(
        train_s,
        oot_s
    )

    train_missing = train_numeric.isna().mean()
    oot_missing = oot_numeric.isna().mean()

    train_mean = train_valid.mean() if len(train_valid) else np.nan
    oot_mean = oot_valid.mean() if len(oot_valid) else np.nan

    train_median = (
        train_valid.median()
        if len(train_valid)
        else np.nan
    )

    oot_median = (
        oot_valid.median()
        if len(oot_valid)
        else np.nan
    )

    train_std = train_valid.std() if len(train_valid) else np.nan
    oot_std = oot_valid.std() if len(oot_valid) else np.nan

    abs_smd = (
        abs(smd)
        if pd.notna(smd)
        else np.nan
    )

    if pd.isna(psi):
        shift_class = "Unavailable"
    elif psi >= 0.25:
        shift_class = "Substantial"
    elif psi >= 0.10:
        shift_class = "Moderate"
    else:
        shift_class = "Low"

    results.append({
        "Feature": feature,
        "Train_Mean": train_mean,
        "OOT_2022_Mean": oot_mean,
        "Train_Median": train_median,
        "OOT_2022_Median": oot_median,
        "Train_SD": train_std,
        "OOT_2022_SD": oot_std,
        "PSI": psi,
        "Abs_SMD": abs_smd,
        "SMD": smd,
        "Train_Missing_%": train_missing * 100,
        "OOT_Missing_%": oot_missing * 100,
        "Missing_Shift_pp": (
            oot_missing - train_missing
        ) * 100,
        "Shift": shift_class
    })

shift_results = pd.DataFrame(results)

shift_results = shift_results.sort_values(
    ["PSI", "Abs_SMD"],
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 100)
print("FEATURE-LEVEL POPULATION SHIFT")
print("=" * 100)

display(
    shift_results[
        [
            "Feature",
            "Train_Mean",
            "OOT_2022_Mean",
            "Train_Median",
            "OOT_2022_Median",
            "PSI",
            "Abs_SMD",
            "SMD",
            "Train_Missing_%",
            "OOT_Missing_%",
            "Missing_Shift_pp",
            "Shift"
        ]
    ].round(4)
)

print("\n" + "=" * 100)
print("TOP 10 FEATURES BY PSI")
print("=" * 100)

top_psi = (
    shift_results
    .sort_values("PSI", ascending=False)
    .head(10)
)

display(
    top_psi[
        [
            "Feature",
            "PSI",
            "Abs_SMD",
            "SMD",
            "Train_Mean",
            "OOT_2022_Mean",
            "Shift"
        ]
    ].round(4)
)

print("\n" + "=" * 100)
print("TOP 10 FEATURES BY ABSOLUTE STANDARDIZED MEAN DIFFERENCE")
print("=" * 100)

top_smd = (
    shift_results
    .sort_values("Abs_SMD", ascending=False)
    .head(10)
)

display(
    top_smd[
        [
            "Feature",
            "Abs_SMD",
            "SMD",
            "PSI",
            "Train_Mean",
            "OOT_2022_Mean",
            "Shift"
        ]
    ].round(4)
)

psi_low = (
    shift_results["PSI"] < 0.10
).sum()

psi_moderate = (
    (shift_results["PSI"] >= 0.10) &
    (shift_results["PSI"] < 0.25)
).sum()

psi_high = (
    shift_results["PSI"] >= 0.25
).sum()

print("\n" + "=" * 100)
print("PSI SUMMARY")
print("=" * 100)

print(
    f"Low shift       (PSI < 0.10) : {psi_low}"
)

print(
    f"Moderate shift  (0.10–0.25)  : {psi_moderate}"
)

print(
    f"Substantial     (PSI >= 0.25) : {psi_high}"
)

valid_psi = shift_results[
    shift_results["PSI"].notna()
]["PSI"]

if len(valid_psi) > 0:

    mean_psi = valid_psi.mean()
    median_psi = valid_psi.median()
    max_psi = valid_psi.max()

else:

    mean_psi = np.nan
    median_psi = np.nan
    max_psi = np.nan

print("\n" + "=" * 100)
print("OVERALL POPULATION SHIFT")
print("=" * 100)

print(
    f"Mean PSI       : {mean_psi:.4f}"
)

print(
    f"Median PSI     : {median_psi:.4f}"
)

print(
    f"Maximum PSI    : {max_psi:.4f}"
)

# This is especially important because it tells us whether the model is
# receiving a fundamentally different risk population in 2022.

print("\n" + "=" * 100)
print("MODEL PREDICTED-RISK DISTRIBUTION SHIFT")
print("=" * 100)

# Generate predictions for training population
X_train_risk = train_pop[feature_cols].copy()
X_oot_risk = oot_2022[feature_cols].copy()

train_pred = model.predict_proba(
    X_train_risk
)[:, 1]

oot_pred = model.predict_proba(
    X_oot_risk
)[:, 1]

risk_train = pd.Series(
    train_pred,
    name="PredictedPD"
)

risk_oot = pd.Series(
    oot_pred,
    name="PredictedPD"
)

risk_psi = calculate_psi(
    risk_train,
    risk_oot
)

print(
    f"Training average predicted PD : "
    f"{risk_train.mean() * 100:.2f}%"
)

print(
    f"2022 average predicted PD     : "
    f"{risk_oot.mean() * 100:.2f}%"
)

print(
    f"Training median predicted PD  : "
    f"{risk_train.median() * 100:.2f}%"
)

print(
    f"2022 median predicted PD      : "
    f"{risk_oot.median() * 100:.2f}%"
)

print(
    f"\nPredicted-risk PSI            : "
    f"{risk_psi:.4f}"
)

if pd.notna(risk_psi):

    if risk_psi >= 0.25:
        risk_shift_interpretation = (
            "SUBSTANTIAL predicted-risk population shift"
        )

    elif risk_psi >= 0.10:
        risk_shift_interpretation = (
            "MODERATE predicted-risk population shift"
        )

    else:
        risk_shift_interpretation = (
            "LOW predicted-risk population shift"
        )

    print(
        f"Interpretation                : "
        f"{risk_shift_interpretation}"
    )

actual_2022 = pd.to_numeric(
    oot_2022["Default_36M"],
    errors="coerce"
)

actual_2022 = actual_2022.dropna()

realized_default_rate = actual_2022.mean()
average_predicted_pd = risk_oot.mean()

print("\n" + "=" * 100)
print("2022 CALIBRATION")
print("=" * 100)

print(
    f"Realized default rate : "
    f"{realized_default_rate * 100:.2f}%"
)

print(
    f"Average predicted PD  : "
    f"{average_predicted_pd * 100:.2f}%"
)

print(
    f"Calibration gap       : "
    f"{(average_predicted_pd - realized_default_rate) * 100:.2f} pp"
)

print("\n" + "=" * 100)
print("STEP 23 — AUTOMATIC DIAGNOSIS")
print("=" * 100)

substantial_features = shift_results[
    shift_results["PSI"] >= 0.25
]["Feature"].tolist()

moderate_features = shift_results[
    (shift_results["PSI"] >= 0.10) &
    (shift_results["PSI"] < 0.25)
]["Feature"].tolist()

if substantial_features:

    print(
        "\n✓ EVIDENCE OF SUBSTANTIAL POPULATION SHIFT"
    )

    print(
        "\nFeatures with PSI >= 0.25:"
    )

    for f in substantial_features:
        print(
            f"   • {f}"
        )

elif moderate_features:

    print(
        "\n⚠ EVIDENCE OF MODERATE POPULATION SHIFT"
    )

    print(
        "\nFeatures with PSI between 0.10 and 0.25:"
    )

    for f in moderate_features:
        print(
            f"   • {f}"
        )

else:

    print(
        "\n⚠ No major feature-level population shift "
        "detected using PSI."
    )

print("\n" + "=" * 100)
print("FINAL INTERPRETATION")
print("=" * 100)

print(
    """
The preceding seasoning analysis established that the 2022 calibration
deterioration is NOT explained by simple right-censoring.

This step tests the next hypothesis: population/model drift.

Interpretation should be based on BOTH:
    1. Feature-level PSI / SMD
    2. Predicted-risk distribution shift

If several economically important model variables show material PSI,
and/or the predicted-risk distribution itself shifts materially, this
provides quantitative evidence that the 2022 OOT population differs from
the population on which the model was trained.

The correct conclusion is NOT that the model has 'failed'.

A model can retain useful discrimination while becoming poorly calibrated
when the underlying population or default environment changes.

Therefore:

    High population shift + retained AUC
        -> strong evidence of model drift / population shift.

    Low population shift + large calibration deterioration
        -> investigate calibration methodology, target definition,
           temporal default-rate changes, or model specification.

    High population shift + major AUC deterioration
        -> stronger evidence of broader model instability.
"""
)

population_shift_results = shift_results.copy()

risk_shift_summary = pd.DataFrame({
    "Metric": [
        "Training average predicted PD",
        "2022 average predicted PD",
        "Training median predicted PD",
        "2022 median predicted PD",
        "Predicted-risk PSI",
        "2022 realized default rate",
        "2022 calibration gap"
    ],
    "Value": [
        risk_train.mean(),
        risk_oot.mean(),
        risk_train.median(),
        risk_oot.median(),
        risk_psi,
        realized_default_rate,
        average_predicted_pd - realized_default_rate
    ]
})

print("\n" + "=" * 100)
print("STEP 23 COMPLETE")
print("=" * 100)

print(
    "\n✓ population_shift_results created."
)

print(
    "✓ risk_shift_summary created."
)

print(
    "\nDo NOT change the model yet."
)

print(
    "Next decision should be based on the magnitude and location "
    "of the observed population shift."
)


In [ ]:
# CORRECTED: REBUILDS ENGINEERED FLAGS BEFORE PREDICTION

import numpy as np
import pandas as pd
import warnings

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

print("=" * 100)
print("STEP 24 — CALIBRATION INTERCEPT, SLOPE & DECILE DRIFT DIAGNOSTIC")
print("=" * 100)

required_objects = [
    "model",
    "feature_cols",
    "oot_train",
    "oot_test"
]

missing_objects = [
    x for x in required_objects
    if x not in globals()
]

if missing_objects:
    raise ValueError(
        f"Missing required objects: {missing_objects}"
    )

print("\n✓ Required objects found.")

train_df = oot_train.copy()
oot_2022 = oot_test.copy()

target_col = "Default_36M"

if target_col not in train_df.columns:
    raise ValueError(
        f"{target_col} missing from training data."
    )

if target_col not in oot_2022.columns:
    raise ValueError(
        f"{target_col} missing from 2022 data."
    )

print("\n" + "=" * 80)
print("POPULATION PREPARATION")
print("=" * 80)

print(
    f"Training observations : {len(train_df):,}"
)

print(
    f"2022 OOT observations : {len(oot_2022):,}"
)

print("\n" + "=" * 80)
print("REBUILDING ENGINEERED FLAGS")
print("=" * 80)

def rebuild_credit_flags(df):

    df = df.copy()

    # Ensure numeric variables are numeric

    numeric_cols = [
        "CreditScore",
        "OriginalLTV",
        "OriginalDTI",
        "OriginalCLTV",
        "OriginalInterestRate"
    ]

    for col in numeric_cols:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    # Low credit

    if "CreditScore" in df.columns:

        df["LowCreditFlag"] = (
            df["CreditScore"] < 680
        ).astype(int)

    # High LTV

    if "OriginalLTV" in df.columns:

        df["HighLTVFlag"] = (
            df["OriginalLTV"] >= 80
        ).astype(int)

    # High DTI

    if "OriginalDTI" in df.columns:

        df["HighDTIFlag"] = (
            df["OriginalDTI"] >= 43
        ).astype(int)

    # High CLTV

    if "OriginalCLTV" in df.columns:

        df["HighCLTVFlag"] = (
            df["OriginalCLTV"] >= 80
        ).astype(int)

    # High interest rate

    if "OriginalInterestRate" in df.columns:

        # Use same training-based threshold if available.
        # Otherwise use the 90th percentile of the training population.
        pass

    # High Credit-LTV combination

    if (
        "CreditScore" in df.columns
        and "OriginalLTV" in df.columns
    ):

        df["HighCreditLTVFlag"] = (
            (df["CreditScore"] < 680)
            &
            (df["OriginalLTV"] >= 80)
        ).astype(int)

    # High DTI-LTV combination

    if (
        "OriginalDTI" in df.columns
        and "OriginalLTV" in df.columns
    ):

        df["HighDTILTVFlag"] = (
            (df["OriginalDTI"] >= 43)
            &
            (df["OriginalLTV"] >= 80)
        ).astype(int)

    return df

train_df = rebuild_credit_flags(train_df)
oot_2022 = rebuild_credit_flags(oot_2022)

if "OriginalInterestRate" in train_df.columns:

    train_rate = pd.to_numeric(
        train_df["OriginalInterestRate"],
        errors="coerce"
    )

    oot_rate = pd.to_numeric(
        oot_2022["OriginalInterestRate"],
        errors="coerce"
    )

    # Reconstruct the same high-rate logic used in the model-building
    # pipeline. If the original threshold object exists, use it.
    possible_threshold_names = [
        "high_rate_threshold",
        "rate_threshold",
        "HighRateThreshold"
    ]

    rate_threshold = None

    for obj_name in possible_threshold_names:

        if obj_name in globals():

            try:

                candidate = float(
                    globals()[obj_name]
                )

                if np.isfinite(candidate):

                    rate_threshold = candidate
                    break

            except Exception:
                pass

    # If no threshold object exists, infer it from the training distribution.
    if rate_threshold is None:

        rate_threshold = train_rate.quantile(
            0.90
        )

    train_df["HighRateFlag"] = (
        train_rate >= rate_threshold
    ).astype(int)

    oot_2022["HighRateFlag"] = (
        oot_rate >= rate_threshold
    ).astype(int)

    print(
        f"HighRateFlag threshold used: "
        f"{rate_threshold:.4f}"
    )

model_features = list(feature_cols)

missing_train = [
    c
    for c in model_features
    if c not in train_df.columns
]

missing_oot = [
    c
    for c in model_features
    if c not in oot_2022.columns
]

print("\nRequired model features:")
print(model_features)

print("\nMissing from training:")
print(missing_train)

print("\nMissing from 2022:")
print(missing_oot)

if missing_train:

    raise ValueError(
        f"Training data still missing model features: "
        f"{missing_train}"
    )

if missing_oot:

    raise ValueError(
        f"2022 data still missing model features: "
        f"{missing_oot}"
    )

print(
    f"\n✓ All {len(model_features)} model features are present."
)

print("\n" + "=" * 80)
print("PREPARING MODEL MATRIX")
print("=" * 80)

X_train = train_df[
    model_features
].copy()

X_2022 = oot_2022[
    model_features
].copy()

y_train = pd.to_numeric(
    train_df[target_col],
    errors="coerce"
).astype(int)

y_2022 = pd.to_numeric(
    oot_2022[target_col],
    errors="coerce"
).astype(int)

# Convert numeric features
for col in model_features:

    X_train[col] = pd.to_numeric(
        X_train[col],
        errors="coerce"
    )

    X_2022[col] = pd.to_numeric(
        X_2022[col],
        errors="coerce"
    )

# Replace inf
X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_2022 = X_2022.replace(
    [np.inf, -np.inf],
    np.nan
)

# Training medians
train_medians = X_train.median()

X_train = X_train.fillna(
    train_medians
)

X_2022 = X_2022.fillna(
    train_medians
)

# Final safety
X_train = X_train.fillna(0)
X_2022 = X_2022.fillna(0)

print("✓ Feature matrices prepared.")

print("\n" + "=" * 80)
print("GENERATING MODEL PREDICTIONS")
print("=" * 80)

train_pd = model.predict_proba(
    X_train
)[:, 1]

oot_pd = model.predict_proba(
    X_2022
)[:, 1]

print("✓ Training predictions generated.")
print("✓ 2022 predictions generated.")

EPS = 1e-6

train_pd_clip = np.clip(
    train_pd,
    EPS,
    1 - EPS
)

oot_pd_clip = np.clip(
    oot_pd,
    EPS,
    1 - EPS
)

print("\n" + "=" * 80)
print("2022 BASELINE PERFORMANCE")
print("=" * 80)

train_default = y_train.mean()
oot_default = y_2022.mean()

train_mean_pd = train_pd.mean()
oot_mean_pd = oot_pd.mean()

train_auc = roc_auc_score(
    y_train,
    train_pd
)

oot_auc = roc_auc_score(
    y_2022,
    oot_pd
)

train_brier = brier_score_loss(
    y_train,
    train_pd
)

oot_brier = brier_score_loss(
    y_2022,
    oot_pd
)

print(
    f"Training default rate : "
    f"{train_default * 100:.2f}%"
)

print(
    f"Training average PD   : "
    f"{train_mean_pd * 100:.2f}%"
)

print(
    f"Training ROC-AUC      : "
    f"{train_auc:.4f}"
)

print(
    f"\n2022 default rate     : "
    f"{oot_default * 100:.2f}%"
)

print(
    f"2022 average PD       : "
    f"{oot_mean_pd * 100:.2f}%"
)

print(
    f"2022 ROC-AUC          : "
    f"{oot_auc:.4f}"
)

print(
    f"2022 Brier score      : "
    f"{oot_brier:.4f}"
)

print("\n" + "=" * 80)
print("CALIBRATION INTERCEPT")
print("=" * 80)

logit_oot = np.log(
    oot_pd_clip /
    (1 - oot_pd_clip)
).reshape(-1, 1)

try:

    intercept_model = LogisticRegression(
        fit_intercept=True,
        penalty=None,
        solver="lbfgs",
        max_iter=2000
    )

    intercept_model.fit(
        logit_oot,
        y_2022
    )

except Exception:

    intercept_model = LogisticRegression(
        fit_intercept=True,
        penalty="none",
        solver="lbfgs",
        max_iter=2000
    )

    intercept_model.fit(
        logit_oot,
        y_2022
    )

calibration_intercept = float(
    intercept_model.intercept_[0]
)

print(
    f"Calibration intercept : "
    f"{calibration_intercept:.4f}"
)

print("\n" + "=" * 80)
print("CALIBRATION SLOPE")
print("=" * 80)

calibration_slope = float(
    intercept_model.coef_[0][0]
)

print(
    f"Calibration slope : "
    f"{calibration_slope:.4f}"
)

print(
    "\nIdeal calibration:"
)

print(
    "Intercept = 0"
)

print(
    "Slope     = 1"
)

print("\n" + "=" * 80)
print("INTERCEPT-ONLY RECALIBRATION")
print("=" * 80)

recal_logit = (
    np.log(
        oot_pd_clip /
        (1 - oot_pd_clip)
    )
    + calibration_intercept
)

oot_pd_intercept_adjusted = (
    1 /
    (
        1 +
        np.exp(-recal_logit)
    )
)

adjusted_brier = brier_score_loss(
    y_2022,
    oot_pd_intercept_adjusted
)

adjusted_auc = roc_auc_score(
    y_2022,
    oot_pd_intercept_adjusted
)

original_gap = (
    oot_pd.mean()
    - y_2022.mean()
)

adjusted_gap = (
    oot_pd_intercept_adjusted.mean()
    - y_2022.mean()
)

print(
    f"Original average PD       : "
    f"{oot_pd.mean() * 100:.2f}%"
)

print(
    f"Adjusted average PD       : "
    f"{oot_pd_intercept_adjusted.mean() * 100:.2f}%"
)

print(
    f"Actual default rate       : "
    f"{y_2022.mean() * 100:.2f}%"
)

print(
    f"\nOriginal calibration gap  : "
    f"{original_gap * 100:.2f} pp"
)

print(
    f"Adjusted calibration gap  : "
    f"{adjusted_gap * 100:.2f} pp"
)

print(
    f"\nOriginal Brier             : "
    f"{oot_brier:.4f}"
)

print(
    f"Adjusted Brier             : "
    f"{adjusted_brier:.4f}"
)

print(
    f"\nOriginal ROC-AUC           : "
    f"{oot_auc:.4f}"
)

print(
    f"Adjusted ROC-AUC           : "
    f"{adjusted_auc:.4f}"
)

print("\n" + "=" * 80)
print("FULL CALIBRATION RECALIBRATION")
print("=" * 80)

full_recal_logit = (
    calibration_intercept
    +
    calibration_slope *
    np.log(
        oot_pd_clip /
        (1 - oot_pd_clip)
    )
)

oot_pd_full_recalibrated = (
    1 /
    (
        1 +
        np.exp(-full_recal_logit)
    )
)

full_recal_brier = brier_score_loss(
    y_2022,
    oot_pd_full_recalibrated
)

full_recal_auc = roc_auc_score(
    y_2022,
    oot_pd_full_recalibrated
)

full_recal_gap = (
    oot_pd_full_recalibrated.mean()
    - y_2022.mean()
)

print(
    f"Original average PD      : "
    f"{oot_pd.mean() * 100:.2f}%"
)

print(
    f"Full recalibrated PD     : "
    f"{oot_pd_full_recalibrated.mean() * 100:.2f}%"
)

print(
    f"Actual default rate      : "
    f"{y_2022.mean() * 100:.2f}%"
)

print(
    f"\nOriginal calibration gap : "
    f"{original_gap * 100:.2f} pp"
)

print(
    f"Full recalibrated gap     : "
    f"{full_recal_gap * 100:.2f} pp"
)

print(
    f"\nOriginal Brier            : "
    f"{oot_brier:.4f}"
)

print(
    f"Full recalibrated Brier   : "
    f"{full_recal_brier:.4f}"
)

print(
    f"\nOriginal ROC-AUC          : "
    f"{oot_auc:.4f}"
)

print(
    f"Full recalibrated AUC     : "
    f"{full_recal_auc:.4f}"
)

print("\n" + "=" * 80)
print("2022 RISK DECILE CALIBRATION")
print("=" * 80)

decile_df = pd.DataFrame({
    "Default_36M": y_2022.values,
    "Predicted_PD": oot_pd
})

decile_df["Risk_Decile"] = pd.qcut(
    decile_df["Predicted_PD"],
    q=10,
    labels=False,
    duplicates="drop"
) + 1

decile_summary = (
    decile_df
    .groupby("Risk_Decile")
    .agg(
        Loans=("Default_36M", "size"),
        Defaults=("Default_36M", "sum"),
        Actual_Default_Rate=("Default_36M", "mean"),
        Average_PD=("Predicted_PD", "mean")
    )
    .reset_index()
)

decile_summary["Actual_Default_Rate_%"] = (
    decile_summary["Actual_Default_Rate"] * 100
)

decile_summary["Average_PD_%"] = (
    decile_summary["Average_PD"] * 100
)

decile_summary["Calibration_Gap_pp"] = (
    decile_summary["Average_PD_%"]
    -
    decile_summary["Actual_Default_Rate_%"]
)

print(
    decile_summary[
        [
            "Risk_Decile",
            "Loans",
            "Defaults",
            "Actual_Default_Rate_%",
            "Average_PD_%",
            "Calibration_Gap_pp"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("TRAINING VS 2022 DECILE COMPARISON")
print("=" * 80)

train_decile_df = pd.DataFrame({
    "Default_36M": y_train.values,
    "Predicted_PD": train_pd
})

train_decile_df["Risk_Decile"] = pd.qcut(
    train_decile_df["Predicted_PD"],
    q=10,
    labels=False,
    duplicates="drop"
) + 1

train_decile_summary = (
    train_decile_df
    .groupby("Risk_Decile")
    .agg(
        Loans=("Default_36M", "size"),
        Defaults=("Default_36M", "sum"),
        Actual_Default_Rate=("Default_36M", "mean"),
        Average_PD=("Predicted_PD", "mean")
    )
    .reset_index()
)

train_decile_summary["Actual_Default_Rate_%"] = (
    train_decile_summary["Actual_Default_Rate"] * 100
)

train_decile_summary["Average_PD_%"] = (
    train_decile_summary["Average_PD"] * 100
)

train_decile_summary["Calibration_Gap_pp"] = (
    train_decile_summary["Average_PD_%"]
    -
    train_decile_summary["Actual_Default_Rate_%"]
)

comparison_deciles = pd.DataFrame({
    "Decile":
        train_decile_summary["Risk_Decile"],

    "Train_Actual_Default_%":
        train_decile_summary["Actual_Default_Rate_%"],

    "Train_Average_PD_%":
        train_decile_summary["Average_PD_%"],

    "OOT_2022_Actual_Default_%":
        decile_summary["Actual_Default_Rate_%"],

    "OOT_2022_Average_PD_%":
        decile_summary["Average_PD_%"]
})

comparison_deciles["Actual_Default_Shift_pp"] = (
    comparison_deciles[
        "OOT_2022_Actual_Default_%"
    ]
    -
    comparison_deciles[
        "Train_Actual_Default_%"
    ]
)

comparison_deciles["PD_Shift_pp"] = (
    comparison_deciles[
        "OOT_2022_Average_PD_%"
    ]
    -
    comparison_deciles[
        "Train_Average_PD_%"
    ]
)

print(
    comparison_deciles.to_string(
        index=False
    )
)

print("\n" + "=" * 100)
print("STEP 24 — FINAL CREDIT-RISK DIAGNOSIS")
print("=" * 100)

gap_improvement_intercept = (
    abs(original_gap)
    -
    abs(adjusted_gap)
)

gap_improvement_full = (
    abs(original_gap)
    -
    abs(full_recal_gap)
)

print(
    f"\nCalibration intercept       : "
    f"{calibration_intercept:.4f}"
)

print(
    f"Calibration slope           : "
    f"{calibration_slope:.4f}"
)

print(
    f"Original calibration gap    : "
    f"{original_gap * 100:.2f} pp"
)

print(
    f"Intercept-adjusted gap      : "
    f"{adjusted_gap * 100:.2f} pp"
)

print(
    f"Full recalibrated gap       : "
    f"{full_recal_gap * 100:.2f} pp"
)

print(
    f"\nGap improvement — intercept : "
    f"{gap_improvement_intercept * 100:.2f} pp"
)

print(
    f"Gap improvement — full      : "
    f"{gap_improvement_full * 100:.2f} pp"
)

print("\n" + "=" * 100)
print("INTERPRETATION")
print("=" * 100)

if (
    abs(calibration_intercept) > 0.50
    and
    0.80 <= calibration_slope <= 1.20
    and
    abs(adjusted_gap) < abs(original_gap)
):

    print(
        """
✓ STRONG EVIDENCE OF CALIBRATION-LEVEL DRIFT

The model's ranking structure is broadly preserved,
but the absolute PD level has shifted in 2022.

This supports a temporal calibration-drift explanation.

The model should NOT be described as having completely failed.

The appropriate remedy is recalibration rather than immediate
full model redevelopment.
"""
    )

elif (
    abs(calibration_intercept) > 0.50
    and
    calibration_slope < 0.80
):

    print(
        """
✓ EVIDENCE OF CALIBRATION + RISK-SCALE DRIFT

The model has both a systematic PD-level shift and evidence
that the predicted probabilities are too extreme.

A recalibration layer should be investigated before considering
complete model redevelopment.
"""
    )

elif (
    abs(calibration_intercept) <= 0.50
    and
    abs(calibration_slope - 1) <= 0.20
):

    print(
        """
✓ CALIBRATION RELATIONSHIP IS RELATIVELY STABLE

The aggregate calibration deterioration therefore requires
further investigation into population composition, target
definition, default timing, or other temporal effects.
"""
    )

else:

    print(
        """
⚠ MIXED CALIBRATION DRIFT

The 2022 relationship between predicted probabilities and
realized defaults has changed.

The evidence supports temporal instability, but does not yet
justify full model redevelopment.
"""
    )

calibration_results = pd.DataFrame({
    "Metric": [
        "2022_Default_Rate",
        "2022_Average_PD",
        "Original_Calibration_Gap",
        "Calibration_Intercept",
        "Calibration_Slope",
        "Intercept_Adjusted_Average_PD",
        "Intercept_Adjusted_Gap",
        "Full_Recalibrated_Average_PD",
        "Full_Recalibrated_Gap",
        "Original_ROC_AUC",
        "Adjusted_ROC_AUC",
        "Full_Recalibrated_ROC_AUC",
        "Original_Brier",
        "Adjusted_Brier",
        "Full_Recalibrated_Brier"
    ],

    "Value": [
        oot_default,
        oot_mean_pd,
        original_gap,
        calibration_intercept,
        calibration_slope,
        oot_pd_intercept_adjusted.mean(),
        adjusted_gap,
        oot_pd_full_recalibrated.mean(),
        full_recal_gap,
        oot_auc,
        adjusted_auc,
        full_recal_auc,
        oot_brier,
        adjusted_brier,
        full_recal_brier
    ]
})

calibration_deciles_2022 = (
    decile_summary.copy()
)

calibration_deciles_train = (
    train_decile_summary.copy()
)

calibration_decile_comparison = (
    comparison_deciles.copy()
)

print("\n" + "=" * 100)
print("STEP 24 COMPLETE")
print("=" * 100)

print(
    "\n✓ calibration_results created."
)

print(
    "✓ calibration_deciles_2022 created."
)

print(
    "✓ calibration_deciles_train created."
)

print(
    "✓ calibration_decile_comparison created."
)


In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    roc_curve
)
from sklearn.linear_model import LogisticRegression

print("=" * 100)
print("STEP 25 — FINAL MODEL VS RECALIBRATED MODEL COMPARISON")
print("=" * 100)

required_objects = [
    "model",
    "model_features",
    "oot_2022"
]

missing_objects = [
    x for x in required_objects
    if x not in globals()
]

if missing_objects:
    raise ValueError(
        f"Missing required objects: {missing_objects}"
    )

print("\n✓ Required objects found.")

def rebuild_engineered_flags(df):

    df = df.copy()

    # Low credit
    if "LowCreditFlag" not in df.columns:
        df["LowCreditFlag"] = (
            pd.to_numeric(
                df["CreditScore"],
                errors="coerce"
            ) < 680
        ).astype(int)

    # High LTV
    if "HighLTVFlag" not in df.columns:
        df["HighLTVFlag"] = (
            pd.to_numeric(
                df["OriginalLTV"],
                errors="coerce"
            ) >= 80
        ).astype(int)

    # High DTI
    if "HighDTIFlag" not in df.columns:
        df["HighDTIFlag"] = (
            pd.to_numeric(
                df["OriginalDTI"],
                errors="coerce"
            ) >= 43
        ).astype(int)

    # High CLTV
    if "HighCLTVFlag" not in df.columns:
        df["HighCLTVFlag"] = (
            pd.to_numeric(
                df["OriginalCLTV"],
                errors="coerce"
            ) >= 80
        ).astype(int)

    # High interest rate
    if "HighRateFlag" not in df.columns:

        # Use the same threshold established in Step 24
        rate_threshold = 4.75

        df["HighRateFlag"] = (
            pd.to_numeric(
                df["OriginalInterestRate"],
                errors="coerce"
            ) >= rate_threshold
        ).astype(int)

    # High credit-LTV interaction
    if "HighCreditLTVFlag" not in df.columns:

        df["HighCreditLTVFlag"] = (
            (
                pd.to_numeric(
                    df["CreditScore"],
                    errors="coerce"
                ) < 680
            )
            &
            (
                pd.to_numeric(
                    df["OriginalLTV"],
                    errors="coerce"
                ) >= 80
            )
        ).astype(int)

    # High DTI-LTV interaction
    if "HighDTILTVFlag" not in df.columns:

        df["HighDTILTVFlag"] = (
            (
                pd.to_numeric(
                    df["OriginalDTI"],
                    errors="coerce"
                ) >= 43
            )
            &
            (
                pd.to_numeric(
                    df["OriginalLTV"],
                    errors="coerce"
                ) >= 80
            )
        ).astype(int)

    return df

oot_final = rebuild_engineered_flags(
    oot_2022.copy()
)

model_features_final = list(model_features)

missing_features = [
    c for c in model_features_final
    if c not in oot_final.columns
]

if missing_features:
    raise ValueError(
        f"2022 data is missing model features: {missing_features}"
    )

print(
    f"\n2022 observations: {len(oot_final):,}"
)

print(
    f"Model features: {len(model_features_final)}"
)

print("✓ All model features available.")

X_2022 = oot_final[
    model_features_final
].copy()

y_2022 = pd.to_numeric(
    oot_final["Default_36M"],
    errors="coerce"
)

valid_mask = (
    y_2022.notna()
    &
    X_2022.notna().all(axis=1)
)

X_2022 = X_2022.loc[valid_mask]
y_2022 = y_2022.loc[valid_mask]

print(
    f"\nFinal valid 2022 observations: {len(X_2022):,}"
)

p_original = model.predict_proba(
    X_2022
)[:, 1]

p_original = np.clip(
    p_original,
    1e-6,
    1 - 1e-6
)

print("✓ Original model predictions generated.")

logit_original = np.log(
    p_original / (1 - p_original)
)

cal_intercept_model = LogisticRegression(
    fit_intercept=True,
    solver="lbfgs",
    max_iter=1000
)

# Intercept-only calibration:
# coefficient fixed at 1, estimate only intercept
cal_intercept = (
    np.log(
        np.mean(y_2022)
        /
        (1 - np.mean(y_2022))
    )
    -
    np.mean(logit_original)
)

p_intercept = 1 / (
    1 + np.exp(
        -(logit_original + cal_intercept)
    )
)

p_intercept = np.clip(
    p_intercept,
    1e-6,
    1 - 1e-6
)

cal_model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000
)

cal_model.fit(
    logit_original.reshape(-1, 1),
    y_2022
)

cal_slope = float(
    cal_model.coef_[0][0]
)

cal_full_intercept = float(
    cal_model.intercept_[0]
)

p_full = cal_model.predict_proba(
    logit_original.reshape(-1, 1)
)[:, 1]

p_full = np.clip(
    p_full,
    1e-6,
    1 - 1e-6
)

print("✓ Intercept-only calibration generated.")
print("✓ Full logistic calibration generated.")

def ks_statistic(y_true, probability):

    fpr, tpr, thresholds = roc_curve(
        y_true,
        probability
    )

    return float(
        np.max(
            np.abs(tpr - fpr)
        )
    )

def calculate_metrics(
    name,
    y_true,
    probability
):

    actual_rate = float(
        np.mean(y_true)
    )

    average_pd = float(
        np.mean(probability)
    )

    calibration_gap = (
        average_pd
        -
        actual_rate
    )

    auc = roc_auc_score(
        y_true,
        probability
    )

    pr_auc = average_precision_score(
        y_true,
        probability
    )

    ks = ks_statistic(
        y_true,
        probability
    )

    brier = brier_score_loss(
        y_true,
        probability
    )

    # Top 10%
    cutoff = np.quantile(
        probability,
        0.90
    )

    top_mask = (
        probability >= cutoff
    )

    top_default_rate = float(
        np.mean(
            y_true[top_mask]
        )
    )

    top_lift = (
        top_default_rate
        /
        actual_rate
        if actual_rate > 0
        else np.nan
    )

    return {
        "Model": name,
        "Loans": len(y_true),
        "Defaults": int(y_true.sum()),
        "Actual_Default_Rate_%": actual_rate * 100,
        "Average_PD_%": average_pd * 100,
        "Calibration_Gap_pp": calibration_gap * 100,
        "ROC_AUC": auc,
        "PR_AUC": pr_auc,
        "KS": ks,
        "Brier": brier,
        "Top_Decile_Default_%": top_default_rate * 100,
        "Top_Decile_Lift": top_lift
    }

final_comparison = pd.DataFrame([

    calculate_metrics(
        "Original Model",
        y_2022,
        p_original
    ),

    calculate_metrics(
        "Intercept Recalibrated",
        y_2022,
        p_intercept
    ),

    calculate_metrics(
        "Full Logistic Recalibration",
        y_2022,
        p_full
    )

])

print("\n")
print("=" * 100)
print("FINAL 2022 MODEL COMPARISON")
print("=" * 100)

print(
    final_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n")
print("=" * 100)
print("CALIBRATION PARAMETERS")
print("=" * 100)

print(
    f"Intercept-only adjustment : {cal_intercept:.4f}"
)

print(
    f"Full calibration intercept: {cal_full_intercept:.4f}"
)

print(
    f"Full calibration slope    : {cal_slope:.4f}"
)

print(
    "\nIdeal calibration:"
)

print(
    "Intercept = 0"
)

print(
    "Slope     = 1"
)

original_gap = abs(
    final_comparison.loc[
        final_comparison["Model"] == "Original Model",
        "Calibration_Gap_pp"
    ].iloc[0]
)

intercept_gap = abs(
    final_comparison.loc[
        final_comparison["Model"] == "Intercept Recalibrated",
        "Calibration_Gap_pp"
    ].iloc[0]
)

full_gap = abs(
    final_comparison.loc[
        final_comparison["Model"] == "Full Logistic Recalibration",
        "Calibration_Gap_pp"
    ].iloc[0]
)

original_brier = final_comparison.loc[
    final_comparison["Model"] == "Original Model",
    "Brier"
].iloc[0]

intercept_brier = final_comparison.loc[
    final_comparison["Model"] == "Intercept Recalibrated",
    "Brier"
].iloc[0]

full_brier = final_comparison.loc[
    final_comparison["Model"] == "Full Logistic Recalibration",
    "Brier"
].iloc[0]

print("\n")
print("=" * 100)
print("IMPROVEMENT SUMMARY")
print("=" * 100)

print(
    f"Original calibration gap       : {original_gap:.2f} pp"
)

print(
    f"Intercept recalibrated gap     : {intercept_gap:.2f} pp"
)

print(
    f"Full recalibrated gap          : {full_gap:.2f} pp"
)

print(
    f"\nGap improvement — intercept    : "
    f"{original_gap - intercept_gap:.2f} pp"
)

print(
    f"Gap improvement — full         : "
    f"{original_gap - full_gap:.2f} pp"
)

print(
    f"\nOriginal Brier                 : {original_brier:.4f}"
)

print(
    f"Intercept recalibrated Brier   : {intercept_brier:.4f}"
)

print(
    f"Full recalibrated Brier        : {full_brier:.4f}"
)

original_auc = final_comparison.loc[
    final_comparison["Model"] == "Original Model",
    "ROC_AUC"
].iloc[0]

full_auc = final_comparison.loc[
    final_comparison["Model"] == "Full Logistic Recalibration",
    "ROC_AUC"
].iloc[0]

print("\n")
print("=" * 100)
print("FINAL CREDIT-RISK MODEL CONCLUSION")
print("=" * 100)

print(
    """
1. DISCRIMINATION
   The model continues to rank 2022 loans meaningfully above/below
   one another by risk. Recalibration does not change ROC-AUC because
   it changes the probability scale rather than the ranking.

2. CALIBRATION
   The original model substantially overpredicts the absolute level
   of 2022 default risk.

3. MODEL DRIFT
   The preceding diagnostics found material population/risk-distribution
   shift between the training population and 2022 OOT population.

4. SEASONING
   The preceding seasoning analysis showed that the deterioration
   remains after restricting to fully seasoned 2022 loans. Therefore,
   right-censoring is not a sufficient explanation.

5. REMEDIATION
   Intercept/full logistic recalibration substantially improves
   probability calibration while preserving discrimination.

6. RECOMMENDATION
   The evidence supports investigating a recalibration layer before
   complete model redevelopment.
   """
)

print(
    f"Original ROC-AUC : {original_auc:.4f}"
)

print(
    f"Recalibrated ROC-AUC : {full_auc:.4f}"
)

print(
    "\n✓ STEP 25 COMPLETE."
)

print(
    "\n✓ FINAL MODEL COMPARISON OBJECT:"
)

print(
    "  final_comparison"
)

print(
    "\n✓ CREDIT-RISK DIAGNOSTIC PIPELINE COMPLETE."
)

print("=" * 100)
